# AIROGS-light classifier panel training

**Public role:** Core frozen-classifier development notebook.

**Relation to the manuscript:** Supports the AIROGS-light classifier panel, internal evaluation, and freezing before external audit.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `07_train_or_load_airogs_light_glaucoma_classifiers.ipynb.ipynb`


In [ ]:
# Cell 1: Load project paths and audit EyePACS-AIROGS-light-V2 classifier inputs

from pathlib import Path
import json
import os
import random
import time
import shutil

import cv2
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing config: {CONFIG_PATH}")

if not PATH_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing path registry: {PATH_REGISTRY_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

path_registry = pd.read_csv(PATH_REGISTRY_PATH)

PATHS = {
    row["path_key"]: Path(row["path"])
    for _, row in path_registry.iterrows()
}

SEED = int(PROJECT_CONFIG.get("seed", 42))

random.seed(SEED)
np.random.seed(SEED)

# New Notebook 07 output folders.
RESULTS_07_DIR = JOURNAL_ROOT / "results" / "07_airogs_light_glaucoma_classifier_benchmark"
RESULTS_07_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFIER_MODEL_ROOT = JOURNAL_ROOT / "models" / "airogs_light_glaucoma_classifiers"
CLASSIFIER_MODEL_ROOT.mkdir(parents=True, exist_ok=True)

# EyePACS-AIROGS-light-V2 location in MyDrive.
# Expected structure:
# ${AQPR_DATA_ROOT}/dataset/eyepac-light-v2-512-jpg/
AI_GLAUCOMA_ROOT = Path(os.environ.get("AQPR_DATA_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma_Data"))
AIROGS_DATASET_PARENT = AI_GLAUCOMA_ROOT / "dataset"
AIROGS_LIGHT_ROOT = AIROGS_DATASET_PARENT / "eyepac-light-v2-512-jpg"

AIROGS_METADATA_CANDIDATES = [
    AIROGS_LIGHT_ROOT / "metadata.csv",
    AIROGS_DATASET_PARENT / "metadata.csv",
]

AIROGS_METADATA_CSV = None

for candidate_csv in AIROGS_METADATA_CANDIDATES:
    if candidate_csv.exists():
        AIROGS_METADATA_CSV = candidate_csv
        break

if not AIROGS_LIGHT_ROOT.exists():
    raise FileNotFoundError(f"Missing AIROGS image folder: {AIROGS_LIGHT_ROOT}")

if AIROGS_METADATA_CSV is None:
    raise FileNotFoundError(
        "Could not find metadata.csv for EyePACS-AIROGS-light-V2. "
        f"Checked: {[str(p) for p in AIROGS_METADATA_CANDIDATES]}"
    )

airogs_metadata = pd.read_csv(AIROGS_METADATA_CSV)

required_columns = [
    "id",
    "file_name",
    "label",
    "label_binary",
    "folder",
    "source_dataset",
    "relative_file_type",
    "file_path",
]

missing_columns = [
    col for col in required_columns
    if col not in airogs_metadata.columns
]

if missing_columns:
    raise ValueError(f"Missing required AIROGS-light metadata columns: {missing_columns}")

airogs_metadata = airogs_metadata.copy().reset_index(drop=True)

airogs_metadata["id"] = airogs_metadata["id"].astype(str)
airogs_metadata["file_name"] = airogs_metadata["file_name"].astype(str)
airogs_metadata["label"] = airogs_metadata["label"].astype(str)
airogs_metadata["label_binary"] = pd.to_numeric(
    airogs_metadata["label_binary"],
    errors="coerce",
).astype("Int64")
airogs_metadata["folder"] = airogs_metadata["folder"].astype(str)
airogs_metadata["source_dataset"] = airogs_metadata["source_dataset"].astype(str)
airogs_metadata["relative_file_type"] = airogs_metadata["relative_file_type"].astype(str)
airogs_metadata["file_path"] = airogs_metadata["file_path"].astype(str)

# Canonical split names used by downstream classifier code.
split_map = {
    "train": "train",
    "validation": "val",
    "val": "val",
    "test": "test",
}

airogs_metadata["classifier_split"] = airogs_metadata["folder"].map(split_map)

if airogs_metadata["classifier_split"].isna().any():
    bad_splits = sorted(
        airogs_metadata.loc[
            airogs_metadata["classifier_split"].isna(),
            "folder",
        ].dropna().unique().tolist()
    )
    raise ValueError(f"Unexpected folder/split values in AIROGS-light metadata: {bad_splits}")

# Construct stable sample IDs.
# The original 'id' column is not used as primary key because it can be duplicated.
airogs_metadata["sample_id"] = (
    airogs_metadata["classifier_split"].astype(str)
    + "__"
    + airogs_metadata["label"].astype(str)
    + "__"
    + airogs_metadata["file_name"].astype(str)
)

duplicate_sample_ids = int(airogs_metadata["sample_id"].duplicated().sum())
duplicate_file_names = int(airogs_metadata["file_name"].duplicated().sum())
duplicate_original_ids = int(airogs_metadata["id"].duplicated().sum())

# Construct image paths.
# The CSV file_path usually starts with:
# /eyepac-light-v2-512-jpg/train/RG/
# Therefore, the correct base is AIROGS_DATASET_PARENT.
def build_image_path(row):
    relative_dir = str(row["file_path"]).strip().lstrip("/")
    file_name = str(row["file_name"])

    candidate_from_parent = AIROGS_DATASET_PARENT / relative_dir / file_name

    # Fallback for metadata variants where file_path may be only train/RG/.
    candidate_from_dataset_root = AIROGS_LIGHT_ROOT / relative_dir / file_name

    if candidate_from_parent.exists():
        return candidate_from_parent

    if candidate_from_dataset_root.exists():
        return candidate_from_dataset_root

    # Return the expected canonical path even if it does not exist,
    # so the missing-file audit can report it clearly.
    return candidate_from_parent

airogs_metadata["classifier_image_path"] = airogs_metadata.apply(build_image_path, axis=1)
airogs_metadata["image_path"] = airogs_metadata["classifier_image_path"].astype(str)

# Compatibility columns for downstream training utilities.
airogs_metadata["glaucoma_binary"] = airogs_metadata["label_binary"].astype(int)
airogs_metadata["glaucoma_label"] = airogs_metadata["label"]

# AIROGS-light-v2 metadata does not provide patient identifiers.
# Do not use pseudo_patient_id for patient-level claims.
airogs_metadata["patient_id"] = pd.NA
airogs_metadata["has_patient_id"] = False
airogs_metadata["pseudo_patient_id"] = airogs_metadata["sample_id"]

# Keep this alias only to reduce downstream refactoring.
# In the new notebook, this is AIROGS-light metadata, not HYGD anatomical proxy data.
analysis_manifest = airogs_metadata.copy()

# Image existence audit.
image_exists = airogs_metadata["classifier_image_path"].apply(lambda p: Path(p).exists())

n_images = int(len(airogs_metadata))
n_missing_images = int((~image_exists).sum())
n_existing_images = int(image_exists.sum())

n_positive = int((airogs_metadata["glaucoma_binary"] == 1).sum())
n_negative = int((airogs_metadata["glaucoma_binary"] == 0).sum())
n_missing_labels = int(airogs_metadata["glaucoma_binary"].isna().sum())

split_label_summary = (
    airogs_metadata
    .groupby(["classifier_split", "label", "glaucoma_binary"], dropna=False)
    .size()
    .reset_index(name="n_images")
    .sort_values(["classifier_split", "label"])
    .reset_index(drop=True)
)

split_summary = (
    airogs_metadata
    .groupby("classifier_split", dropna=False)
    .agg(
        n_images=("sample_id", "count"),
        n_unique_file_names=("file_name", "nunique"),
        n_unique_sample_ids=("sample_id", "nunique"),
        n_rg=("glaucoma_binary", lambda x: int((x == 1).sum())),
        n_nrg=("glaucoma_binary", lambda x: int((x == 0).sum())),
        n_existing_images=("classifier_image_path", lambda x: int(sum(Path(p).exists() for p in x))),
    )
    .reset_index()
    .sort_values("classifier_split")
)

source_summary = (
    airogs_metadata
    .groupby("source_dataset", dropna=False)
    .size()
    .reset_index(name="n_images")
    .sort_values("n_images", ascending=False)
    .reset_index(drop=True)
)

# Save runtime manifest and audit outputs.
AIROGS_RUNTIME_MANIFEST_CSV = RESULTS_07_DIR / "notebook07_airogs_light_runtime_manifest.csv"
SETUP_SUMMARY_JSON = RESULTS_07_DIR / "notebook07_setup_summary.json"
SPLIT_SUMMARY_CSV = RESULTS_07_DIR / "notebook07_airogs_light_split_summary.csv"
SPLIT_LABEL_SUMMARY_CSV = RESULTS_07_DIR / "notebook07_airogs_light_split_label_summary.csv"
SOURCE_SUMMARY_CSV = RESULTS_07_DIR / "notebook07_airogs_light_source_summary.csv"

airogs_metadata.to_csv(AIROGS_RUNTIME_MANIFEST_CSV, index=False)
split_summary.to_csv(SPLIT_SUMMARY_CSV, index=False)
split_label_summary.to_csv(SPLIT_LABEL_SUMMARY_CSV, index=False)
source_summary.to_csv(SOURCE_SUMMARY_CSV, index=False)

setup_summary = {
    "notebook": "07_train_or_load_airogs_light_glaucoma_classifiers",
    "seed": SEED,
    "journal_root": str(JOURNAL_ROOT),
    "results_07_dir": str(RESULTS_07_DIR),
    "classifier_model_root": str(CLASSIFIER_MODEL_ROOT),
    "ai_glaucoma_root": str(AI_GLAUCOMA_ROOT),
    "airogs_dataset_parent": str(AIROGS_DATASET_PARENT),
    "airogs_light_root": str(AIROGS_LIGHT_ROOT),
    "airogs_metadata_csv": str(AIROGS_METADATA_CSV),
    "airogs_runtime_manifest_csv": str(AIROGS_RUNTIME_MANIFEST_CSV),
    "n_images": n_images,
    "n_existing_images": n_existing_images,
    "n_missing_images": n_missing_images,
    "n_rg_images": n_positive,
    "n_nrg_images": n_negative,
    "n_missing_labels": n_missing_labels,
    "duplicate_sample_ids": duplicate_sample_ids,
    "duplicate_file_names": duplicate_file_names,
    "duplicate_original_ids": duplicate_original_ids,
    "has_patient_id": False,
    "patient_level_internal_evaluation_available": False,
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "external_datasets_reserved_for_later_aqpr": [
        "HYGD",
        "PAPILA",
        "SMDG",
    ],
}

with open(SETUP_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(setup_summary, f, indent=2)

print("EyePACS-AIROGS-light-V2 setup audit")
print(f"Dataset parent: {AIROGS_DATASET_PARENT}")
print(f"Dataset root: {AIROGS_LIGHT_ROOT}")
print(f"Metadata CSV: {AIROGS_METADATA_CSV}")
print(f"Images: {n_images}")
print(f"Existing image files: {n_existing_images}")
print(f"Missing image files: {n_missing_images}")
print(f"RG images: {n_positive}")
print(f"NRG images: {n_negative}")
print(f"Missing labels: {n_missing_labels}")
print(f"Duplicate original IDs: {duplicate_original_ids}")
print(f"Duplicate file names: {duplicate_file_names}")
print(f"Duplicate sample IDs: {duplicate_sample_ids}")

print("\nSplit summary:")
print(split_summary.to_string(index=False))

print("\nSplit-label summary:")
print(split_label_summary.to_string(index=False))

print(f"\nRuntime manifest: {AIROGS_RUNTIME_MANIFEST_CSV}")
print(f"Setup summary: {SETUP_SUMMARY_JSON}")

expected_counts = {
    "train": 8000,
    "val": 770,
    "test": 770,
}

observed_counts = split_summary.set_index("classifier_split")["n_images"].to_dict()

for split_name, expected_count in expected_counts.items():
    observed_count = int(observed_counts.get(split_name, -1))

    if observed_count != expected_count:
        raise RuntimeError(
            f"Unexpected {split_name} image count: "
            f"expected={expected_count}, observed={observed_count}"
        )

if n_images != 9540:
    raise RuntimeError(f"Expected 9540 AIROGS-light images, found {n_images}")

if n_positive != 4770:
    raise RuntimeError(f"Expected 4770 RG images, found {n_positive}")

if n_negative != 4770:
    raise RuntimeError(f"Expected 4770 NRG images, found {n_negative}")

if n_missing_labels > 0:
    raise RuntimeError(f"Missing labels found: {n_missing_labels}")

if duplicate_sample_ids > 0:
    raise RuntimeError(f"Duplicate sample IDs found: {duplicate_sample_ids}")

if duplicate_file_names > 0:
    raise RuntimeError(f"Duplicate file names found: {duplicate_file_names}")

if n_missing_images > 0:
    missing_preview = airogs_metadata.loc[
        ~image_exists,
        ["sample_id", "classifier_image_path"]
    ].head(20)

    print("\nMissing image preview:")
    print(missing_preview.to_string(index=False))

    raise FileNotFoundError(
        f"Missing {n_missing_images} image files. "
        "Check AIROGS_DATASET_PARENT, AIROGS_LIGHT_ROOT, and the dataset folder structure."
    )

print("Status: PASS")

In [ ]:
# Cell 2: Create locked AIROGS-light-V2 classifier split manifest from official splits

SPLIT_DIR = JOURNAL_ROOT / "project_data" / "airogs_light_glaucoma_classifier_splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SPLIT_MANIFEST_CSV = SPLIT_DIR / f"airogs_light_image_level_split_manifest_seed{SEED}.csv"
SPLIT_COMPATIBILITY_CSV = SPLIT_DIR / f"airogs_light_split_compatibility_manifest_seed{SEED}.csv"

RESULTS_IMAGE_SPLIT_CSV = RESULTS_07_DIR / "notebook07_image_level_split_manifest.csv"
RESULTS_SPLIT_COMPATIBILITY_CSV = RESULTS_07_DIR / "notebook07_split_compatibility_manifest.csv"
SPLIT_SUMMARY_CSV = RESULTS_07_DIR / "notebook07_split_summary.csv"
SPLIT_LABEL_SUMMARY_CSV = RESULTS_07_DIR / "notebook07_split_label_summary.csv"
SPLIT_PROXY_QC_SUMMARY_CSV = RESULTS_07_DIR / "notebook07_split_proxy_qc_summary.csv"
SPLIT_CONFIG_JSON = RESULTS_07_DIR / "notebook07_split_config.json"

FORCE_RECREATE_SPLIT = False


def as_bool_series(series):
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "1": True,
                "yes": True,
                "y": True,
                "false": False,
                "0": False,
                "no": False,
                "n": False,
                "nan": False,
                "none": False,
                "": False,
            }
        )
        .fillna(False)
        .astype(bool)
    )


if "analysis_manifest" not in globals():
    if not AIROGS_RUNTIME_MANIFEST_CSV.exists():
        raise FileNotFoundError(f"Missing AIROGS runtime manifest: {AIROGS_RUNTIME_MANIFEST_CSV}")

    analysis_manifest = pd.read_csv(AIROGS_RUNTIME_MANIFEST_CSV)

analysis_manifest = analysis_manifest.copy().reset_index(drop=True)

required_manifest_columns = [
    "sample_id",
    "file_name",
    "label",
    "label_binary",
    "glaucoma_binary",
    "classifier_split",
    "classifier_image_path",
    "image_path",
]

missing_manifest_columns = [
    col for col in required_manifest_columns
    if col not in analysis_manifest.columns
]

if missing_manifest_columns:
    raise ValueError(f"Missing required AIROGS split manifest columns: {missing_manifest_columns}")

analysis_manifest["sample_id"] = analysis_manifest["sample_id"].astype(str)
analysis_manifest["file_name"] = analysis_manifest["file_name"].astype(str)
analysis_manifest["label"] = analysis_manifest["label"].astype(str)
analysis_manifest["label_binary"] = pd.to_numeric(
    analysis_manifest["label_binary"],
    errors="coerce",
).astype(int)
analysis_manifest["glaucoma_binary"] = pd.to_numeric(
    analysis_manifest["glaucoma_binary"],
    errors="coerce",
).astype(int)
analysis_manifest["classifier_split"] = analysis_manifest["classifier_split"].astype(str)
analysis_manifest["classifier_image_path"] = analysis_manifest["classifier_image_path"].astype(str)
analysis_manifest["image_path"] = analysis_manifest["image_path"].astype(str)

# Compatibility aliases for the existing downstream classifier code.
analysis_manifest["glaucoma_split"] = analysis_manifest["classifier_split"]
analysis_manifest["gon_binary"] = analysis_manifest["glaucoma_binary"]
analysis_manifest["gon_label"] = np.where(
    analysis_manifest["gon_binary"].astype(int) == 1,
    "GON+",
    "GON-",
)

# AIROGS-light-v2 does not provide true patient identifiers.
# For internal training utilities only, each image receives a pseudo patient ID.
# This prevents accidental aggregation of all images into one missing patient group.
# No patient-level internal claim is allowed from this pseudo ID.
analysis_manifest["true_patient_id_available"] = False
analysis_manifest["patient_id_policy"] = "image_as_pseudo_patient_for_internal_training_compatibility"
analysis_manifest["patient_id"] = "image_as_patient__" + analysis_manifest["sample_id"].astype(str)

# AIROGS-light-v2 does not provide HYGD-like quality scores or anatomical proxy masks.
# These columns are kept only for compatibility with the existing training/evaluation code.
analysis_manifest["quality_score"] = 1.0
analysis_manifest["quality_score_available"] = False
analysis_manifest["quality_score_policy"] = "constant_placeholder_not_used_for_internal_quality_claims"

analysis_manifest["include_strict_anatomical_analysis"] = False
analysis_manifest["include_high_confidence_anatomical_analysis"] = False
analysis_manifest["low_confidence_anatomical_proxy"] = False
analysis_manifest["excluded_from_anatomical_analysis"] = False
analysis_manifest["anatomical_proxy_available"] = False
analysis_manifest["anatomical_proxy_policy"] = "not_available_for_airogs_light_classifier_training"

if FORCE_RECREATE_SPLIT:
    for path in [
        IMAGE_SPLIT_MANIFEST_CSV,
        SPLIT_COMPATIBILITY_CSV,
    ]:
        if path.exists():
            path.unlink()

if IMAGE_SPLIT_MANIFEST_CSV.exists() and SPLIT_COMPATIBILITY_CSV.exists():
    image_split_manifest = pd.read_csv(IMAGE_SPLIT_MANIFEST_CSV)
    split_compatibility_manifest = pd.read_csv(SPLIT_COMPATIBILITY_CSV)

    split_source = "loaded_existing_official_airogs_light_split_manifest"

else:
    image_split_manifest = analysis_manifest.copy()

    split_compatibility_manifest = (
        image_split_manifest[
            [
                "patient_id",
                "sample_id",
                "glaucoma_split",
                "gon_binary",
                "gon_label",
                "true_patient_id_available",
                "patient_id_policy",
            ]
        ]
        .copy()
        .rename(columns={"gon_binary": "gon_binary_patient", "gon_label": "gon_label_patient"})
    )

    split_compatibility_manifest["n_images"] = 1
    split_compatibility_manifest["n_strict_valid_proxy_images"] = 0
    split_compatibility_manifest["n_high_confidence_proxy_images"] = 0
    split_compatibility_manifest["n_low_confidence_proxy_images"] = 0
    split_compatibility_manifest["n_failed_proxy_images"] = 0
    split_compatibility_manifest["quality_score_mean"] = 1.0
    split_compatibility_manifest["quality_score_min"] = 1.0
    split_compatibility_manifest["quality_score_max"] = 1.0

    image_split_manifest.to_csv(IMAGE_SPLIT_MANIFEST_CSV, index=False)
    split_compatibility_manifest.to_csv(SPLIT_COMPATIBILITY_CSV, index=False)

    split_source = "created_from_official_airogs_light_splits"

image_split_manifest = image_split_manifest.copy().reset_index(drop=True)
split_compatibility_manifest = split_compatibility_manifest.copy().reset_index(drop=True)

image_split_manifest["sample_id"] = image_split_manifest["sample_id"].astype(str)
image_split_manifest["patient_id"] = image_split_manifest["patient_id"].astype(str)
image_split_manifest["gon_binary"] = pd.to_numeric(
    image_split_manifest["gon_binary"],
    errors="coerce",
).astype(int)
image_split_manifest["glaucoma_split"] = image_split_manifest["glaucoma_split"].astype(str)
image_split_manifest["classifier_image_path"] = image_split_manifest["classifier_image_path"].astype(str)
image_split_manifest["quality_score"] = pd.to_numeric(
    image_split_manifest["quality_score"],
    errors="coerce",
).astype(float)

for bool_col in [
    "include_strict_anatomical_analysis",
    "include_high_confidence_anatomical_analysis",
    "low_confidence_anatomical_proxy",
    "excluded_from_anatomical_analysis",
    "true_patient_id_available",
    "quality_score_available",
    "anatomical_proxy_available",
]:
    if bool_col in image_split_manifest.columns:
        image_split_manifest[bool_col] = as_bool_series(image_split_manifest[bool_col])

image_split_manifest["classifier_image_exists"] = image_split_manifest["classifier_image_path"].apply(
    lambda x: Path(str(x)).exists()
)

missing_split_assignments = int(image_split_manifest["glaucoma_split"].isna().sum())
missing_classifier_images = int((~image_split_manifest["classifier_image_exists"]).sum())
duplicate_sample_ids = int(image_split_manifest["sample_id"].duplicated().sum())
duplicate_file_names = int(image_split_manifest["file_name"].duplicated().sum())

split_summary = (
    image_split_manifest
    .groupby("glaucoma_split")
    .agg(
        n_images=("sample_id", "count"),
        n_unique_file_names=("file_name", "nunique"),
        n_pseudo_patients=("patient_id", "nunique"),
        n_rg_images=("gon_binary", lambda x: int((x == 1).sum())),
        n_nrg_images=("gon_binary", lambda x: int((x == 0).sum())),
        n_existing_images=("classifier_image_exists", "sum"),
    )
    .reset_index()
)

split_label_summary = (
    image_split_manifest
    .groupby(["glaucoma_split", "gon_label", "gon_binary"])
    .agg(
        n_images=("sample_id", "count"),
        n_pseudo_patients=("patient_id", "nunique"),
    )
    .reset_index()
)

split_proxy_qc_summary = (
    image_split_manifest
    .groupby("glaucoma_split")
    .agg(
        n_images=("sample_id", "count"),
        strict_valid_proxy_images=("include_strict_anatomical_analysis", "sum"),
        high_confidence_proxy_images=("include_high_confidence_anatomical_analysis", "sum"),
        low_confidence_proxy_images=("low_confidence_anatomical_proxy", "sum"),
        failed_proxy_images=("excluded_from_anatomical_analysis", "sum"),
        anatomical_proxy_available_images=("anatomical_proxy_available", "sum"),
    )
    .reset_index()
)

split_order = {"train": 0, "val": 1, "test": 2}

split_summary["split_order"] = split_summary["glaucoma_split"].map(split_order)
split_label_summary["split_order"] = split_label_summary["glaucoma_split"].map(split_order)
split_proxy_qc_summary["split_order"] = split_proxy_qc_summary["glaucoma_split"].map(split_order)

split_summary = (
    split_summary
    .sort_values("split_order")
    .drop(columns="split_order")
    .reset_index(drop=True)
)

split_label_summary = (
    split_label_summary
    .sort_values(["split_order", "gon_label"])
    .drop(columns="split_order")
    .reset_index(drop=True)
)

split_proxy_qc_summary = (
    split_proxy_qc_summary
    .sort_values("split_order")
    .drop(columns="split_order")
    .reset_index(drop=True)
)

image_split_manifest.to_csv(IMAGE_SPLIT_MANIFEST_CSV, index=False)
image_split_manifest.to_csv(RESULTS_IMAGE_SPLIT_CSV, index=False)

split_compatibility_manifest.to_csv(SPLIT_COMPATIBILITY_CSV, index=False)
split_compatibility_manifest.to_csv(RESULTS_SPLIT_COMPATIBILITY_CSV, index=False)

split_summary.to_csv(SPLIT_SUMMARY_CSV, index=False)
split_label_summary.to_csv(SPLIT_LABEL_SUMMARY_CSV, index=False)
split_proxy_qc_summary.to_csv(SPLIT_PROXY_QC_SUMMARY_CSV, index=False)

train_ids = set(image_split_manifest.loc[image_split_manifest["glaucoma_split"] == "train", "sample_id"])
val_ids = set(image_split_manifest.loc[image_split_manifest["glaucoma_split"] == "val", "sample_id"])
test_ids = set(image_split_manifest.loc[image_split_manifest["glaucoma_split"] == "test", "sample_id"])

leakage_checks = {
    "train_val_sample_overlap": len(train_ids & val_ids),
    "train_test_sample_overlap": len(train_ids & test_ids),
    "val_test_sample_overlap": len(val_ids & test_ids),
}

split_config = {
    "split_source": split_source,
    "seed": SEED,
    "split_type": "official_airogs_light_v2_train_val_test",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "true_patient_id_available": False,
    "patient_id_policy": "image_as_pseudo_patient_for_internal_training_compatibility",
    "patient_level_internal_claim_allowed": False,
    "quality_score_available": False,
    "quality_score_policy": "constant_placeholder_not_used_for_internal_quality_claims",
    "anatomical_proxy_available_for_training_dataset": False,
    "anatomical_proxy_policy": "not_available_for_airogs_light_classifier_training",
    "locked_test_policy": "test split is not used for model selection or architecture screening",
    "image_split_manifest_csv": str(IMAGE_SPLIT_MANIFEST_CSV),
    "split_compatibility_manifest_csv": str(SPLIT_COMPATIBILITY_CSV),
    "leakage_checks": leakage_checks,
    "external_datasets_reserved_for_aqpr": [
        "HYGD",
        "PAPILA",
        "SMDG",
    ],
}

with open(SPLIT_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(split_config, f, indent=2)

print(f"Split source: {split_source}")
print(f"Image split manifest: {IMAGE_SPLIT_MANIFEST_CSV}")
print(f"Split compatibility manifest: {SPLIT_COMPATIBILITY_CSV}")

print("\nAIROGS-light split summary:")
print(split_summary.to_string(index=False))

print("\nImage-level label summary:")
print(split_label_summary.to_string(index=False))

print("\nAnatomical proxy compatibility summary:")
print(split_proxy_qc_summary.to_string(index=False))

print("\nSample leakage checks:")
print(json.dumps(leakage_checks, indent=2))

print(f"\nSplit config: {SPLIT_CONFIG_JSON}")

expected_split_sizes = {
    "train": 8000,
    "val": 770,
    "test": 770,
}

observed_split_sizes = (
    image_split_manifest
    .groupby("glaucoma_split")["sample_id"]
    .count()
    .to_dict()
)

for split_name, expected_size in expected_split_sizes.items():
    observed_size = int(observed_split_sizes.get(split_name, 0))

    if observed_size != expected_size:
        raise RuntimeError(
            f"Unexpected image count for {split_name}: "
            f"expected={expected_size}, observed={observed_size}"
        )

if len(image_split_manifest) != 9540:
    raise RuntimeError(f"Expected 9540 image split rows, found {len(image_split_manifest)}")

if missing_split_assignments > 0:
    raise RuntimeError(f"Missing split assignments: {missing_split_assignments}")

if missing_classifier_images > 0:
    raise RuntimeError(f"Missing classifier image files: {missing_classifier_images}")

if duplicate_sample_ids > 0:
    raise RuntimeError(f"Duplicate sample IDs found in image split manifest: {duplicate_sample_ids}")

if duplicate_file_names > 0:
    raise RuntimeError(f"Duplicate file names found in image split manifest: {duplicate_file_names}")

if any(value > 0 for value in leakage_checks.values()):
    raise RuntimeError(f"Sample leakage detected: {leakage_checks}")

for split_name in ["train", "val", "test"]:
    split_df = image_split_manifest[image_split_manifest["glaucoma_split"] == split_name]
    n_pos = int((split_df["gon_binary"] == 1).sum())
    n_neg = int((split_df["gon_binary"] == 0).sum())

    if n_pos != n_neg:
        raise RuntimeError(
            f"Expected balanced split for {split_name}, found positives={n_pos}, negatives={n_neg}"
        )

print("Status: PASS")

In [ ]:
# Cell 3: Configure classifier runtime and define the 10-model AIROGS-light benchmark plan

import sys
import subprocess
import importlib.util

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    brier_score_loss,
)

# Install timm only if it is not available in the current Colab runtime.
if importlib.util.find_spec("timm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])

import timm

SEED = int(PROJECT_CONFIG.get("seed", 42))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

CLASS_NAMES = {
    0: "NRG",
    1: "RG",
}

NUM_CLASSES = 1

# Screening stage: lower resolution to compare model families efficiently.
# Full-training stage: higher resolution for the selected candidates.
SCREENING_INPUT_SIZE = 224
FULL_INPUT_SIZE = 384

SCREENING_BATCH_SIZE = 16 if DEVICE.type == "cuda" else 4
FULL_BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2

NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0

SCREENING_MAX_EPOCHS = 12
SCREENING_EARLY_STOP_PATIENCE = 5

FULL_MAX_EPOCHS = 80
FULL_EARLY_STOP_PATIENCE = 15

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# AIROGS-light-v2 does not provide true patient IDs.
# Therefore, internal model selection must be image-level.
PRIMARY_SELECTION_METRIC = "image_val_auroc"

SECONDARY_SELECTION_METRICS = [
    "image_val_auprc",
    "image_val_balanced_accuracy",
    "image_val_brier_score",
    "image_val_accuracy",
]

USE_PRETRAINED_WEIGHTS = True

SCREENING_ROOT = CLASSIFIER_MODEL_ROOT / "architecture_screening"
FULL_TRAINING_ROOT = CLASSIFIER_MODEL_ROOT / "full_training"
FINAL_CLASSIFIER_ROOT = CLASSIFIER_MODEL_ROOT / "final_selected"

for path in [
    SCREENING_ROOT,
    FULL_TRAINING_ROOT,
    FINAL_CLASSIFIER_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

available_timm_models = set(timm.list_models())


def resolve_timm_model_name(candidates):
    for candidate in candidates:
        if candidate in available_timm_models:
            return candidate

    return None


MODEL_BENCHMARK_CANDIDATES = [
    {
        "model_name": "ResNet18",
        "family": "residual_cnn",
        "timm_candidates": ["resnet18"],
        "screening_role": "low_capacity_residual_baseline",
        "rationale": (
            "Tests whether a compact residual CNN is sufficient and provides a low-capacity "
            "control against larger models."
        ),
    },
    {
        "model_name": "ResNet50",
        "family": "residual_cnn",
        "timm_candidates": ["resnet50"],
        "screening_role": "canonical_residual_baseline",
        "rationale": (
            "Provides a standard residual CNN baseline commonly used in medical image "
            "classification and glaucoma screening studies."
        ),
    },
    {
        "model_name": "DenseNet121",
        "family": "dense_connectivity_cnn",
        "timm_candidates": ["densenet121"],
        "screening_role": "feature_reuse_baseline",
        "rationale": (
            "Evaluates dense feature reuse, which can be beneficial in medical image "
            "classification with moderate-size training sets."
        ),
    },
    {
        "model_name": "EfficientNetB0",
        "family": "compound_scaled_efficient_cnn",
        "timm_candidates": ["efficientnet_b0", "tf_efficientnet_b0"],
        "screening_role": "compact_compound_scaled_cnn",
        "rationale": (
            "Tests a compact compound-scaled CNN with a strong accuracy-efficiency trade-off."
        ),
    },
    {
        "model_name": "EfficientNetB2",
        "family": "compound_scaled_efficient_cnn",
        "timm_candidates": ["efficientnet_b2", "tf_efficientnet_b2"],
        "screening_role": "medium_compound_scaled_cnn",
        "rationale": (
            "Tests whether additional compound-scaled capacity improves over the compact "
            "EfficientNet baseline."
        ),
    },
    {
        "model_name": "ConvNeXtTiny",
        "family": "modern_convolutional_network",
        "timm_candidates": ["convnext_tiny", "convnext_tiny.fb_in22k_ft_in1k"],
        "screening_role": "modern_convnet_baseline",
        "rationale": (
            "Represents a modern convolutional design with improved normalization and "
            "large-kernel design choices while preserving CNN spatial priors."
        ),
    },
    {
        "model_name": "MobileNetV3Large",
        "family": "lightweight_deployable_cnn",
        "timm_candidates": ["mobilenetv3_large_100"],
        "screening_role": "deployment_oriented_baseline",
        "rationale": (
            "Provides a lightweight reference to test whether high performance requires "
            "large backbones."
        ),
    },
    {
        "model_name": "SwinTiny",
        "family": "window_attention_transformer",
        "timm_candidates": ["swin_tiny_patch4_window7_224", "swin_tiny_patch4_window7_224.ms_in1k"],
        "screening_role": "window_attention_transformer_baseline",
        "rationale": (
            "Tests whether local-window self-attention improves broader retinal context "
            "modeling beyond convolutional locality."
        ),
    },
    {
        "model_name": "DeiTSmall",
        "family": "data_efficient_vision_transformer",
        "timm_candidates": ["deit_small_patch16_224", "deit_small_patch16_224.fb_in1k"],
        "screening_role": "data_efficient_transformer_baseline",
        "rationale": (
            "Evaluates a data-efficient transformer designed for supervised transfer in "
            "moderate-size image datasets."
        ),
    },
    {
        "model_name": "DINOv2ViTS14",
        "family": "self_supervised_foundation_transformer",
        "timm_candidates": [
            "vit_small_patch14_dinov2.lvd142m",
            "vit_small_patch14_dinov2",
            "vit_small_patch14_reg4_dinov2.lvd142m",
        ],
        "screening_role": "self_supervised_representation_baseline",
        "rationale": (
            "Tests whether a self-supervised pretrained visual representation transfers "
            "to AIROGS-light glaucoma screening under the same image-level validation protocol."
        ),
    },
]

benchmark_rows = []

for item in MODEL_BENCHMARK_CANDIDATES:
    resolved_name = resolve_timm_model_name(item["timm_candidates"])

    benchmark_rows.append(
        {
            "model_name": item["model_name"],
            "family": item["family"],
            "timm_model_name": resolved_name,
            "candidate_names": "|".join(item["timm_candidates"]),
            "screening_role": item["screening_role"],
            "rationale": item["rationale"],
            "available_in_timm": resolved_name is not None,
            "include_in_screening": resolved_name is not None,
            "use_pretrained_weights": USE_PRETRAINED_WEIGHTS,
            "screening_input_size": SCREENING_INPUT_SIZE,
            "full_training_input_size": FULL_INPUT_SIZE,
        }
    )

MODEL_BENCHMARK_PLAN = pd.DataFrame(benchmark_rows)

missing_model_rows = MODEL_BENCHMARK_PLAN[
    ~MODEL_BENCHMARK_PLAN["available_in_timm"]
].copy()

MODEL_BENCHMARK_PLAN_CSV = RESULTS_07_DIR / "notebook07_model_benchmark_plan.csv"
MODEL_BENCHMARK_PLAN_JSON = RESULTS_07_DIR / "notebook07_model_benchmark_plan.json"
RUNTIME_CONFIG_JSON = RESULTS_07_DIR / "notebook07_runtime_config.json"

MODEL_BENCHMARK_PLAN.to_csv(MODEL_BENCHMARK_PLAN_CSV, index=False)

benchmark_plan_json = {
    "benchmark_design": "10-model screening grouped by architectural family",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "internal_split_policy": "official train/validation/test split from AIROGS-light-V2 metadata",
    "patient_id_available_for_internal_selection": False,
    "selection_principle": (
        "The model set covers complementary inductive biases: residual CNNs, dense CNNs, "
        "compound-scaled efficient CNNs, modern ConvNets, lightweight deployment-oriented CNNs, "
        "window-attention transformers, data-efficient transformers, and self-supervised "
        "foundation-style visual representations."
    ),
    "main_screening_policy": (
        "All available models are screened under the same official AIROGS-light train/validation split, "
        "same preprocessing, same optimizer family, same early stopping, and same primary image-level "
        "validation metric. The locked test split is not used for model selection."
    ),
    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
    "secondary_selection_metrics": SECONDARY_SELECTION_METRICS,
    "models": MODEL_BENCHMARK_PLAN.to_dict(orient="records"),
}

with open(MODEL_BENCHMARK_PLAN_JSON, "w", encoding="utf-8") as f:
    json.dump(benchmark_plan_json, f, indent=2)

runtime_config = {
    "notebook": "07_train_or_load_airogs_light_glaucoma_classifiers",
    "seed": SEED,
    "device": str(DEVICE),
    "cuda_available": bool(torch.cuda.is_available()),
    "torch_version": torch.__version__,
    "timm_version": getattr(timm, "__version__", "unknown"),
    "use_amp": bool(USE_AMP),
    "use_pretrained_weights": bool(USE_PRETRAINED_WEIGHTS),
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "screening_input_size": SCREENING_INPUT_SIZE,
    "full_input_size": FULL_INPUT_SIZE,
    "screening_batch_size": SCREENING_BATCH_SIZE,
    "full_batch_size": FULL_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "screening_max_epochs": SCREENING_MAX_EPOCHS,
    "screening_early_stop_patience": SCREENING_EARLY_STOP_PATIENCE,
    "full_max_epochs": FULL_MAX_EPOCHS,
    "full_early_stop_patience": FULL_EARLY_STOP_PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
    "secondary_selection_metrics": SECONDARY_SELECTION_METRICS,
    "screening_root": str(SCREENING_ROOT),
    "full_training_root": str(FULL_TRAINING_ROOT),
    "final_classifier_root": str(FINAL_CLASSIFIER_ROOT),
    "n_benchmark_models_requested": int(len(MODEL_BENCHMARK_PLAN)),
    "n_benchmark_models_available": int(MODEL_BENCHMARK_PLAN["available_in_timm"].sum()),
    "internal_patient_level_claim_allowed": False,
    "external_patient_level_evaluation_reserved_for": ["HYGD"],
    "external_quality_evaluation_reserved_for": ["HYGD"],
    "external_domain_shift_evaluation_reserved_for": ["HYGD", "PAPILA", "SMDG"],
}

with open(RUNTIME_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(runtime_config, f, indent=2)

print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"AMP enabled: {USE_AMP}")
print(f"torch: {torch.__version__}")
print(f"timm: {getattr(timm, '__version__', 'unknown')}")
print(f"Screening input size: {SCREENING_INPUT_SIZE}")
print(f"Full-training input size: {FULL_INPUT_SIZE}")
print(f"Screening batch size: {SCREENING_BATCH_SIZE}")
print(f"Full-training batch size: {FULL_BATCH_SIZE}")
print(f"Primary selection metric: {PRIMARY_SELECTION_METRIC}")

print("\nModel benchmark plan:")
print(
    MODEL_BENCHMARK_PLAN[
        [
            "model_name",
            "family",
            "timm_model_name",
            "screening_role",
            "available_in_timm",
        ]
    ].to_string(index=False)
)

print(f"\nModel benchmark plan CSV: {MODEL_BENCHMARK_PLAN_CSV}")
print(f"Model benchmark plan JSON: {MODEL_BENCHMARK_PLAN_JSON}")
print(f"Runtime config: {RUNTIME_CONFIG_JSON}")

if len(MODEL_BENCHMARK_PLAN) != 10:
    raise RuntimeError(f"Expected 10 benchmark models, found {len(MODEL_BENCHMARK_PLAN)}")

if len(missing_model_rows) > 0:
    print("\nUnavailable models:")
    print(missing_model_rows[["model_name", "candidate_names"]].to_string(index=False))

    raise RuntimeError(
        "One or more requested benchmark models are unavailable in the installed timm version. "
        "Update timm or replace the missing model names before proceeding."
    )

if MODEL_BENCHMARK_PLAN["family"].nunique() < 7:
    raise RuntimeError("Expected at least 7 architectural families in the benchmark plan.")

print("Status: PASS")

In [ ]:
# Cell 4: Define AIROGS-light classification dataset and dataloader smoke test

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

if "IMAGE_SPLIT_MANIFEST_CSV" not in globals():
    IMAGE_SPLIT_MANIFEST_CSV = SPLIT_DIR / f"airogs_light_image_level_split_manifest_seed{SEED}.csv"

if not Path(IMAGE_SPLIT_MANIFEST_CSV).exists():
    raise FileNotFoundError(f"Missing image split manifest: {IMAGE_SPLIT_MANIFEST_CSV}")

image_split_manifest = pd.read_csv(IMAGE_SPLIT_MANIFEST_CSV)

required_loader_columns = [
    "sample_id",
    "patient_id",
    "glaucoma_split",
    "gon_binary",
    "gon_label",
    "classifier_image_path",
    "quality_score",
    "include_strict_anatomical_analysis",
    "include_high_confidence_anatomical_analysis",
    "low_confidence_anatomical_proxy",
    "excluded_from_anatomical_analysis",
]

missing_loader_columns = [
    col for col in required_loader_columns
    if col not in image_split_manifest.columns
]

if missing_loader_columns:
    raise ValueError(f"Missing required dataloader columns: {missing_loader_columns}")

image_split_manifest = image_split_manifest.copy().reset_index(drop=True)

image_split_manifest["sample_id"] = image_split_manifest["sample_id"].astype(str)
image_split_manifest["patient_id"] = image_split_manifest["patient_id"].astype(str)
image_split_manifest["glaucoma_split"] = image_split_manifest["glaucoma_split"].astype(str)
image_split_manifest["gon_binary"] = pd.to_numeric(
    image_split_manifest["gon_binary"],
    errors="coerce",
).astype(int)
image_split_manifest["gon_label"] = image_split_manifest["gon_label"].astype(str)
image_split_manifest["classifier_image_path"] = image_split_manifest["classifier_image_path"].astype(str)
image_split_manifest["quality_score"] = pd.to_numeric(
    image_split_manifest["quality_score"],
    errors="coerce",
).astype(float)

for bool_col in [
    "include_strict_anatomical_analysis",
    "include_high_confidence_anatomical_analysis",
    "low_confidence_anatomical_proxy",
    "excluded_from_anatomical_analysis",
    "true_patient_id_available",
    "quality_score_available",
    "anatomical_proxy_available",
]:
    if bool_col in image_split_manifest.columns:
        image_split_manifest[bool_col] = as_bool_series(image_split_manifest[bool_col])

image_split_manifest["classifier_image_exists"] = image_split_manifest["classifier_image_path"].apply(
    lambda x: Path(str(x)).exists()
)

missing_images = int((~image_split_manifest["classifier_image_exists"]).sum())
duplicate_sample_ids = int(image_split_manifest["sample_id"].duplicated().sum())

if missing_images > 0:
    missing_preview = image_split_manifest.loc[
        ~image_split_manifest["classifier_image_exists"],
        ["sample_id", "classifier_image_path"],
    ].head(20)

    print("Missing image preview:")
    print(missing_preview.to_string(index=False))

    raise FileNotFoundError(f"Missing classifier images in split manifest: {missing_images}")

if duplicate_sample_ids > 0:
    raise RuntimeError(f"Duplicate sample IDs found: {duplicate_sample_ids}")

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def pad_to_square_rgb(image_rgb, pad_value=0):
    height, width = image_rgb.shape[:2]

    if height == width:
        return image_rgb

    side = max(height, width)

    padded = np.full(
        (side, side, 3),
        fill_value=pad_value,
        dtype=image_rgb.dtype,
    )

    y0 = (side - height) // 2
    x0 = (side - width) // 2

    padded[y0:y0 + height, x0:x0 + width] = image_rgb

    return padded


def resize_rgb(image_rgb, input_size):
    return cv2.resize(
        image_rgb,
        (int(input_size), int(input_size)),
        interpolation=cv2.INTER_AREA,
    )


def apply_train_augmentation(image_rgb):
    image = image_rgb.copy()

    if np.random.rand() < 0.5:
        image = np.fliplr(image).copy()

    if np.random.rand() < 0.35:
        angle = float(np.random.uniform(-12.0, 12.0))
        height, width = image.shape[:2]
        center = (width / 2.0, height / 2.0)

        rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)

        image = cv2.warpAffine(
            image,
            rotation_matrix,
            (width, height),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT_101,
        )

    if np.random.rand() < 0.35:
        alpha = float(np.random.uniform(0.88, 1.12))
        beta = float(np.random.uniform(-10.0, 10.0))

        image = image.astype(np.float32) * alpha + beta
        image = np.clip(image, 0, 255).astype(np.uint8)

    if np.random.rand() < 0.20:
        gamma = float(np.random.uniform(0.85, 1.15))
        image_float = image.astype(np.float32) / 255.0
        image_float = np.power(np.clip(image_float, 0.0, 1.0), gamma)
        image = np.clip(image_float * 255.0, 0, 255).astype(np.uint8)

    return image


def preprocess_classifier_image(image_bgr, input_size, augment=False):
    if image_bgr is None:
        raise ValueError("Received empty image.")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_rgb = pad_to_square_rgb(image_rgb, pad_value=0)
    image_rgb = resize_rgb(image_rgb, input_size=input_size)

    if augment:
        image_rgb = apply_train_augmentation(image_rgb)

    image = image_rgb.astype(np.float32) / 255.0
    image = (image - IMAGENET_MEAN) / IMAGENET_STD
    image = np.transpose(image, (2, 0, 1))

    image_tensor = torch.from_numpy(np.ascontiguousarray(image)).float()

    return image_tensor


class AIROGSLightGlaucomaClassificationDataset(Dataset):
    def __init__(self, dataframe, input_size, augment=False):
        self.df = dataframe.reset_index(drop=True).copy()
        self.input_size = int(input_size)
        self.augment = bool(augment)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(str(row["classifier_image_path"]))
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

        if image_bgr is None:
            raise RuntimeError(f"Unreadable image: {image_path}")

        image_tensor = preprocess_classifier_image(
            image_bgr=image_bgr,
            input_size=self.input_size,
            augment=self.augment,
        )

        label = float(row["gon_binary"])

        return {
            "image": image_tensor,
            "label": torch.tensor(label, dtype=torch.float32),
            "sample_id": row["sample_id"],
            "patient_id": row["patient_id"],
            "glaucoma_split": row["glaucoma_split"],
            "quality_score": torch.tensor(float(row["quality_score"]), dtype=torch.float32),
            "strict_anatomical_proxy_valid": bool(row["include_strict_anatomical_analysis"]),
            "high_confidence_anatomical_proxy": bool(row["include_high_confidence_anatomical_analysis"]),
            "low_confidence_anatomical_proxy": bool(row["low_confidence_anatomical_proxy"]),
            "failed_anatomical_proxy": bool(row["excluded_from_anatomical_analysis"]),
            "image_path": str(image_path),
        }


def collate_airogs_light_classification_batch(batch):
    return {
        "image": torch.stack([item["image"] for item in batch], dim=0),
        "label": torch.stack([item["label"] for item in batch], dim=0),
        "sample_id": [item["sample_id"] for item in batch],
        "patient_id": [item["patient_id"] for item in batch],
        "glaucoma_split": [item["glaucoma_split"] for item in batch],
        "quality_score": torch.stack([item["quality_score"] for item in batch], dim=0),
        "strict_anatomical_proxy_valid": [item["strict_anatomical_proxy_valid"] for item in batch],
        "high_confidence_anatomical_proxy": [item["high_confidence_anatomical_proxy"] for item in batch],
        "low_confidence_anatomical_proxy": [item["low_confidence_anatomical_proxy"] for item in batch],
        "failed_anatomical_proxy": [item["failed_anatomical_proxy"] for item in batch],
        "image_path": [item["image_path"] for item in batch],
    }


def get_classifier_split(split_name):
    subset = image_split_manifest[
        image_split_manifest["glaucoma_split"] == split_name
    ].copy()

    if len(subset) == 0:
        raise RuntimeError(f"Empty classifier split: {split_name}")

    return subset.reset_index(drop=True)


def make_balanced_train_sampler(dataframe):
    labels = dataframe["gon_binary"].astype(int).values
    class_counts = pd.Series(labels).value_counts().to_dict()

    if len(class_counts) != 2:
        raise RuntimeError(f"Expected binary labels in training sampler, found: {class_counts}")

    sample_weights = np.array(
        [
            1.0 / class_counts[int(label)]
            for label in labels
        ],
        dtype=np.float64,
    )

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )

    return sampler


def make_classifier_loader(
    dataframe,
    input_size,
    batch_size,
    split_name,
    augment=False,
    balanced_sampler=False,
):
    dataset = AIROGSLightGlaucomaClassificationDataset(
        dataframe=dataframe,
        input_size=input_size,
        augment=augment,
    )

    sampler = None
    shuffle = False

    if split_name == "train":
        if balanced_sampler:
            sampler = make_balanced_train_sampler(dataframe)
            shuffle = False
        else:
            sampler = None
            shuffle = True

    loader = DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=shuffle,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        drop_last=False,
        collate_fn=collate_airogs_light_classification_batch,
    )

    return loader


train_df = get_classifier_split("train")
val_df = get_classifier_split("val")
test_df = get_classifier_split("test")

train_loader_smoke = make_classifier_loader(
    dataframe=train_df,
    input_size=SCREENING_INPUT_SIZE,
    batch_size=SCREENING_BATCH_SIZE,
    split_name="train",
    augment=True,
    balanced_sampler=True,
)

val_loader_smoke = make_classifier_loader(
    dataframe=val_df,
    input_size=SCREENING_INPUT_SIZE,
    batch_size=SCREENING_BATCH_SIZE,
    split_name="val",
    augment=False,
    balanced_sampler=False,
)

test_loader_smoke = make_classifier_loader(
    dataframe=test_df,
    input_size=SCREENING_INPUT_SIZE,
    batch_size=SCREENING_BATCH_SIZE,
    split_name="test",
    augment=False,
    balanced_sampler=False,
)

train_batch = next(iter(train_loader_smoke))
val_batch = next(iter(val_loader_smoke))
test_batch = next(iter(test_loader_smoke))

loader_smoke_rows = []

for split_name, split_df, batch in [
    ("train", train_df, train_batch),
    ("val", val_df, val_batch),
    ("test", test_df, test_batch),
]:
    labels = split_df["gon_binary"].astype(int)

    loader_smoke_rows.append(
        {
            "split": split_name,
            "n_images": int(len(split_df)),
            "n_pseudo_patients": int(split_df["patient_id"].nunique()),
            "n_rg_images": int((labels == 1).sum()),
            "n_nrg_images": int((labels == 0).sum()),
            "batch_image_shape": str(tuple(batch["image"].shape)),
            "batch_label_shape": str(tuple(batch["label"].shape)),
            "batch_label_min": float(batch["label"].min().item()),
            "batch_label_max": float(batch["label"].max().item()),
            "batch_quality_min": float(batch["quality_score"].min().item()),
            "batch_quality_max": float(batch["quality_score"].max().item()),
            "batch_failed_proxy_count": int(sum(batch["failed_anatomical_proxy"])),
            "batch_low_confidence_count": int(sum(batch["low_confidence_anatomical_proxy"])),
        }
    )

loader_smoke_test = pd.DataFrame(loader_smoke_rows)

loader_smoke_csv = RESULTS_07_DIR / "notebook07_dataloader_smoke_test.csv"
loader_smoke_test.to_csv(loader_smoke_csv, index=False)

print("Dataloader smoke test:")
print(loader_smoke_test.to_string(index=False))
print(f"Dataloader smoke test CSV: {loader_smoke_csv}")

expected_split_sizes = {
    "train": 8000,
    "val": 770,
    "test": 770,
}

for split_name, expected_size in expected_split_sizes.items():
    observed_size = int(
        loader_smoke_test.loc[
            loader_smoke_test["split"] == split_name,
            "n_images",
        ].iloc[0]
    )

    if observed_size != expected_size:
        raise RuntimeError(
            f"Unexpected image count for {split_name}: "
            f"expected={expected_size}, observed={observed_size}"
        )

for _, row in loader_smoke_test.iterrows():
    shape = eval(row["batch_image_shape"])

    if len(shape) != 4:
        raise RuntimeError(f"Unexpected image batch rank for {row['split']}: {shape}")

    if shape[1:] != (3, SCREENING_INPUT_SIZE, SCREENING_INPUT_SIZE):
        raise RuntimeError(f"Unexpected image tensor shape for {row['split']}: {shape}")

    if shape[0] > SCREENING_BATCH_SIZE:
        raise RuntimeError(f"Batch size exceeds configured batch size for {row['split']}: {shape}")

if not set(loader_smoke_test["batch_label_min"]).issubset({0.0, 1.0}):
    raise RuntimeError("Unexpected label minimum found.")

if not set(loader_smoke_test["batch_label_max"]).issubset({0.0, 1.0}):
    raise RuntimeError("Unexpected label maximum found.")

if int(loader_smoke_test["batch_failed_proxy_count"].sum()) != 0:
    raise RuntimeError("AIROGS-light training split should not have failed anatomical proxies.")

if int(loader_smoke_test["batch_low_confidence_count"].sum()) != 0:
    raise RuntimeError("AIROGS-light training split should not have low-confidence anatomical proxies.")

print("Status: PASS")

In [ ]:
# Cell 4B: Cache AIROGS-light images to local Colab disk with visible progress

from pathlib import Path
import shutil
import time

try:
    from tqdm.auto import tqdm
except Exception:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm

USE_LOCAL_IMAGE_CACHE = True

LOCAL_CACHE_PARENT = Path("/content/airogs_light_v2_cache")
LOCAL_AIROGS_LIGHT_ROOT = LOCAL_CACHE_PARENT / "eyepac-light-v2-512-jpg"

DRIVE_AIROGS_LIGHT_ROOT = Path(AIROGS_LIGHT_ROOT)

if not USE_LOCAL_IMAGE_CACHE:
    print("Local cache disabled. Training will read images directly from Drive.")
    print("Status: PASS")

else:
    if "image_split_manifest" not in globals():
        if not Path(IMAGE_SPLIT_MANIFEST_CSV).exists():
            raise FileNotFoundError(f"Missing image split manifest: {IMAGE_SPLIT_MANIFEST_CSV}")

        image_split_manifest = pd.read_csv(IMAGE_SPLIT_MANIFEST_CSV)

    image_split_manifest = image_split_manifest.copy().reset_index(drop=True)

    if "classifier_image_path" not in image_split_manifest.columns:
        raise ValueError("Missing classifier_image_path column in image_split_manifest.")

    LOCAL_CACHE_PARENT.mkdir(parents=True, exist_ok=True)
    LOCAL_AIROGS_LIGHT_ROOT.mkdir(parents=True, exist_ok=True)

    print(f"Drive dataset root: {DRIVE_AIROGS_LIGHT_ROOT}")
    print(f"Local cache root: {LOCAL_AIROGS_LIGHT_ROOT}")

    if not DRIVE_AIROGS_LIGHT_ROOT.exists():
        raise FileNotFoundError(f"Missing Drive dataset root: {DRIVE_AIROGS_LIGHT_ROOT}")

    # Copy metadata for traceability.
    drive_metadata_path = DRIVE_AIROGS_LIGHT_ROOT / "metadata.csv"
    local_metadata_path = LOCAL_AIROGS_LIGHT_ROOT / "metadata.csv"

    if drive_metadata_path.exists():
        shutil.copyfile(str(drive_metadata_path), str(local_metadata_path))

    # Build source-target copy table directly from the manifest.
    copy_rows = []

    for source_path_value in image_split_manifest["classifier_image_path"].astype(str).tolist():
        source_path = Path(source_path_value)

        if not source_path.exists():
            raise FileNotFoundError(f"Missing source image in Drive: {source_path}")

        try:
            relative_path = source_path.relative_to(DRIVE_AIROGS_LIGHT_ROOT)
        except ValueError:
            raise ValueError(
                "Image path is not inside DRIVE_AIROGS_LIGHT_ROOT. "
                f"Image path: {source_path} | Drive root: {DRIVE_AIROGS_LIGHT_ROOT}"
            )

        target_path = LOCAL_AIROGS_LIGHT_ROOT / relative_path

        copy_rows.append(
            {
                "source_path": source_path,
                "target_path": target_path,
                "relative_path": relative_path,
            }
        )

    n_total = len(copy_rows)

    if n_total != 9540:
        raise RuntimeError(f"Expected 9540 image paths to cache, found {n_total}")

    # Determine which files still need to be copied.
    pending_rows = []

    for row in copy_rows:
        source_path = row["source_path"]
        target_path = row["target_path"]

        if target_path.exists() and target_path.stat().st_size == source_path.stat().st_size:
            continue

        pending_rows.append(row)

    print(f"Total manifest images: {n_total}")
    print(f"Already cached images: {n_total - len(pending_rows)}")
    print(f"Pending images to copy: {len(pending_rows)}")

    start_time = time.time()

    for row in tqdm(pending_rows, desc="Caching AIROGS-light images", unit="img"):
        source_path = row["source_path"]
        target_path = row["target_path"]

        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(str(source_path), str(target_path))

    elapsed_min = (time.time() - start_time) / 60.0

    # Verify local cache completeness using the manifest, not a broad rglob.
    missing_local_rows = []

    local_paths = []

    for row in copy_rows:
        target_path = row["target_path"]
        local_paths.append(str(target_path))

        if not target_path.exists():
            missing_local_rows.append(
                {
                    "source_path": str(row["source_path"]),
                    "target_path": str(target_path),
                }
            )

    if len(missing_local_rows) > 0:
        missing_df = pd.DataFrame(missing_local_rows)
        print("Missing local image preview:")
        print(missing_df.head(20).to_string(index=False))
        raise FileNotFoundError(f"Missing local cached images: {len(missing_local_rows)}")

    # Update manifests in memory to point to /content instead of Drive.
    source_to_local = {
        str(row["source_path"]): str(row["target_path"])
        for row in copy_rows
    }

    image_split_manifest["classifier_image_path"] = image_split_manifest["classifier_image_path"].astype(str).map(source_to_local)
    image_split_manifest["image_path"] = image_split_manifest["classifier_image_path"]

    image_split_manifest["classifier_image_exists"] = image_split_manifest["classifier_image_path"].apply(
        lambda x: Path(str(x)).exists()
    )

    missing_after_update = int((~image_split_manifest["classifier_image_exists"]).sum())

    if missing_after_update > 0:
        raise RuntimeError(f"Missing local images after manifest update: {missing_after_update}")

    # Refresh split dataframes used by the training loop.
    train_df = (
        image_split_manifest[image_split_manifest["glaucoma_split"] == "train"]
        .copy()
        .reset_index(drop=True)
    )

    val_df = (
        image_split_manifest[image_split_manifest["glaucoma_split"] == "val"]
        .copy()
        .reset_index(drop=True)
    )

    test_df = (
        image_split_manifest[image_split_manifest["glaucoma_split"] == "test"]
        .copy()
        .reset_index(drop=True)
    )

    LOCAL_CACHE_MANIFEST_CSV = RESULTS_07_DIR / "notebook07_airogs_light_local_cache_split_manifest.csv"
    image_split_manifest.to_csv(LOCAL_CACHE_MANIFEST_CSV, index=False)

    print("Local cache enabled.")
    print(f"Elapsed copy time: {elapsed_min:.2f} min")
    print(f"Train images: {len(train_df)}")
    print(f"Val images: {len(val_df)}")
    print(f"Test images: {len(test_df)}")
    print(f"Local-cache manifest: {LOCAL_CACHE_MANIFEST_CSV}")
    print("Status: PASS")

In [ ]:
# Cell 5: Define classifier model registry and run AIROGS-light 10-model smoke test

if "MODEL_BENCHMARK_PLAN" not in globals():
    MODEL_BENCHMARK_PLAN_CSV = RESULTS_07_DIR / "notebook07_model_benchmark_plan.csv"

    if not MODEL_BENCHMARK_PLAN_CSV.exists():
        raise FileNotFoundError(f"Missing model benchmark plan: {MODEL_BENCHMARK_PLAN_CSV}")

    MODEL_BENCHMARK_PLAN = pd.read_csv(MODEL_BENCHMARK_PLAN_CSV)

required_model_plan_columns = [
    "model_name",
    "family",
    "timm_model_name",
    "available_in_timm",
    "include_in_screening",
]

missing_model_plan_columns = [
    col for col in required_model_plan_columns
    if col not in MODEL_BENCHMARK_PLAN.columns
]

if missing_model_plan_columns:
    raise ValueError(f"Missing model benchmark plan columns: {missing_model_plan_columns}")

MODEL_BENCHMARK_PLAN = MODEL_BENCHMARK_PLAN.copy()

if "as_bool_series" not in globals():
    def as_bool_series(series):
        if series.dtype == bool:
            return series

        return (
            series.astype(str)
            .str.strip()
            .str.lower()
            .map(
                {
                    "true": True,
                    "1": True,
                    "yes": True,
                    "y": True,
                    "false": False,
                    "0": False,
                    "no": False,
                    "n": False,
                    "nan": False,
                    "none": False,
                    "": False,
                }
            )
            .fillna(False)
            .astype(bool)
        )

MODEL_BENCHMARK_PLAN["available_in_timm"] = as_bool_series(
    MODEL_BENCHMARK_PLAN["available_in_timm"]
)

MODEL_BENCHMARK_PLAN["include_in_screening"] = as_bool_series(
    MODEL_BENCHMARK_PLAN["include_in_screening"]
)

SCREENING_MODEL_NAMES = MODEL_BENCHMARK_PLAN.loc[
    MODEL_BENCHMARK_PLAN["include_in_screening"],
    "model_name",
].tolist()

MODEL_NAME_TO_TIMM = dict(
    zip(
        MODEL_BENCHMARK_PLAN["model_name"],
        MODEL_BENCHMARK_PLAN["timm_model_name"],
    )
)

MODEL_NAME_TO_FAMILY = dict(
    zip(
        MODEL_BENCHMARK_PLAN["model_name"],
        MODEL_BENCHMARK_PLAN["family"],
    )
)


def count_trainable_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def count_total_parameters(model):
    return int(sum(p.numel() for p in model.parameters()))


def create_timm_binary_classifier(timm_model_name, pretrained, input_size):
    create_kwargs = {
        "pretrained": bool(pretrained),
        "num_classes": 1,
    }

    # Transformer-like models may accept img_size explicitly.
    # CNNs usually reject it, so this fallback keeps the registry generic.
    try:
        model = timm.create_model(
            timm_model_name,
            img_size=int(input_size),
            **create_kwargs,
        )
    except TypeError:
        model = timm.create_model(
            timm_model_name,
            **create_kwargs,
        )

    return model


def build_classifier_model(
    model_name,
    pretrained=USE_PRETRAINED_WEIGHTS,
    input_size=SCREENING_INPUT_SIZE,
):
    if model_name not in MODEL_NAME_TO_TIMM:
        raise ValueError(f"Unknown classifier model: {model_name}")

    timm_model_name = MODEL_NAME_TO_TIMM[model_name]

    if pd.isna(timm_model_name) or str(timm_model_name).strip() == "":
        raise ValueError(f"Missing timm model name for {model_name}")

    model = create_timm_binary_classifier(
        timm_model_name=str(timm_model_name),
        pretrained=pretrained,
        input_size=input_size,
    )

    return model


def amp_autocast_context():
    return torch.amp.autocast(
        device_type="cuda",
        enabled=USE_AMP,
    )


def make_grad_scaler():
    return torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )


criterion = nn.BCEWithLogitsLoss()

# Smoke test uses pretrained=False to avoid forcing weight downloads here.
# Real training will use USE_PRETRAINED_WEIGHTS from Cell 3.
MODEL_SMOKE_USE_PRETRAINED = False
MODEL_SMOKE_BATCH_SIZE = 1
RUN_BACKWARD_IN_SMOKE = DEVICE.type == "cuda"

if "train_df" not in globals():
    train_df = get_classifier_split("train")

smoke_df = train_df.head(MODEL_SMOKE_BATCH_SIZE).copy()

smoke_loader = make_classifier_loader(
    dataframe=smoke_df,
    input_size=SCREENING_INPUT_SIZE,
    batch_size=MODEL_SMOKE_BATCH_SIZE,
    split_name="train",
    augment=False,
    balanced_sampler=False,
)

smoke_batch = next(iter(smoke_loader))

smoke_images = smoke_batch["image"].to(DEVICE, non_blocking=True)
smoke_labels = smoke_batch["label"].to(DEVICE, non_blocking=True)

model_smoke_rows = []

for model_name in SCREENING_MODEL_NAMES:
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    timm_model_name = MODEL_NAME_TO_TIMM[model_name]
    model_family = MODEL_NAME_TO_FAMILY[model_name]

    start_time = time.time()

    try:
        model = build_classifier_model(
            model_name=model_name,
            pretrained=MODEL_SMOKE_USE_PRETRAINED,
            input_size=SCREENING_INPUT_SIZE,
        ).to(DEVICE)

        model.train()

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

        optimizer.zero_grad(set_to_none=True)

        with amp_autocast_context():
            logits = model(smoke_images)

            if logits.ndim == 2 and logits.shape[1] == 1:
                logits = logits[:, 0]
            elif logits.ndim == 1:
                pass
            else:
                raise RuntimeError(
                    f"Unexpected logits shape for {model_name}: {tuple(logits.shape)}"
                )

            loss = criterion(logits, smoke_labels)

        if RUN_BACKWARD_IN_SMOKE:
            scaler = make_grad_scaler()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            scaler = None

        elapsed_sec = time.time() - start_time

        model_smoke_rows.append(
            {
                "model_name": model_name,
                "family": model_family,
                "timm_model_name": timm_model_name,
                "input_shape": str(tuple(smoke_images.shape)),
                "logits_shape": str(tuple(logits.shape)),
                "loss": float(loss.detach().cpu()),
                "total_parameters": count_total_parameters(model),
                "trainable_parameters": count_trainable_parameters(model),
                "pretrained_used_in_smoke": bool(MODEL_SMOKE_USE_PRETRAINED),
                "actual_training_use_pretrained": bool(USE_PRETRAINED_WEIGHTS),
                "backward_tested": bool(RUN_BACKWARD_IN_SMOKE),
                "elapsed_sec": round(elapsed_sec, 4),
                "status": "pass",
                "error": "",
            }
        )

        del model, optimizer, logits, loss, scaler

    except Exception as exc:
        elapsed_sec = time.time() - start_time

        model_smoke_rows.append(
            {
                "model_name": model_name,
                "family": model_family,
                "timm_model_name": timm_model_name,
                "input_shape": str(tuple(smoke_images.shape)),
                "logits_shape": "",
                "loss": np.nan,
                "total_parameters": np.nan,
                "trainable_parameters": np.nan,
                "pretrained_used_in_smoke": bool(MODEL_SMOKE_USE_PRETRAINED),
                "actual_training_use_pretrained": bool(USE_PRETRAINED_WEIGHTS),
                "backward_tested": bool(RUN_BACKWARD_IN_SMOKE),
                "elapsed_sec": round(elapsed_sec, 4),
                "status": "fail",
                "error": repr(exc),
            }
        )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

model_smoke_test = pd.DataFrame(model_smoke_rows)

MODEL_SMOKE_TEST_CSV = RESULTS_07_DIR / "notebook07_model_registry_smoke_test.csv"
MODEL_REGISTRY_CONFIG_JSON = RESULTS_07_DIR / "notebook07_model_registry_config.json"

model_smoke_test.to_csv(MODEL_SMOKE_TEST_CSV, index=False)

model_registry_config = {
    "n_screening_models": int(len(SCREENING_MODEL_NAMES)),
    "screening_models": SCREENING_MODEL_NAMES,
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "internal_split_policy": "official_train_val_test_image_level",
    "patient_id_available_for_internal_training": False,
    "patient_level_internal_claim_allowed": False,
    "model_smoke_use_pretrained": bool(MODEL_SMOKE_USE_PRETRAINED),
    "actual_training_use_pretrained": bool(USE_PRETRAINED_WEIGHTS),
    "screening_input_size": int(SCREENING_INPUT_SIZE),
    "model_smoke_batch_size": int(MODEL_SMOKE_BATCH_SIZE),
    "run_backward_in_smoke": bool(RUN_BACKWARD_IN_SMOKE),
    "criterion": "BCEWithLogitsLoss",
    "note": (
        "The smoke test uses pretrained=False to verify architecture construction, "
        "forward tensor shape, and optional backward pass without forcing weight downloads. "
        "Training functions will use USE_PRETRAINED_WEIGHTS."
    ),
}

with open(MODEL_REGISTRY_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(model_registry_config, f, indent=2)

print("Model smoke test:")
print(
    model_smoke_test[
        [
            "model_name",
            "family",
            "timm_model_name",
            "logits_shape",
            "loss",
            "trainable_parameters",
            "backward_tested",
            "elapsed_sec",
            "status",
        ]
    ].to_string(index=False)
)

print(f"\nModel smoke test CSV: {MODEL_SMOKE_TEST_CSV}")
print(f"Model registry config: {MODEL_REGISTRY_CONFIG_JSON}")

if len(model_smoke_test) != 10:
    raise RuntimeError(f"Expected 10 model smoke-test rows, found {len(model_smoke_test)}")

failed_models = model_smoke_test.loc[
    model_smoke_test["status"] != "pass",
    ["model_name", "error"],
]

if len(failed_models) > 0:
    print("\nFailed models:")
    print(failed_models.to_string(index=False))
    raise RuntimeError("One or more model smoke tests failed.")

if not np.isfinite(model_smoke_test["loss"]).all():
    raise RuntimeError("Non-finite loss found in model smoke test.")

expected_logits_shape = str((MODEL_SMOKE_BATCH_SIZE,))

if not model_smoke_test["logits_shape"].eq(expected_logits_shape).all():
    raise RuntimeError("Unexpected logits shape in one or more models.")

if int(model_smoke_test["trainable_parameters"].isna().sum()) > 0:
    raise RuntimeError("Missing trainable parameter count in model smoke test.")

print("Status: PASS")

In [ ]:
# Cell 6: Define AIROGS-light classifier training, evaluation, and checkpoint utilities

def flatten_logits(logits):
    if logits.ndim == 2 and logits.shape[1] == 1:
        return logits[:, 0]

    if logits.ndim == 1:
        return logits

    raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")


def sigmoid_numpy(logits):
    logits = np.asarray(logits, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-logits))


def compute_binary_classification_metrics(y_true, y_prob, threshold=0.5, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    valid_mask = np.isfinite(y_prob)
    y_true = y_true[valid_mask]
    y_prob = y_prob[valid_mask]

    if len(y_true) == 0:
        raise ValueError("No valid samples for metric computation.")

    y_pred = (y_prob >= threshold).astype(int)

    metrics = {}

    if len(np.unique(y_true)) == 2:
        metrics[f"{prefix}auroc"] = float(roc_auc_score(y_true, y_prob))
        metrics[f"{prefix}auprc"] = float(average_precision_score(y_true, y_prob))
    else:
        metrics[f"{prefix}auroc"] = np.nan
        metrics[f"{prefix}auprc"] = np.nan

    metrics[f"{prefix}accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics[f"{prefix}balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
    metrics[f"{prefix}brier_score"] = float(brier_score_loss(y_true, y_prob))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    metrics[f"{prefix}precision"] = float(precision)
    metrics[f"{prefix}recall_sensitivity"] = float(recall)
    metrics[f"{prefix}f1"] = float(f1)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    metrics[f"{prefix}specificity"] = float(specificity)
    metrics[f"{prefix}tn"] = int(tn)
    metrics[f"{prefix}fp"] = int(fp)
    metrics[f"{prefix}fn"] = int(fn)
    metrics[f"{prefix}tp"] = int(tp)
    metrics[f"{prefix}threshold"] = float(threshold)
    metrics[f"{prefix}n_samples"] = int(len(y_true))
    metrics[f"{prefix}n_positive"] = int((y_true == 1).sum())
    metrics[f"{prefix}n_negative"] = int((y_true == 0).sum())

    return metrics


def aggregate_predictions_to_group_level(
    image_predictions,
    group_column,
    method="mean",
    output_prefix="group",
):
    required_columns = [
        group_column,
        "y_true",
        "y_prob",
        "quality_score",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in image_predictions.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing prediction columns for group aggregation: {missing_columns}")

    group_rows = []

    for group_id, group_df in image_predictions.groupby(group_column):
        labels = sorted(group_df["y_true"].dropna().astype(int).unique().tolist())

        if len(labels) != 1:
            raise RuntimeError(f"Mixed or missing labels inside group: {group_id}")

        probs = pd.to_numeric(group_df["y_prob"], errors="coerce").astype(float).values
        quality = pd.to_numeric(group_df["quality_score"], errors="coerce").astype(float).values

        if method == "mean":
            group_prob = float(np.nanmean(probs))

        elif method == "max":
            group_prob = float(np.nanmax(probs))

        elif method == "quality_weighted_mean":
            weights = np.nan_to_num(quality, nan=np.nanmedian(quality))

            if not np.isfinite(weights).all() or np.sum(weights) <= 0:
                group_prob = float(np.nanmean(probs))
            else:
                weights = np.clip(weights, a_min=0.0, a_max=None)
                group_prob = float(np.average(probs, weights=weights))

        else:
            raise ValueError(f"Unknown group aggregation method: {method}")

        group_rows.append(
            {
                f"{output_prefix}_id": str(group_id),
                "y_true": int(labels[0]),
                "y_prob": group_prob,
                "aggregation_method": method,
                "n_images": int(len(group_df)),
                "mean_quality_score": float(np.nanmean(quality)),
                "n_strict_valid_proxy_images": int(group_df["strict_anatomical_proxy_valid"].sum())
                if "strict_anatomical_proxy_valid" in group_df.columns
                else np.nan,
                "n_high_confidence_proxy_images": int(group_df["high_confidence_anatomical_proxy"].sum())
                if "high_confidence_anatomical_proxy" in group_df.columns
                else np.nan,
                "n_low_confidence_proxy_images": int(group_df["low_confidence_anatomical_proxy"].sum())
                if "low_confidence_anatomical_proxy" in group_df.columns
                else np.nan,
                "n_failed_proxy_images": int(group_df["failed_anatomical_proxy"].sum())
                if "failed_anatomical_proxy" in group_df.columns
                else np.nan,
            }
        )

    return pd.DataFrame(group_rows)


def aggregate_predictions_to_patient_level(image_predictions, method="mean"):
    # Compatibility wrapper.
    # In AIROGS-light-v2 this is a pseudo-patient aggregation because there is no true patient_id.
    return aggregate_predictions_to_group_level(
        image_predictions=image_predictions,
        group_column="patient_id",
        method=method,
        output_prefix="patient",
    )


def evaluate_prediction_dataframe(
    image_predictions,
    threshold=0.5,
    compute_pseudo_patient_metrics=True,
):
    image_metrics = compute_binary_classification_metrics(
        y_true=image_predictions["y_true"].values,
        y_prob=image_predictions["y_prob"].values,
        threshold=threshold,
        prefix="image_",
    )

    pseudo_patient_metric_rows = []

    if compute_pseudo_patient_metrics:
        for method in ["mean", "max", "quality_weighted_mean"]:
            pseudo_patient_predictions = aggregate_predictions_to_patient_level(
                image_predictions=image_predictions,
                method=method,
            )

            metrics = compute_binary_classification_metrics(
                y_true=pseudo_patient_predictions["y_true"].values,
                y_prob=pseudo_patient_predictions["y_prob"].values,
                threshold=threshold,
                prefix=f"pseudo_patient_{method}_",
            )

            metrics["pseudo_patient_aggregation_method"] = method
            metrics["internal_patient_level_claim_allowed"] = False
            pseudo_patient_metric_rows.append(metrics)

    pseudo_patient_metrics = pd.DataFrame(pseudo_patient_metric_rows)

    return image_metrics, pseudo_patient_metrics


def get_primary_pseudo_patient_metrics(pseudo_patient_metrics, aggregation_method="mean"):
    if len(pseudo_patient_metrics) == 0:
        raise RuntimeError("Pseudo-patient metrics were not computed.")

    row = pseudo_patient_metrics[
        pseudo_patient_metrics["pseudo_patient_aggregation_method"] == aggregation_method
    ]

    if len(row) != 1:
        raise RuntimeError(
            f"Missing pseudo-patient metrics for aggregation method: {aggregation_method}"
        )

    return row.iloc[0].to_dict()


@torch.no_grad()
def predict_classifier(model, loader, device, split_name, model_name):
    model.eval()

    prediction_rows = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)

        with amp_autocast_context():
            logits = model(images)
            logits = flatten_logits(logits)

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        logits_np = logits.detach().cpu().numpy()
        labels_np = batch["label"].detach().cpu().numpy()

        for i in range(len(batch["sample_id"])):
            prediction_rows.append(
                {
                    "model_name": model_name,
                    "split": split_name,
                    "sample_id": batch["sample_id"][i],
                    "patient_id": batch["patient_id"][i],
                    "y_true": int(labels_np[i]),
                    "logit": float(logits_np[i]),
                    "y_prob": float(probs[i]),
                    "quality_score": float(batch["quality_score"][i].detach().cpu().item()),
                    "strict_anatomical_proxy_valid": bool(batch["strict_anatomical_proxy_valid"][i]),
                    "high_confidence_anatomical_proxy": bool(batch["high_confidence_anatomical_proxy"][i]),
                    "low_confidence_anatomical_proxy": bool(batch["low_confidence_anatomical_proxy"][i]),
                    "failed_anatomical_proxy": bool(batch["failed_anatomical_proxy"][i]),
                    "image_path": batch["image_path"][i],
                    "true_patient_id_available": False,
                    "patient_id_policy": "image_as_pseudo_patient_for_internal_training_compatibility",
                    "internal_patient_level_claim_allowed": False,
                }
            )

    return pd.DataFrame(prediction_rows)


def train_one_epoch_classifier(model, loader, criterion, optimizer, scaler, device):
    model.train()

    loss_values = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with amp_autocast_context():
            logits = model(images)
            logits = flatten_logits(logits)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_values.append(float(loss.detach().cpu()))

    return {
        "train_loss": float(np.mean(loss_values)),
        "n_batches": int(len(loss_values)),
    }


def build_epoch_loaders(input_size, batch_size):
    train_loader = make_classifier_loader(
        dataframe=train_df,
        input_size=input_size,
        batch_size=batch_size,
        split_name="train",
        augment=True,
        balanced_sampler=True,
    )

    val_loader = make_classifier_loader(
        dataframe=val_df,
        input_size=input_size,
        batch_size=batch_size,
        split_name="val",
        augment=False,
        balanced_sampler=False,
    )

    test_loader = make_classifier_loader(
        dataframe=test_df,
        input_size=input_size,
        batch_size=batch_size,
        split_name="test",
        augment=False,
        balanced_sampler=False,
    )

    return train_loader, val_loader, test_loader


def get_classifier_paths(root_dir, model_name, run_id):
    run_dir = Path(root_dir) / model_name / run_id
    checkpoint_dir = run_dir / "checkpoints"
    log_dir = run_dir / "logs"
    prediction_dir = run_dir / "predictions"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)
    prediction_dir.mkdir(parents=True, exist_ok=True)

    return {
        "run_dir": run_dir,
        "checkpoint_dir": checkpoint_dir,
        "log_dir": log_dir,
        "prediction_dir": prediction_dir,
        "latest_checkpoint": checkpoint_dir / "latest_checkpoint.pt",
        "best_checkpoint": checkpoint_dir / "best_checkpoint.pt",
        "training_log_csv": log_dir / "training_log.csv",
        "config_json": run_dir / "run_config.json",
        "val_predictions_csv": prediction_dir / "val_image_predictions.csv",
        "test_predictions_csv": prediction_dir / "test_image_predictions.csv",
        "val_pseudo_patient_predictions_csv": prediction_dir / "val_pseudo_patient_predictions_mean.csv",
        "test_pseudo_patient_predictions_csv": prediction_dir / "test_pseudo_patient_predictions_mean.csv",
        "val_patient_predictions_csv": prediction_dir / "val_pseudo_patient_predictions_mean.csv",
        "test_patient_predictions_csv": prediction_dir / "test_pseudo_patient_predictions_mean.csv",
    }


def save_classifier_checkpoint(
    checkpoint_path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    best_metric,
    config,
    history,
):
    payload = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_metric": float(best_metric),
        "config": config,
        "history": history,
    }

    torch.save(payload, checkpoint_path)


def load_classifier_checkpoint_if_available(
    checkpoint_path,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    device=DEVICE,
):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        return 0, -np.inf, []

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = int(checkpoint["epoch"]) + 1
    best_metric = float(checkpoint.get("best_metric", -np.inf))
    history = checkpoint.get("history", [])

    return start_epoch, best_metric, history


def compute_epoch_validation_metrics(model, val_loader, device, model_name, threshold=0.5):
    val_predictions = predict_classifier(
        model=model,
        loader=val_loader,
        device=device,
        split_name="val",
        model_name=model_name,
    )

    image_metrics, pseudo_patient_metrics = evaluate_prediction_dataframe(
        image_predictions=val_predictions,
        threshold=threshold,
        compute_pseudo_patient_metrics=True,
    )

    pseudo_patient_mean_metrics = get_primary_pseudo_patient_metrics(
        pseudo_patient_metrics=pseudo_patient_metrics,
        aggregation_method="mean",
    )

    epoch_metrics = {
        "image_val_auroc": image_metrics["image_auroc"],
        "image_val_auprc": image_metrics["image_auprc"],
        "image_val_accuracy": image_metrics["image_accuracy"],
        "image_val_balanced_accuracy": image_metrics["image_balanced_accuracy"],
        "image_val_brier_score": image_metrics["image_brier_score"],
        "image_val_sensitivity": image_metrics["image_recall_sensitivity"],
        "image_val_specificity": image_metrics["image_specificity"],
        "image_val_f1": image_metrics["image_f1"],

        # Compatibility only. AIROGS-light-v2 has no true patient_id.
        "pseudo_patient_val_auroc": pseudo_patient_mean_metrics["pseudo_patient_mean_auroc"],
        "pseudo_patient_val_auprc": pseudo_patient_mean_metrics["pseudo_patient_mean_auprc"],
        "pseudo_patient_val_accuracy": pseudo_patient_mean_metrics["pseudo_patient_mean_accuracy"],
        "pseudo_patient_val_balanced_accuracy": pseudo_patient_mean_metrics[
            "pseudo_patient_mean_balanced_accuracy"
        ],
        "pseudo_patient_val_brier_score": pseudo_patient_mean_metrics[
            "pseudo_patient_mean_brier_score"
        ],
        "pseudo_patient_val_sensitivity": pseudo_patient_mean_metrics[
            "pseudo_patient_mean_recall_sensitivity"
        ],
        "pseudo_patient_val_specificity": pseudo_patient_mean_metrics[
            "pseudo_patient_mean_specificity"
        ],
        "pseudo_patient_val_f1": pseudo_patient_mean_metrics["pseudo_patient_mean_f1"],

        # Explicitly disabled true patient-level metrics.
        "patient_val_auroc": np.nan,
        "patient_val_auprc": np.nan,
        "patient_val_accuracy": np.nan,
        "patient_val_balanced_accuracy": np.nan,
        "patient_val_brier_score": np.nan,
        "patient_val_sensitivity": np.nan,
        "patient_val_specificity": np.nan,
        "patient_val_f1": np.nan,
        "internal_patient_level_claim_allowed": False,
    }

    return val_predictions, image_metrics, pseudo_patient_metrics, epoch_metrics


def run_classifier_training_loop(
    model_name,
    run_id,
    root_dir,
    input_size,
    batch_size,
    max_epochs,
    patience,
    pretrained,
    stage,
):
    paths = get_classifier_paths(
        root_dir=root_dir,
        model_name=model_name,
        run_id=run_id,
    )

    config = {
        "stage": stage,
        "model_name": model_name,
        "family": MODEL_NAME_TO_FAMILY[model_name],
        "timm_model_name": MODEL_NAME_TO_TIMM[model_name],
        "run_id": run_id,
        "seed": SEED,
        "input_size": int(input_size),
        "batch_size": int(batch_size),
        "max_epochs": int(max_epochs),
        "patience": int(patience),
        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),
        "use_pretrained_weights": bool(pretrained),
        "criterion": "BCEWithLogitsLoss",
        "train_sampling": "WeightedRandomSampler balanced by image-level class",
        "primary_selection_metric": PRIMARY_SELECTION_METRIC,
        "primary_metric_level": "image_level",
        "threshold_for_epoch_metrics": 0.5,
        "device": str(DEVICE),
        "use_amp": bool(USE_AMP),
        "primary_training_dataset": "EyePACS-AIROGS-light-V2",
        "internal_split_policy": "official_train_val_test",
        "true_patient_id_available": False,
        "internal_patient_level_claim_allowed": False,
        "pseudo_patient_metrics_policy": (
            "computed only for compatibility; each image is treated as a pseudo-patient; "
            "not valid for patient-level claims"
        ),
        "checkpoint_policy": "save_latest_every_epoch_and_best_on_primary_metric_improvement",
        "resume_policy": "resume_from_latest_checkpoint_if_available",
    }

    with open(paths["config_json"], "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    train_loader, val_loader, _ = build_epoch_loaders(
        input_size=input_size,
        batch_size=batch_size,
    )

    model = build_classifier_model(
        model_name=model_name,
        pretrained=pretrained,
        input_size=input_size,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=max(2, patience // 3),
    )

    scaler = make_grad_scaler()

    start_epoch, best_metric, history = load_classifier_checkpoint_if_available(
        checkpoint_path=paths["latest_checkpoint"],
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
    )

    epochs_without_improvement = 0

    if len(history) > 0:
        finite_metric_rows = [
            row for row in history
            if PRIMARY_SELECTION_METRIC in row and np.isfinite(row[PRIMARY_SELECTION_METRIC])
        ]

        if len(finite_metric_rows) > 0:
            historical_best = max(row[PRIMARY_SELECTION_METRIC] for row in finite_metric_rows)

            trailing_rows = list(reversed(history))

            for row in trailing_rows:
                if (
                    PRIMARY_SELECTION_METRIC in row
                    and np.isfinite(row[PRIMARY_SELECTION_METRIC])
                    and row[PRIMARY_SELECTION_METRIC] >= historical_best
                ):
                    break

                epochs_without_improvement += 1

    for epoch in range(start_epoch, max_epochs):
        epoch_start = time.time()

        train_metrics = train_one_epoch_classifier(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
        )

        val_predictions, image_metrics, pseudo_patient_metrics, val_epoch_metrics = (
            compute_epoch_validation_metrics(
                model=model,
                val_loader=val_loader,
                device=DEVICE,
                model_name=model_name,
                threshold=0.5,
            )
        )

        current_metric = float(val_epoch_metrics[PRIMARY_SELECTION_METRIC])

        if np.isfinite(current_metric):
            scheduler.step(current_metric)

        improved = np.isfinite(current_metric) and current_metric > best_metric

        if improved:
            best_metric = current_metric
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        epoch_row = {
            "epoch": int(epoch),
            "model_name": model_name,
            "run_id": run_id,
            "stage": stage,
            "train_loss": float(train_metrics["train_loss"]),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "elapsed_sec": round(time.time() - epoch_start, 2),
            "improved": bool(improved),
            "best_metric": float(best_metric),
            **{
                key: float(value) if isinstance(value, (int, float, np.integer, np.floating)) else value
                for key, value in val_epoch_metrics.items()
            },
        }

        history.append(epoch_row)

        pd.DataFrame(history).to_csv(paths["training_log_csv"], index=False)

        save_classifier_checkpoint(
            checkpoint_path=paths["latest_checkpoint"],
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch,
            best_metric=best_metric,
            config=config,
            history=history,
        )

        if improved:
            save_classifier_checkpoint(
                checkpoint_path=paths["best_checkpoint"],
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                best_metric=best_metric,
                config=config,
                history=history,
            )

            val_predictions.to_csv(paths["val_predictions_csv"], index=False)

            val_pseudo_patient_predictions = aggregate_predictions_to_patient_level(
                image_predictions=val_predictions,
                method="mean",
            )

            val_pseudo_patient_predictions.to_csv(
                paths["val_pseudo_patient_predictions_csv"],
                index=False,
            )

            pseudo_patient_metrics.to_csv(
                paths["prediction_dir"] / "val_pseudo_patient_metrics_by_aggregation.csv",
                index=False,
            )

        print(
            f"{model_name} | {stage} | epoch {epoch + 1}/{max_epochs} | "
            f"train_loss={train_metrics['train_loss']:.4f} | "
            f"image_val_auroc={val_epoch_metrics['image_val_auroc']:.4f} | "
            f"image_val_auprc={val_epoch_metrics['image_val_auprc']:.4f} | "
            f"image_val_brier={val_epoch_metrics['image_val_brier_score']:.4f} | "
            f"best={best_metric:.4f} | "
            f"elapsed={epoch_row['elapsed_sec']}s"
        )

        if epochs_without_improvement >= patience:
            break

    if len(history) == 0:
        raise RuntimeError(f"No training history produced for {model_name}.")

    finite_history = [
        row for row in history
        if PRIMARY_SELECTION_METRIC in row and np.isfinite(row[PRIMARY_SELECTION_METRIC])
    ]

    if len(finite_history) == 0:
        raise RuntimeError(f"No finite primary metric found for {model_name}.")

    best_row = max(finite_history, key=lambda row: row[PRIMARY_SELECTION_METRIC])

    result = {
        "model_name": model_name,
        "family": MODEL_NAME_TO_FAMILY[model_name],
        "timm_model_name": MODEL_NAME_TO_TIMM[model_name],
        "run_id": run_id,
        "stage": stage,
        "best_epoch": int(best_row["epoch"]),
        "best_epoch_one_indexed": int(best_row["epoch"]) + 1,
        "primary_selection_metric": PRIMARY_SELECTION_METRIC,
        "best_primary_metric": float(best_row[PRIMARY_SELECTION_METRIC]),

        "best_image_val_auroc": float(best_row["image_val_auroc"]),
        "best_image_val_auprc": float(best_row["image_val_auprc"]),
        "best_image_val_accuracy": float(best_row["image_val_accuracy"]),
        "best_image_val_balanced_accuracy": float(best_row["image_val_balanced_accuracy"]),
        "best_image_val_brier_score": float(best_row["image_val_brier_score"]),
        "best_image_val_sensitivity": float(best_row["image_val_sensitivity"]),
        "best_image_val_specificity": float(best_row["image_val_specificity"]),
        "best_image_val_f1": float(best_row["image_val_f1"]),

        # Compatibility only. Do not use as true patient-level evidence.
        "best_pseudo_patient_val_auroc": float(best_row["pseudo_patient_val_auroc"]),
        "best_pseudo_patient_val_auprc": float(best_row["pseudo_patient_val_auprc"]),
        "best_pseudo_patient_val_balanced_accuracy": float(
            best_row["pseudo_patient_val_balanced_accuracy"]
        ),
        "best_pseudo_patient_val_brier_score": float(
            best_row["pseudo_patient_val_brier_score"]
        ),

        "best_patient_val_auroc": np.nan,
        "best_patient_val_auprc": np.nan,
        "best_patient_val_balanced_accuracy": np.nan,
        "best_patient_val_brier_score": np.nan,

        "internal_patient_level_claim_allowed": False,
        "true_patient_id_available": False,
        "epochs_ran": int(len(history)),
        "best_checkpoint": str(paths["best_checkpoint"]),
        "latest_checkpoint": str(paths["latest_checkpoint"]),
        "training_log": str(paths["training_log_csv"]),
        "val_predictions": str(paths["val_predictions_csv"]),
        "val_pseudo_patient_predictions": str(paths["val_pseudo_patient_predictions_csv"]),
    }

    del model, optimizer, scheduler, scaler

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


utility_smoke_predictions = pd.DataFrame(
    {
        "patient_id": [
            "image_as_patient__s1",
            "image_as_patient__s2",
            "image_as_patient__s3",
            "image_as_patient__s4",
        ],
        "y_true": [1, 1, 0, 0],
        "y_prob": [0.9, 0.8, 0.2, 0.4],
        "quality_score": [1.0, 1.0, 1.0, 1.0],
        "strict_anatomical_proxy_valid": [False, False, False, False],
        "high_confidence_anatomical_proxy": [False, False, False, False],
        "low_confidence_anatomical_proxy": [False, False, False, False],
        "failed_anatomical_proxy": [False, False, False, False],
    }
)

utility_image_metrics, utility_pseudo_patient_metrics = evaluate_prediction_dataframe(
    image_predictions=utility_smoke_predictions,
    threshold=0.5,
    compute_pseudo_patient_metrics=True,
)

utility_pseudo_patient_mean = get_primary_pseudo_patient_metrics(
    pseudo_patient_metrics=utility_pseudo_patient_metrics,
    aggregation_method="mean",
)

utility_smoke_summary = {
    "image_auroc": utility_image_metrics["image_auroc"],
    "image_auprc": utility_image_metrics["image_auprc"],
    "image_balanced_accuracy": utility_image_metrics["image_balanced_accuracy"],
    "pseudo_patient_mean_auroc": utility_pseudo_patient_mean["pseudo_patient_mean_auroc"],
    "pseudo_patient_mean_auprc": utility_pseudo_patient_mean["pseudo_patient_mean_auprc"],
    "pseudo_patient_mean_balanced_accuracy": utility_pseudo_patient_mean[
        "pseudo_patient_mean_balanced_accuracy"
    ],
    "internal_patient_level_claim_allowed": False,
}

UTILITY_CONFIG_JSON = RESULTS_07_DIR / "notebook07_training_utility_config.json"
UTILITY_SMOKE_JSON = RESULTS_07_DIR / "notebook07_training_utility_smoke_test.json"

utility_config = {
    "training_loop_defined": True,
    "prediction_function_defined": True,
    "image_level_metrics_defined": True,
    "pseudo_patient_aggregation_methods": ["mean", "max", "quality_weighted_mean"],
    "primary_pseudo_patient_aggregation": "mean",
    "checkpoint_resume_supported": True,
    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
    "primary_metric_level": "image_level",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "true_patient_id_available": False,
    "internal_patient_level_claim_allowed": False,
    "patient_level_external_evaluation_reserved_for": ["HYGD"],
    "utility_smoke_summary": utility_smoke_summary,
}

with open(UTILITY_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(utility_config, f, indent=2)

with open(UTILITY_SMOKE_JSON, "w", encoding="utf-8") as f:
    json.dump(utility_smoke_summary, f, indent=2)

print("Training utilities ready.")
print(f"Primary selection metric: {PRIMARY_SELECTION_METRIC}")
print("Internal metric level: image-level")
print("True patient-level internal claim allowed: False")
print("Pseudo-patient aggregation: compatibility only")
print(f"Utility config: {UTILITY_CONFIG_JSON}")
print(f"Utility smoke test: {UTILITY_SMOKE_JSON}")
print(json.dumps(utility_smoke_summary, indent=2))

if PRIMARY_SELECTION_METRIC != "image_val_auroc":
    raise RuntimeError(
        f"Expected image_val_auroc as primary metric for AIROGS-light, found: {PRIMARY_SELECTION_METRIC}"
    )

if not np.isfinite(utility_smoke_summary["image_auroc"]):
    raise RuntimeError("Utility smoke image AUROC is not finite.")

if not np.isfinite(utility_smoke_summary["pseudo_patient_mean_auroc"]):
    raise RuntimeError("Utility smoke pseudo-patient AUROC is not finite.")

if utility_smoke_summary["internal_patient_level_claim_allowed"]:
    raise RuntimeError("Internal patient-level claims must be disabled for AIROGS-light-v2.")

print("Status: PASS")

In [ ]:
# Cell 7: Configure AIROGS-light 10-model architecture screening plan

SCREENING_RUNS_DIR = SCREENING_ROOT
SCREENING_RUNS_DIR.mkdir(parents=True, exist_ok=True)

SCREENING_PLAN_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_plan.csv"
SCREENING_CONFIG_JSON = RESULTS_07_DIR / "notebook07_architecture_screening_config.json"

if "MODEL_BENCHMARK_PLAN" not in globals():
    MODEL_BENCHMARK_PLAN_CSV = RESULTS_07_DIR / "notebook07_model_benchmark_plan.csv"

    if not MODEL_BENCHMARK_PLAN_CSV.exists():
        raise FileNotFoundError(f"Missing model benchmark plan: {MODEL_BENCHMARK_PLAN_CSV}")

    MODEL_BENCHMARK_PLAN = pd.read_csv(MODEL_BENCHMARK_PLAN_CSV)

if "train_df" not in globals() or "val_df" not in globals() or "test_df" not in globals():
    train_df = get_classifier_split("train")
    val_df = get_classifier_split("val")
    test_df = get_classifier_split("test")

MODEL_BENCHMARK_PLAN = MODEL_BENCHMARK_PLAN.copy()
MODEL_BENCHMARK_PLAN["include_in_screening"] = as_bool_series(
    MODEL_BENCHMARK_PLAN["include_in_screening"]
)

screening_models_df = (
    MODEL_BENCHMARK_PLAN[
        MODEL_BENCHMARK_PLAN["include_in_screening"]
    ]
    .copy()
    .reset_index(drop=True)
)

if len(screening_models_df) != 10:
    raise RuntimeError(f"Expected 10 screening models, found {len(screening_models_df)}")

if PRIMARY_SELECTION_METRIC != "image_val_auroc":
    raise RuntimeError(
        f"AIROGS-light screening must use image_val_auroc as primary metric, found: {PRIMARY_SELECTION_METRIC}"
    )

screening_plan_rows = []

for _, row in screening_models_df.iterrows():
    model_name = row["model_name"]
    family = row["family"]
    timm_model_name = row["timm_model_name"]

    run_id = f"{model_name}__airogs_light_screening_seed{SEED}"

    paths = get_classifier_paths(
        root_dir=SCREENING_RUNS_DIR,
        model_name=model_name,
        run_id=run_id,
    )

    screening_plan_rows.append(
        {
            "run_id": run_id,
            "model_name": model_name,
            "family": family,
            "timm_model_name": timm_model_name,
            "stage": "architecture_screening",
            "primary_training_dataset": "EyePACS-AIROGS-light-V2",
            "split_policy": "official_train_val_test_image_level",
            "input_size": int(SCREENING_INPUT_SIZE),
            "batch_size": int(SCREENING_BATCH_SIZE),
            "max_epochs": int(SCREENING_MAX_EPOCHS),
            "early_stop_patience": int(SCREENING_EARLY_STOP_PATIENCE),
            "learning_rate": float(LEARNING_RATE),
            "weight_decay": float(WEIGHT_DECAY),
            "use_pretrained_weights": bool(USE_PRETRAINED_WEIGHTS),
            "primary_selection_metric": PRIMARY_SELECTION_METRIC,
            "primary_metric_level": "image_level",
            "secondary_selection_metrics": "|".join(SECONDARY_SELECTION_METRICS),

            "train_images": int(len(train_df)),
            "val_images": int(len(val_df)),
            "test_images_locked_not_used": int(len(test_df)),

            "train_rg_images": int((train_df["gon_binary"].astype(int) == 1).sum()),
            "train_nrg_images": int((train_df["gon_binary"].astype(int) == 0).sum()),
            "val_rg_images": int((val_df["gon_binary"].astype(int) == 1).sum()),
            "val_nrg_images": int((val_df["gon_binary"].astype(int) == 0).sum()),
            "test_rg_images_locked_not_used": int((test_df["gon_binary"].astype(int) == 1).sum()),
            "test_nrg_images_locked_not_used": int((test_df["gon_binary"].astype(int) == 0).sum()),

            "true_patient_id_available": False,
            "internal_patient_level_claim_allowed": False,
            "train_pseudo_patients_for_loader_compatibility": int(train_df["patient_id"].nunique()),
            "val_pseudo_patients_for_loader_compatibility": int(val_df["patient_id"].nunique()),
            "test_pseudo_patients_locked_not_used": int(test_df["patient_id"].nunique()),

            "quality_score_available": False,
            "anatomical_proxy_available_for_training_dataset": False,
            "external_patient_level_evaluation_reserved_for": "HYGD",
            "external_quality_evaluation_reserved_for": "HYGD",
            "external_domain_shift_evaluation_reserved_for": "HYGD|PAPILA|SMDG",

            "run_dir": str(paths["run_dir"]),
            "latest_checkpoint": str(paths["latest_checkpoint"]),
            "best_checkpoint": str(paths["best_checkpoint"]),
            "training_log": str(paths["training_log_csv"]),
            "val_predictions": str(paths["val_predictions_csv"]),
            "val_pseudo_patient_predictions": str(paths["val_pseudo_patient_predictions_csv"]),
        }
    )

architecture_screening_plan = pd.DataFrame(screening_plan_rows)

architecture_screening_plan.to_csv(SCREENING_PLAN_CSV, index=False)

screening_config = {
    "stage": "architecture_screening",
    "purpose": (
        "Benchmark 10 representative classifier architectures on EyePACS-AIROGS-light-V2 "
        "using the official image-level train/validation split before selecting candidates "
        "for full training."
    ),
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "split_policy": "official_train_val_test_image_level",
    "n_models": int(len(architecture_screening_plan)),
    "models": architecture_screening_plan["model_name"].tolist(),
    "families": sorted(architecture_screening_plan["family"].unique().tolist()),
    "input_size": int(SCREENING_INPUT_SIZE),
    "batch_size": int(SCREENING_BATCH_SIZE),
    "max_epochs": int(SCREENING_MAX_EPOCHS),
    "early_stop_patience": int(SCREENING_EARLY_STOP_PATIENCE),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "use_pretrained_weights": bool(USE_PRETRAINED_WEIGHTS),
    "train_sampling": "WeightedRandomSampler balanced by image-level class",
    "augmentation_policy": (
        "Horizontal flip, mild rotation, brightness/contrast jitter, and gamma jitter "
        "for training only; deterministic square padding, resizing, and ImageNet normalization "
        "for validation/test."
    ),
    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
    "primary_metric_level": "image_level",
    "secondary_selection_metrics": SECONDARY_SELECTION_METRICS,
    "true_patient_id_available": False,
    "internal_patient_level_claim_allowed": False,
    "pseudo_patient_policy": (
        "Pseudo-patient IDs are used only for compatibility with previous code. "
        "Each AIROGS-light image is treated as an independent pseudo-patient. "
        "These metrics are not valid patient-level evidence."
    ),
    "quality_score_available": False,
    "anatomical_proxy_available_for_training_dataset": False,
    "test_set_policy": "The locked AIROGS-light test split is not used during screening or model selection.",
    "checkpoint_policy": "latest checkpoint every epoch and best checkpoint on primary metric improvement",
    "resume_policy": "automatic resume from latest checkpoint when available",
    "screening_plan_csv": str(SCREENING_PLAN_CSV),
    "screening_root": str(SCREENING_RUNS_DIR),
    "external_aqpr_policy": {
        "HYGD": "reserved for external patient-level, quality, anatomical plausibility, and calibration audits",
        "PAPILA": "reserved for external anatomy/domain-shift and label-policy audits",
        "SMDG": "reserved for curated multi-source external domain-shift audit",
    },
}

with open(SCREENING_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(screening_config, f, indent=2)

print("AIROGS-light architecture screening plan:")
print(
    architecture_screening_plan[
        [
            "run_id",
            "model_name",
            "family",
            "input_size",
            "batch_size",
            "max_epochs",
            "early_stop_patience",
            "primary_selection_metric",
            "primary_metric_level",
        ]
    ].to_string(index=False)
)

print("\nSplit counts used for screening:")
print(
    architecture_screening_plan[
        [
            "model_name",
            "train_images",
            "train_rg_images",
            "train_nrg_images",
            "val_images",
            "val_rg_images",
            "val_nrg_images",
            "test_images_locked_not_used",
        ]
    ].head(10).to_string(index=False)
)

print(f"\nScreening plan CSV: {SCREENING_PLAN_CSV}")
print(f"Screening config: {SCREENING_CONFIG_JSON}")

if architecture_screening_plan["run_id"].duplicated().any():
    raise RuntimeError("Duplicate screening run IDs found.")

if architecture_screening_plan["family"].nunique() < 7:
    raise RuntimeError("Expected at least 7 model families in screening plan.")

if not architecture_screening_plan["primary_selection_metric"].eq(PRIMARY_SELECTION_METRIC).all():
    raise RuntimeError("Primary selection metric mismatch in screening plan.")

if not architecture_screening_plan["primary_metric_level"].eq("image_level").all():
    raise RuntimeError("AIROGS-light screening must use image-level primary metric.")

expected_counts = {
    "train_images": 8000,
    "val_images": 770,
    "test_images_locked_not_used": 770,
    "train_rg_images": 4000,
    "train_nrg_images": 4000,
    "val_rg_images": 385,
    "val_nrg_images": 385,
    "test_rg_images_locked_not_used": 385,
    "test_nrg_images_locked_not_used": 385,
}

for column, expected_value in expected_counts.items():
    observed_values = architecture_screening_plan[column].unique().tolist()

    if observed_values != [expected_value]:
        raise RuntimeError(
            f"Unexpected {column}: expected all {expected_value}, observed {observed_values}"
        )

if architecture_screening_plan["internal_patient_level_claim_allowed"].any():
    raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

print("Status: PASS")

In [ ]:
# Cell 8: Run one resumable AIROGS-light architecture-screening training job

SCREENING_PLAN_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_plan.csv"

if not SCREENING_PLAN_CSV.exists():
    raise FileNotFoundError(f"Missing screening plan: {SCREENING_PLAN_CSV}")

architecture_screening_plan = pd.read_csv(SCREENING_PLAN_CSV)

required_screening_columns = [
    "run_id",
    "model_name",
    "family",
    "timm_model_name",
    "input_size",
    "batch_size",
    "max_epochs",
    "early_stop_patience",
    "use_pretrained_weights",
    "primary_selection_metric",
    "primary_metric_level",
    "best_checkpoint",
    "latest_checkpoint",
    "training_log",
]

missing_screening_columns = [
    col for col in required_screening_columns
    if col not in architecture_screening_plan.columns
]

if missing_screening_columns:
    raise ValueError(f"Missing screening plan columns: {missing_screening_columns}")


def parse_bool_value(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, (np.bool_,)):
        return bool(value)

    value_str = str(value).strip().lower()

    if value_str in {"true", "1", "yes", "y"}:
        return True

    if value_str in {"false", "0", "no", "n", "nan", "none", ""}:
        return False

    raise ValueError(f"Cannot parse boolean value: {value}")


architecture_screening_plan["use_pretrained_weights"] = architecture_screening_plan[
    "use_pretrained_weights"
].apply(parse_bool_value)

architecture_screening_plan["primary_metric_level"] = architecture_screening_plan[
    "primary_metric_level"
].astype(str)

architecture_screening_plan["primary_selection_metric"] = architecture_screening_plan[
    "primary_selection_metric"
].astype(str)

if not architecture_screening_plan["primary_selection_metric"].eq("image_val_auroc").all():
    raise RuntimeError("AIROGS-light screening must use image_val_auroc as primary metric.")

if not architecture_screening_plan["primary_metric_level"].eq("image_level").all():
    raise RuntimeError("AIROGS-light screening must use image-level validation metrics.")

# Leave this as None to automatically train the next pending screening run.
# To force a specific run, use for example:
# RUN_ID_TO_TRAIN = "ResNet18__airogs_light_screening_seed42"
RUN_ID_TO_TRAIN = None

# Keep this True to avoid wasting hours if the runtime accidentally starts without GPU.
SKIP_TRAINING_IF_NO_CUDA = True

SCREENING_RESULT_DIR = RESULTS_07_DIR / "screening_run_results"
SCREENING_RESULT_DIR.mkdir(parents=True, exist_ok=True)


def screening_result_path(run_id):
    safe_run_id = str(run_id).replace("/", "_")
    return SCREENING_RESULT_DIR / f"notebook07_screening_result_{safe_run_id}.csv"


def screening_run_completed(plan_row):
    run_id = plan_row["run_id"]
    result_csv = screening_result_path(run_id)
    best_checkpoint = Path(str(plan_row["best_checkpoint"]))
    training_log = Path(str(plan_row["training_log"]))

    if not result_csv.exists():
        return False

    if not best_checkpoint.exists():
        return False

    if not training_log.exists():
        return False

    try:
        result_df = pd.read_csv(result_csv)
        log_df = pd.read_csv(training_log)
    except Exception:
        return False

    if len(result_df) != 1 or len(log_df) == 0:
        return False

    required_result_columns = [
        "best_primary_metric",
        "best_image_val_auroc",
        "best_image_val_auprc",
        "best_image_val_balanced_accuracy",
        "best_image_val_brier_score",
        "best_checkpoint",
        "training_log",
    ]

    missing_result_columns = [
        col for col in required_result_columns
        if col not in result_df.columns
    ]

    if missing_result_columns:
        return False

    if not np.isfinite(pd.to_numeric(result_df["best_primary_metric"], errors="coerce").iloc[0]):
        return False

    if not np.isfinite(pd.to_numeric(result_df["best_image_val_auroc"], errors="coerce").iloc[0]):
        return False

    if not Path(str(result_df["best_checkpoint"].iloc[0])).exists():
        return False

    if not Path(str(result_df["training_log"].iloc[0])).exists():
        return False

    return True


architecture_screening_plan["result_csv"] = architecture_screening_plan["run_id"].apply(
    lambda run_id: str(screening_result_path(run_id))
)

architecture_screening_plan["completed"] = architecture_screening_plan.apply(
    screening_run_completed,
    axis=1,
)

pending_runs = architecture_screening_plan[
    ~architecture_screening_plan["completed"]
].copy().reset_index(drop=True)

if RUN_ID_TO_TRAIN is None:
    if len(pending_runs) == 0:
        selected_run = None
    else:
        selected_run = pending_runs.iloc[0].to_dict()
else:
    matching_runs = architecture_screening_plan[
        architecture_screening_plan["run_id"] == RUN_ID_TO_TRAIN
    ].copy()

    if len(matching_runs) != 1:
        raise ValueError(f"RUN_ID_TO_TRAIN not found or not unique: {RUN_ID_TO_TRAIN}")

    selected_run = matching_runs.iloc[0].to_dict()

screening_progress_csv = RESULTS_07_DIR / "notebook07_architecture_screening_progress.csv"
architecture_screening_plan.to_csv(screening_progress_csv, index=False)

print("AIROGS-light architecture screening progress:")
print(
    architecture_screening_plan[
        [
            "run_id",
            "model_name",
            "family",
            "primary_selection_metric",
            "completed",
            "result_csv",
        ]
    ].to_string(index=False)
)

print(f"\nCompleted runs: {int(architecture_screening_plan['completed'].sum())}/{len(architecture_screening_plan)}")
print(f"Pending runs: {len(pending_runs)}")
print(f"Progress CSV: {screening_progress_csv}")

if selected_run is None:
    print("\nAll AIROGS-light screening runs are already completed.")
    print("Status: PASS")

else:
    selected_run_pretrained = parse_bool_value(selected_run["use_pretrained_weights"])

    print("\nSelected screening run:")
    print(json.dumps(
        {
            "run_id": selected_run["run_id"],
            "model_name": selected_run["model_name"],
            "family": selected_run["family"],
            "timm_model_name": selected_run["timm_model_name"],
            "input_size": int(selected_run["input_size"]),
            "batch_size": int(selected_run["batch_size"]),
            "max_epochs": int(selected_run["max_epochs"]),
            "early_stop_patience": int(selected_run["early_stop_patience"]),
            "use_pretrained_weights": bool(selected_run_pretrained),
            "primary_selection_metric": selected_run["primary_selection_metric"],
            "primary_metric_level": selected_run["primary_metric_level"],
            "primary_training_dataset": "EyePACS-AIROGS-light-V2",
            "device": str(DEVICE),
            "cuda_available": bool(torch.cuda.is_available()),
            "internal_patient_level_claim_allowed": False,
        },
        indent=2,
    ))

    if DEVICE.type != "cuda" and SKIP_TRAINING_IF_NO_CUDA:
        print("\nTraining skipped because CUDA is not available.")
        print("Set SKIP_TRAINING_IF_NO_CUDA = False only if CPU training is intentional.")
        print("Status: PASS")

    else:
        run_start_time = time.time()

        result = run_classifier_training_loop(
            model_name=str(selected_run["model_name"]),
            run_id=str(selected_run["run_id"]),
            root_dir=SCREENING_ROOT,
            input_size=int(selected_run["input_size"]),
            batch_size=int(selected_run["batch_size"]),
            max_epochs=int(selected_run["max_epochs"]),
            patience=int(selected_run["early_stop_patience"]),
            pretrained=bool(selected_run_pretrained),
            stage="architecture_screening",
        )

        result["elapsed_min"] = round((time.time() - run_start_time) / 60.0, 3)
        result["primary_training_dataset"] = "EyePACS-AIROGS-light-V2"
        result["primary_metric_level"] = "image_level"
        result["internal_patient_level_claim_allowed"] = False
        result["true_patient_id_available"] = False

        result_df = pd.DataFrame([result])
        result_csv = screening_result_path(selected_run["run_id"])
        result_df.to_csv(result_csv, index=False)

        print("\nScreening run result:")
        print(
            result_df[
                [
                    "model_name",
                    "family",
                    "best_epoch_one_indexed",
                    "primary_selection_metric",
                    "best_primary_metric",
                    "best_image_val_auroc",
                    "best_image_val_auprc",
                    "best_image_val_balanced_accuracy",
                    "best_image_val_brier_score",
                    "best_image_val_sensitivity",
                    "best_image_val_specificity",
                    "best_image_val_f1",
                    "epochs_ran",
                    "elapsed_min",
                    "internal_patient_level_claim_allowed",
                ]
            ].to_string(index=False)
        )

        print(f"\nRun result CSV: {result_csv}")
        print(f"Best checkpoint: {result['best_checkpoint']}")
        print(f"Training log: {result['training_log']}")

        if not Path(result["best_checkpoint"]).exists():
            raise RuntimeError("Best checkpoint was not saved.")

        if not Path(result["training_log"]).exists():
            raise RuntimeError("Training log was not saved.")

        if not np.isfinite(float(result["best_primary_metric"])):
            raise RuntimeError("Best primary validation metric is not finite.")

        if not np.isfinite(float(result["best_image_val_auroc"])):
            raise RuntimeError("Best image-level validation AUROC is not finite.")

        if result["primary_selection_metric"] != "image_val_auroc":
            raise RuntimeError("Screening result used an unexpected primary metric.")

        if bool(result["internal_patient_level_claim_allowed"]):
            raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

        print("Status: PASS")

In [ ]:
# Cell 9: Consolidate AIROGS-light architecture-screening results and rank candidate classifiers

SCREENING_PLAN_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_plan.csv"
SCREENING_PROGRESS_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_progress.csv"

SCREENING_RESULTS_CONSOLIDATED_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_results_consolidated.csv"
SCREENING_RANKING_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_ranking.csv"
SCREENING_FAMILY_SUMMARY_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_family_summary.csv"
SCREENING_ARTIFACT_AUDIT_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_artifact_audit.csv"
SCREENING_SELECTED_MODELS_CSV = RESULTS_07_DIR / "notebook07_selected_screening_models_for_full_training.csv"
SCREENING_CONSOLIDATION_JSON = RESULTS_07_DIR / "notebook07_architecture_screening_consolidation_summary.json"

# Keep False for the formal pipeline.
# Set True only if you want to inspect partial ranking before all 10 screening runs are completed.
ALLOW_PARTIAL_SCREENING_CONSOLIDATION = False

if SCREENING_PROGRESS_CSV.exists():
    screening_plan_runtime = pd.read_csv(SCREENING_PROGRESS_CSV)
elif SCREENING_PLAN_CSV.exists():
    screening_plan_runtime = pd.read_csv(SCREENING_PLAN_CSV)
else:
    raise FileNotFoundError("Missing screening plan/progress CSV.")

SCREENING_RESULT_DIR = RESULTS_07_DIR / "screening_run_results"
SCREENING_RESULT_DIR.mkdir(parents=True, exist_ok=True)

if "result_csv" not in screening_plan_runtime.columns:
    screening_plan_runtime["result_csv"] = screening_plan_runtime["run_id"].apply(
        lambda run_id: str(SCREENING_RESULT_DIR / f"notebook07_screening_result_{run_id}.csv")
    )

# Compatibility with the previous HYGD-oriented column name.
# In the AIROGS-light version, this artifact is pseudo-patient only and must not be used for true patient-level claims.
if "val_pseudo_patient_predictions" not in screening_plan_runtime.columns:
    if "val_patient_predictions" in screening_plan_runtime.columns:
        screening_plan_runtime["val_pseudo_patient_predictions"] = screening_plan_runtime["val_patient_predictions"]
    else:
        screening_plan_runtime["val_pseudo_patient_predictions"] = ""

required_plan_cols = [
    "run_id",
    "model_name",
    "family",
    "timm_model_name",
    "result_csv",
    "best_checkpoint",
    "latest_checkpoint",
    "training_log",
    "val_predictions",
    "val_pseudo_patient_predictions",
]

missing_plan_cols = [
    col for col in required_plan_cols
    if col not in screening_plan_runtime.columns
]

if missing_plan_cols:
    raise ValueError(f"Missing screening plan columns: {missing_plan_cols}")

result_rows = []
artifact_rows = []

for _, plan_row in screening_plan_runtime.iterrows():
    run_id = str(plan_row["run_id"])

    result_csv = Path(str(plan_row["result_csv"]))
    best_checkpoint = Path(str(plan_row["best_checkpoint"]))
    latest_checkpoint = Path(str(plan_row["latest_checkpoint"]))
    training_log = Path(str(plan_row["training_log"]))
    val_predictions = Path(str(plan_row["val_predictions"]))

    val_pseudo_patient_predictions_raw = str(plan_row["val_pseudo_patient_predictions"])
    val_pseudo_patient_predictions = (
        Path(val_pseudo_patient_predictions_raw)
        if val_pseudo_patient_predictions_raw.strip() not in {"", "nan", "None"}
        else None
    )

    artifact_row = {
        "run_id": run_id,
        "model_name": str(plan_row["model_name"]),
        "family": str(plan_row["family"]),
        "result_csv": str(result_csv),
        "result_csv_exists": result_csv.exists(),
        "best_checkpoint": str(best_checkpoint),
        "best_checkpoint_exists": best_checkpoint.exists(),
        "latest_checkpoint": str(latest_checkpoint),
        "latest_checkpoint_exists": latest_checkpoint.exists(),
        "training_log": str(training_log),
        "training_log_exists": training_log.exists(),
        "val_predictions": str(val_predictions),
        "val_predictions_exists": val_predictions.exists(),
        "val_pseudo_patient_predictions": (
            str(val_pseudo_patient_predictions)
            if val_pseudo_patient_predictions is not None
            else ""
        ),
        "val_pseudo_patient_predictions_exists": (
            val_pseudo_patient_predictions.exists()
            if val_pseudo_patient_predictions is not None
            else False
        ),
    }

    artifact_rows.append(artifact_row)

    if not result_csv.exists():
        continue

    result_df = pd.read_csv(result_csv)

    if len(result_df) != 1:
        raise RuntimeError(f"Expected one row in result CSV for {run_id}, found {len(result_df)}")

    result_row = result_df.iloc[0].to_dict()

    for key in [
        "run_id",
        "model_name",
        "family",
        "timm_model_name",
        "best_checkpoint",
        "latest_checkpoint",
        "training_log",
        "val_predictions",
        "val_pseudo_patient_predictions",
    ]:
        if key not in result_row or pd.isna(result_row[key]):
            result_row[key] = plan_row[key] if key in plan_row.index else ""

    result_row["true_patient_id_available"] = False
    result_row["internal_patient_level_claim_allowed"] = False
    result_row["primary_training_dataset"] = "EyePACS-AIROGS-light-V2"
    result_row["primary_metric_level"] = "image_level"

    result_rows.append(result_row)

screening_artifact_audit = pd.DataFrame(artifact_rows)

if len(result_rows) == 0:
    raise RuntimeError(
        "No screening result rows were found. "
        "Run Cell 8 until at least one screening model finishes."
    )

screening_results = pd.DataFrame(result_rows)

expected_n_runs = int(len(screening_plan_runtime))
observed_n_results = int(len(screening_results))

if not ALLOW_PARTIAL_SCREENING_CONSOLIDATION and observed_n_results != expected_n_runs:
    missing_runs = screening_plan_runtime.loc[
        ~screening_plan_runtime["run_id"].isin(screening_results["run_id"].astype(str)),
        ["run_id", "model_name", "family", "result_csv"],
    ]

    print("Missing screening runs:")
    print(missing_runs.to_string(index=False))

    raise RuntimeError(
        f"Screening is incomplete: expected {expected_n_runs} results, found {observed_n_results}. "
        "Run Cell 8 repeatedly until all 10 architecture-screening runs are completed, "
        "or set ALLOW_PARTIAL_SCREENING_CONSOLIDATION = True for a temporary partial audit."
    )

numeric_cols = [
    "best_epoch",
    "best_epoch_one_indexed",
    "best_primary_metric",
    "best_image_val_auroc",
    "best_image_val_auprc",
    "best_image_val_accuracy",
    "best_image_val_balanced_accuracy",
    "best_image_val_brier_score",
    "best_image_val_sensitivity",
    "best_image_val_specificity",
    "best_image_val_f1",
    "best_pseudo_patient_val_auroc",
    "best_pseudo_patient_val_auprc",
    "best_pseudo_patient_val_balanced_accuracy",
    "best_pseudo_patient_val_brier_score",
    "epochs_ran",
    "elapsed_min",
]

for col in numeric_cols:
    if col in screening_results.columns:
        screening_results[col] = pd.to_numeric(screening_results[col], errors="coerce")

required_result_cols = [
    "run_id",
    "model_name",
    "family",
    "timm_model_name",
    "best_primary_metric",
    "best_image_val_auroc",
    "best_image_val_auprc",
    "best_image_val_balanced_accuracy",
    "best_image_val_brier_score",
    "best_image_val_sensitivity",
    "best_image_val_specificity",
    "best_image_val_f1",
    "epochs_ran",
    "best_checkpoint",
    "training_log",
    "val_predictions",
]

missing_result_cols = [
    col for col in required_result_cols
    if col not in screening_results.columns
]

if missing_result_cols:
    raise ValueError(f"Missing screening result columns: {missing_result_cols}")

finite_metric_checks = [
    "best_primary_metric",
    "best_image_val_auroc",
    "best_image_val_auprc",
    "best_image_val_balanced_accuracy",
    "best_image_val_brier_score",
    "best_image_val_sensitivity",
    "best_image_val_specificity",
    "best_image_val_f1",
]

for col in finite_metric_checks:
    if not np.isfinite(screening_results[col]).all():
        bad_rows = screening_results.loc[
            ~np.isfinite(screening_results[col]),
            ["run_id", "model_name", col],
        ]

        print(f"Rows with non-finite {col}:")
        print(bad_rows.to_string(index=False))

        raise RuntimeError(f"Non-finite values found in {col}")

if "primary_selection_metric" in screening_results.columns:
    bad_metric_rows = screening_results[
        screening_results["primary_selection_metric"].astype(str) != "image_val_auroc"
    ]

    if len(bad_metric_rows) > 0:
        print(bad_metric_rows[["run_id", "model_name", "primary_selection_metric"]].to_string(index=False))
        raise RuntimeError("All screening runs must use image_val_auroc as primary selection metric.")

if "internal_patient_level_claim_allowed" in screening_results.columns:
    screening_results["internal_patient_level_claim_allowed"] = as_bool_series(
        screening_results["internal_patient_level_claim_allowed"]
    )

    if screening_results["internal_patient_level_claim_allowed"].any():
        raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

# Ranking rule for AIROGS-light architecture screening.
# Primary emphasis: image-level AUROC and AUPRC.
# Secondary emphasis: balanced accuracy, sensitivity, and specificity at threshold 0.5.
# Penalty: worse Brier score and longer runtime.
# This ranking is used only to select candidates for full training.
if "elapsed_min" in screening_results.columns:
    elapsed = screening_results["elapsed_min"].replace([np.inf, -np.inf], np.nan)

    if elapsed.notna().sum() > 1 and elapsed.max() > elapsed.min():
        screening_results["elapsed_min_norm"] = (
            (elapsed - elapsed.min()) / (elapsed.max() - elapsed.min())
        )
    else:
        screening_results["elapsed_min_norm"] = 0.0
else:
    screening_results["elapsed_min_norm"] = 0.0

screening_results["selection_score"] = (
    0.35 * screening_results["best_image_val_auroc"]
    + 0.25 * screening_results["best_image_val_auprc"]
    + 0.20 * screening_results["best_image_val_balanced_accuracy"]
    + 0.10 * screening_results["best_image_val_sensitivity"]
    + 0.10 * screening_results["best_image_val_specificity"]
    - 0.08 * screening_results["best_image_val_brier_score"]
    - 0.02 * screening_results["elapsed_min_norm"].fillna(0.0)
)

screening_results = screening_results.sort_values(
    [
        "selection_score",
        "best_image_val_auroc",
        "best_image_val_auprc",
        "best_image_val_balanced_accuracy",
        "best_image_val_brier_score",
        "elapsed_min",
    ],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)

screening_results["screening_rank"] = np.arange(1, len(screening_results) + 1)

TOP_N_FOR_FULL_TRAINING = 3

screening_results["decision"] = np.where(
    screening_results["screening_rank"] <= TOP_N_FOR_FULL_TRAINING,
    "selected_for_full_training",
    "not_selected_after_screening",
)

screening_family_summary = (
    screening_results.groupby("family")
    .agg(
        n_models=("model_name", "count"),
        best_selection_score=("selection_score", "max"),
        mean_selection_score=("selection_score", "mean"),
        best_image_val_auroc=("best_image_val_auroc", "max"),
        best_image_val_auprc=("best_image_val_auprc", "max"),
        best_image_val_balanced_accuracy=("best_image_val_balanced_accuracy", "max"),
        best_image_val_brier_score=("best_image_val_brier_score", "min"),
        best_image_val_sensitivity=("best_image_val_sensitivity", "max"),
        best_image_val_specificity=("best_image_val_specificity", "max"),
        best_rank=("screening_rank", "min"),
    )
    .reset_index()
    .sort_values(["best_rank", "best_selection_score"], ascending=[True, False])
    .reset_index(drop=True)
)

selected_screening_models = screening_results[
    screening_results["decision"] == "selected_for_full_training"
].copy().reset_index(drop=True)

screening_results.to_csv(SCREENING_RESULTS_CONSOLIDATED_CSV, index=False)
screening_results.to_csv(SCREENING_RANKING_CSV, index=False)
screening_family_summary.to_csv(SCREENING_FAMILY_SUMMARY_CSV, index=False)
screening_artifact_audit.to_csv(SCREENING_ARTIFACT_AUDIT_CSV, index=False)
selected_screening_models.to_csv(SCREENING_SELECTED_MODELS_CSV, index=False)

complete_artifact_mask = (
    screening_artifact_audit["result_csv_exists"]
    & screening_artifact_audit["best_checkpoint_exists"]
    & screening_artifact_audit["training_log_exists"]
    & screening_artifact_audit["val_predictions_exists"]
)

consolidation_summary = {
    "stage": "architecture_screening_consolidation",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "split_policy": "official_train_val_test_image_level",
    "n_expected_runs": expected_n_runs,
    "n_result_rows": observed_n_results,
    "allow_partial_screening_consolidation": bool(ALLOW_PARTIAL_SCREENING_CONSOLIDATION),
    "n_complete_artifact_sets": int(complete_artifact_mask.sum()),
    "top_n_for_full_training": int(TOP_N_FOR_FULL_TRAINING),
    "selected_models": selected_screening_models["model_name"].tolist(),
    "selected_families": selected_screening_models["family"].tolist(),
    "ranking_csv": str(SCREENING_RANKING_CSV),
    "selected_models_csv": str(SCREENING_SELECTED_MODELS_CSV),
    "family_summary_csv": str(SCREENING_FAMILY_SUMMARY_CSV),
    "artifact_audit_csv": str(SCREENING_ARTIFACT_AUDIT_CSV),
    "selection_score_formula": (
        "0.35*image_AUROC + 0.25*image_AUPRC + 0.20*image_balanced_accuracy "
        "+ 0.10*image_sensitivity + 0.10*image_specificity "
        "- 0.08*image_Brier - 0.02*normalized_elapsed_min"
    ),
    "primary_selection_metric": "image_val_auroc",
    "primary_metric_level": "image_level",
    "true_patient_id_available": False,
    "internal_patient_level_claim_allowed": False,
    "test_set_policy": "The locked AIROGS-light test split remains unused at screening consolidation.",
    "selection_note": (
        "The ranking is used only for selecting candidates for full training. "
        "No locked-test result is used in this cell."
    ),
}

with open(SCREENING_CONSOLIDATION_JSON, "w", encoding="utf-8") as f:
    json.dump(consolidation_summary, f, indent=2)

print("AIROGS-light architecture screening ranking:")
print(
    screening_results[
        [
            "screening_rank",
            "model_name",
            "family",
            "best_epoch_one_indexed",
            "best_image_val_auroc",
            "best_image_val_auprc",
            "best_image_val_balanced_accuracy",
            "best_image_val_brier_score",
            "best_image_val_sensitivity",
            "best_image_val_specificity",
            "epochs_ran",
            "elapsed_min",
            "selection_score",
            "decision",
        ]
    ].to_string(index=False)
)

print("\nFamily summary:")
print(screening_family_summary.to_string(index=False))

print("\nArtifact audit:")
print(
    screening_artifact_audit[
        [
            "model_name",
            "result_csv_exists",
            "best_checkpoint_exists",
            "training_log_exists",
            "val_predictions_exists",
            "val_pseudo_patient_predictions_exists",
        ]
    ].to_string(index=False)
)

print(f"\nConsolidated results: {SCREENING_RESULTS_CONSOLIDATED_CSV}")
print(f"Screening ranking: {SCREENING_RANKING_CSV}")
print(f"Selected screening models: {SCREENING_SELECTED_MODELS_CSV}")
print(f"Family summary: {SCREENING_FAMILY_SUMMARY_CSV}")
print(f"Artifact audit: {SCREENING_ARTIFACT_AUDIT_CSV}")
print(f"Consolidation summary: {SCREENING_CONSOLIDATION_JSON}")

if not ALLOW_PARTIAL_SCREENING_CONSOLIDATION and len(screening_results) != 10:
    raise RuntimeError(f"Expected 10 screening results, found {len(screening_results)}")

if len(selected_screening_models) != min(TOP_N_FOR_FULL_TRAINING, len(screening_results)):
    raise RuntimeError(
        f"Expected {TOP_N_FOR_FULL_TRAINING} selected models, found {len(selected_screening_models)}"
    )

screening_result_run_ids = set(screening_results["run_id"].astype(str))

missing_artifacts = screening_artifact_audit[
    screening_artifact_audit["run_id"].astype(str).isin(screening_result_run_ids)
    & ~(
        screening_artifact_audit["result_csv_exists"]
        & screening_artifact_audit["best_checkpoint_exists"]
        & screening_artifact_audit["training_log_exists"]
        & screening_artifact_audit["val_predictions_exists"]
    )
]

if len(missing_artifacts) > 0:
    print("\nRuns with missing required artifacts:")
    print(missing_artifacts.to_string(index=False))
    raise RuntimeError("One or more completed screening runs have missing required artifacts.")

if not screening_results["screening_rank"].is_unique:
    raise RuntimeError("Screening ranks are not unique.")

if screening_results["selection_score"].isna().any():
    raise RuntimeError("Selection score contains NaN values.")

if "best_patient_val_auroc" in screening_results.columns:
    finite_patient_metric_count = int(
        np.isfinite(pd.to_numeric(screening_results["best_patient_val_auroc"], errors="coerce")).sum()
    )

    if finite_patient_metric_count > 0:
        raise RuntimeError(
            "Unexpected finite true patient-level metrics found. "
            "AIROGS-light-v2 internal screening must remain image-level."
        )

print("Status: PASS")

In [ ]:
# Cell 10: Configure AIROGS-light full-training plan using the best classifier from each architectural family

SCREENING_RANKING_CSV = RESULTS_07_DIR / "notebook07_architecture_screening_ranking.csv"

FULL_TRAINING_PLAN_CSV = RESULTS_07_DIR / "notebook07_full_training_plan.csv"
FULL_TRAINING_CONFIG_JSON = RESULTS_07_DIR / "notebook07_full_training_config.json"
FULL_TRAINING_SELECTION_CSV = RESULTS_07_DIR / "notebook07_selected_classifiers_for_full_training.csv"

if not SCREENING_RANKING_CSV.exists():
    raise FileNotFoundError(f"Missing screening ranking: {SCREENING_RANKING_CSV}")

screening_ranking = pd.read_csv(SCREENING_RANKING_CSV)

required_ranking_cols = [
    "screening_rank",
    "model_name",
    "family",
    "timm_model_name",
    "best_image_val_auroc",
    "best_image_val_auprc",
    "best_image_val_balanced_accuracy",
    "best_image_val_brier_score",
    "best_image_val_sensitivity",
    "best_image_val_specificity",
    "best_image_val_f1",
    "selection_score",
    "decision",
    "best_checkpoint",
    "training_log",
    "val_predictions",
]

missing_ranking_cols = [
    col for col in required_ranking_cols
    if col not in screening_ranking.columns
]

if missing_ranking_cols:
    raise ValueError(f"Missing screening ranking columns: {missing_ranking_cols}")

screening_ranking = screening_ranking.copy()

numeric_ranking_cols = [
    "screening_rank",
    "best_image_val_auroc",
    "best_image_val_auprc",
    "best_image_val_balanced_accuracy",
    "best_image_val_brier_score",
    "best_image_val_sensitivity",
    "best_image_val_specificity",
    "best_image_val_f1",
    "selection_score",
]

for col in numeric_ranking_cols:
    screening_ranking[col] = pd.to_numeric(screening_ranking[col], errors="coerce")

if not np.isfinite(screening_ranking["selection_score"]).all():
    raise RuntimeError("Screening ranking contains non-finite selection scores.")

if "internal_patient_level_claim_allowed" in screening_ranking.columns:
    screening_ranking["internal_patient_level_claim_allowed"] = as_bool_series(
        screening_ranking["internal_patient_level_claim_allowed"]
    )

    if screening_ranking["internal_patient_level_claim_allowed"].any():
        raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

if "primary_metric_level" in screening_ranking.columns:
    bad_metric_level_rows = screening_ranking[
        screening_ranking["primary_metric_level"].astype(str) != "image_level"
    ]

    if len(bad_metric_level_rows) > 0:
        print(bad_metric_level_rows[["model_name", "primary_metric_level"]].to_string(index=False))
        raise RuntimeError("Full-training candidates must come from image-level AIROGS-light screening.")

if "primary_selection_metric" in screening_ranking.columns:
    bad_primary_metric_rows = screening_ranking[
        screening_ranking["primary_selection_metric"].astype(str) != "image_val_auroc"
    ]

    if len(bad_primary_metric_rows) > 0:
        print(bad_primary_metric_rows[["model_name", "primary_selection_metric"]].to_string(index=False))
        raise RuntimeError("Full-training candidates must come from image_val_auroc screening.")

# Formal JBHI-oriented policy:
# Keep the best model from each architectural family.
# This preserves architectural diversity and makes the comparison more defensible.
FULL_TRAINING_SELECTION_POLICY = "best_model_per_architectural_family_by_screening_selection_score"

screening_ranking_sorted = (
    screening_ranking
    .sort_values(
        [
            "family",
            "selection_score",
            "best_image_val_auroc",
            "best_image_val_auprc",
            "best_image_val_balanced_accuracy",
            "best_image_val_brier_score",
        ],
        ascending=[True, False, False, False, False, True],
    )
    .reset_index(drop=True)
)

selected_for_full_training = (
    screening_ranking_sorted
    .groupby("family", as_index=False)
    .head(1)
    .copy()
    .sort_values(
        [
            "selection_score",
            "best_image_val_auroc",
            "best_image_val_auprc",
            "best_image_val_balanced_accuracy",
        ],
        ascending=[False, False, False, False],
    )
    .reset_index(drop=True)
)

# Backup candidates are the non-selected models from families that had more than one candidate.
selected_run_ids = set(selected_for_full_training["run_id"].astype(str)) if "run_id" in selected_for_full_training.columns else set()

if "run_id" in screening_ranking.columns:
    backup_candidates = (
        screening_ranking[
            ~screening_ranking["run_id"].astype(str).isin(selected_run_ids)
        ]
        .copy()
        .sort_values(
            [
                "family",
                "selection_score",
                "best_image_val_auroc",
                "best_image_val_auprc",
            ],
            ascending=[True, False, False, False],
        )
        .reset_index(drop=True)
    )
else:
    selected_keys = set(
        zip(
            selected_for_full_training["model_name"].astype(str),
            selected_for_full_training["family"].astype(str),
        )
    )

    backup_candidates = (
        screening_ranking[
            ~screening_ranking.apply(
                lambda row: (str(row["model_name"]), str(row["family"])) in selected_keys,
                axis=1,
            )
        ]
        .copy()
        .sort_values(
            [
                "family",
                "selection_score",
                "best_image_val_auroc",
                "best_image_val_auprc",
            ],
            ascending=[True, False, False, False],
        )
        .reset_index(drop=True)
    )

selected_for_full_training["full_training_decision"] = "selected_best_model_per_family"
backup_candidates["full_training_decision"] = "backup_non_selected_same_family_or_lower_ranked"

selection_table = pd.concat(
    [
        selected_for_full_training,
        backup_candidates,
    ],
    axis=0,
    ignore_index=True,
)

selection_table["primary_training_dataset"] = "EyePACS-AIROGS-light-V2"
selection_table["primary_metric_level"] = "image_level"
selection_table["primary_selection_metric"] = "image_val_auroc"
selection_table["true_patient_id_available"] = False
selection_table["internal_patient_level_claim_allowed"] = False

full_training_rows = []

for _, row in selected_for_full_training.iterrows():
    model_name = str(row["model_name"])
    run_id = f"{model_name}__airogs_light_full_familywinner_seed{SEED}"

    paths = get_classifier_paths(
        root_dir=FULL_TRAINING_ROOT,
        model_name=model_name,
        run_id=run_id,
    )

    full_training_rows.append(
        {
            "run_id": run_id,
            "model_name": model_name,
            "family": str(row["family"]),
            "timm_model_name": str(row["timm_model_name"]),
            "screening_rank": int(row["screening_rank"]),
            "screening_selection_score": float(row["selection_score"]),

            "screening_image_val_auroc": float(row["best_image_val_auroc"]),
            "screening_image_val_auprc": float(row["best_image_val_auprc"]),
            "screening_image_val_balanced_accuracy": float(row["best_image_val_balanced_accuracy"]),
            "screening_image_val_brier_score": float(row["best_image_val_brier_score"]),
            "screening_image_val_sensitivity": float(row["best_image_val_sensitivity"]),
            "screening_image_val_specificity": float(row["best_image_val_specificity"]),
            "screening_image_val_f1": float(row["best_image_val_f1"]),

            "stage": "full_training",
            "primary_training_dataset": "EyePACS-AIROGS-light-V2",
            "split_policy": "official_train_val_test_image_level",
            "selection_policy": FULL_TRAINING_SELECTION_POLICY,
            "input_size": int(FULL_INPUT_SIZE),
            "batch_size": int(FULL_BATCH_SIZE),
            "max_epochs": int(FULL_MAX_EPOCHS),
            "early_stop_patience": int(FULL_EARLY_STOP_PATIENCE),
            "learning_rate": float(LEARNING_RATE),
            "weight_decay": float(WEIGHT_DECAY),
            "use_pretrained_weights": bool(USE_PRETRAINED_WEIGHTS),
            "primary_selection_metric": PRIMARY_SELECTION_METRIC,
            "primary_metric_level": "image_level",

            "train_images": int(len(train_df)),
            "val_images": int(len(val_df)),
            "test_images_locked_not_used": int(len(test_df)),

            "train_rg_images": int((train_df["gon_binary"].astype(int) == 1).sum()),
            "train_nrg_images": int((train_df["gon_binary"].astype(int) == 0).sum()),
            "val_rg_images": int((val_df["gon_binary"].astype(int) == 1).sum()),
            "val_nrg_images": int((val_df["gon_binary"].astype(int) == 0).sum()),
            "test_rg_images_locked_not_used": int((test_df["gon_binary"].astype(int) == 1).sum()),
            "test_nrg_images_locked_not_used": int((test_df["gon_binary"].astype(int) == 0).sum()),

            "true_patient_id_available": False,
            "internal_patient_level_claim_allowed": False,
            "train_pseudo_patients_for_loader_compatibility": int(train_df["patient_id"].nunique()),
            "val_pseudo_patients_for_loader_compatibility": int(val_df["patient_id"].nunique()),
            "test_pseudo_patients_locked_not_used": int(test_df["patient_id"].nunique()),

            "quality_score_available": False,
            "anatomical_proxy_available_for_training_dataset": False,
            "external_patient_level_evaluation_reserved_for": "HYGD",
            "external_quality_evaluation_reserved_for": "HYGD",
            "external_domain_shift_evaluation_reserved_for": "HYGD|PAPILA|SMDG",

            "run_dir": str(paths["run_dir"]),
            "latest_checkpoint": str(paths["latest_checkpoint"]),
            "best_checkpoint": str(paths["best_checkpoint"]),
            "training_log": str(paths["training_log_csv"]),
            "val_predictions": str(paths["val_predictions_csv"]),
            "val_pseudo_patient_predictions": str(paths["val_pseudo_patient_predictions_csv"]),

            # Backward-compatible alias for later cells that may still expect this column name.
            # This is not a true patient-level artifact.
            "val_patient_predictions": str(paths["val_pseudo_patient_predictions_csv"]),
        }
    )

full_training_plan = pd.DataFrame(full_training_rows)

selection_table.to_csv(FULL_TRAINING_SELECTION_CSV, index=False)
full_training_plan.to_csv(FULL_TRAINING_PLAN_CSV, index=False)

selected_family_counts = (
    selected_for_full_training
    .groupby("family")
    .size()
    .reset_index(name="n_selected_models")
    .sort_values("family")
    .reset_index(drop=True)
)

backup_family_counts = (
    backup_candidates
    .groupby("family")
    .size()
    .reset_index(name="n_backup_models")
    .sort_values("family")
    .reset_index(drop=True)
)

n_unique_families_in_screening = int(screening_ranking["family"].nunique())
n_selected_families = int(selected_for_full_training["family"].nunique())

full_training_config = {
    "stage": "full_training",
    "purpose": (
        "Train the best AIROGS-light classifier from each architectural family more deeply "
        "at higher image resolution before validation-only final model selection."
    ),
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "split_policy": "official_train_val_test_image_level",
    "selection_policy": FULL_TRAINING_SELECTION_POLICY,
    "n_screened_models": int(len(screening_ranking)),
    "n_unique_families_in_screening": n_unique_families_in_screening,
    "n_selected_models": int(len(selected_for_full_training)),
    "n_selected_families": n_selected_families,
    "selected_models": selected_for_full_training["model_name"].tolist(),
    "selected_families": selected_for_full_training["family"].tolist(),
    "selected_family_counts": selected_family_counts.to_dict(orient="records"),
    "backup_candidates": backup_candidates["model_name"].tolist(),
    "backup_families": backup_candidates["family"].tolist(),
    "backup_family_counts": backup_family_counts.to_dict(orient="records"),
    "selection_rule": (
        "Within each architectural family, the model with the highest AIROGS-light screening "
        "selection score is selected for full training. This prevents the full-training comparison "
        "from being dominated by multiple variants of the same family and supports a family-level "
        "comparison suitable for the JBHI manuscript."
    ),
    "screening_selection_score_formula": (
        "0.35*image_AUROC + 0.25*image_AUPRC + 0.20*image_balanced_accuracy "
        "+ 0.10*image_sensitivity + 0.10*image_specificity "
        "- 0.08*image_Brier - 0.02*normalized_elapsed_min"
    ),
    "full_training_input_size": int(FULL_INPUT_SIZE),
    "full_training_batch_size": int(FULL_BATCH_SIZE),
    "full_training_max_epochs": int(FULL_MAX_EPOCHS),
    "full_training_early_stop_patience": int(FULL_EARLY_STOP_PATIENCE),
    "learning_rate": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "use_pretrained_weights": bool(USE_PRETRAINED_WEIGHTS),
    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
    "primary_metric_level": "image_level",
    "train_sampling": "WeightedRandomSampler balanced by image-level class",
    "test_set_policy": "The locked AIROGS-light test split remains unused during full-training model selection.",
    "checkpoint_policy": "latest checkpoint every epoch and best checkpoint on image-level validation AUROC improvement",
    "resume_policy": "automatic resume from latest checkpoint when available",
    "true_patient_id_available": False,
    "internal_patient_level_claim_allowed": False,
    "pseudo_patient_policy": (
        "Pseudo-patient IDs are retained only for compatibility with previous code. "
        "They are not valid patient-level evidence."
    ),
    "external_aqpr_policy": {
        "HYGD": "reserved for external patient-level, quality, anatomical plausibility, and calibration audits",
        "PAPILA": "reserved for external anatomy/domain-shift and label-policy audits",
        "SMDG": "reserved for curated multi-source external domain-shift audit",
    },
    "full_training_plan_csv": str(FULL_TRAINING_PLAN_CSV),
    "selected_classifiers_csv": str(FULL_TRAINING_SELECTION_CSV),
    "full_training_root": str(FULL_TRAINING_ROOT),
}

with open(FULL_TRAINING_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(full_training_config, f, indent=2)

print("Selected classifiers for AIROGS-light full training: best model per architectural family")
print(
    selected_for_full_training[
        [
            "screening_rank",
            "model_name",
            "family",
            "best_image_val_auroc",
            "best_image_val_auprc",
            "best_image_val_balanced_accuracy",
            "best_image_val_brier_score",
            "best_image_val_sensitivity",
            "best_image_val_specificity",
            "selection_score",
            "full_training_decision",
        ]
    ].to_string(index=False)
)

print("\nBackup/non-selected candidates:")
if len(backup_candidates) > 0:
    print(
        backup_candidates[
            [
                "screening_rank",
                "model_name",
                "family",
                "best_image_val_auroc",
                "best_image_val_auprc",
                "best_image_val_balanced_accuracy",
                "best_image_val_brier_score",
                "selection_score",
                "full_training_decision",
            ]
        ].to_string(index=False)
    )
else:
    print("None")

print("\nSelected family counts:")
print(selected_family_counts.to_string(index=False))

print("\nBackup family counts:")
if len(backup_family_counts) > 0:
    print(backup_family_counts.to_string(index=False))
else:
    print("None")

print("\nAIROGS-light full-training plan:")
print(
    full_training_plan[
        [
            "run_id",
            "model_name",
            "family",
            "input_size",
            "batch_size",
            "max_epochs",
            "early_stop_patience",
            "primary_selection_metric",
            "primary_metric_level",
        ]
    ].to_string(index=False)
)

print(f"\nSelected classifiers CSV: {FULL_TRAINING_SELECTION_CSV}")
print(f"Full-training plan CSV: {FULL_TRAINING_PLAN_CSV}")
print(f"Full-training config: {FULL_TRAINING_CONFIG_JSON}")

if n_selected_families != n_unique_families_in_screening:
    raise RuntimeError(
        f"Expected one selected model per family. "
        f"Screening families={n_unique_families_in_screening}, selected families={n_selected_families}"
    )

if len(selected_for_full_training) != n_unique_families_in_screening:
    raise RuntimeError(
        f"Expected {n_unique_families_in_screening} selected models, "
        f"found {len(selected_for_full_training)}"
    )

if len(full_training_plan) != n_unique_families_in_screening:
    raise RuntimeError(
        f"Expected {n_unique_families_in_screening} full-training runs, "
        f"found {len(full_training_plan)}"
    )

if full_training_plan["run_id"].duplicated().any():
    raise RuntimeError("Duplicate full-training run IDs found.")

if selected_for_full_training["family"].duplicated().any():
    duplicated_families = selected_for_full_training.loc[
        selected_for_full_training["family"].duplicated(),
        "family",
    ].tolist()

    raise RuntimeError(f"More than one selected model found for family: {duplicated_families}")

if not full_training_plan["primary_selection_metric"].eq("image_val_auroc").all():
    raise RuntimeError("Full-training plan must use image_val_auroc as primary metric.")

if not full_training_plan["primary_metric_level"].eq("image_level").all():
    raise RuntimeError("Full-training plan must remain image-level for AIROGS-light.")

if full_training_plan["internal_patient_level_claim_allowed"].any():
    raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

expected_counts = {
    "train_images": 8000,
    "val_images": 770,
    "test_images_locked_not_used": 770,
    "train_rg_images": 4000,
    "train_nrg_images": 4000,
    "val_rg_images": 385,
    "val_nrg_images": 385,
    "test_rg_images_locked_not_used": 385,
    "test_nrg_images_locked_not_used": 385,
}

for column, expected_value in expected_counts.items():
    observed_values = full_training_plan[column].unique().tolist()

    if observed_values != [expected_value]:
        raise RuntimeError(
            f"Unexpected {column}: expected all {expected_value}, observed {observed_values}"
        )

for artifact_col in [
    "run_dir",
    "latest_checkpoint",
    "best_checkpoint",
    "training_log",
    "val_predictions",
    "val_pseudo_patient_predictions",
]:
    if artifact_col not in full_training_plan.columns:
        raise RuntimeError(f"Missing artifact column in full-training plan: {artifact_col}")

print("Status: PASS")

In [ ]:
# Cell 11: Run one resumable AIROGS-light full-training job

FULL_TRAINING_PLAN_CSV = RESULTS_07_DIR / "notebook07_full_training_plan.csv"

if not FULL_TRAINING_PLAN_CSV.exists():
    raise FileNotFoundError(f"Missing full-training plan: {FULL_TRAINING_PLAN_CSV}")

full_training_plan = pd.read_csv(FULL_TRAINING_PLAN_CSV)

required_full_training_columns = [
    "run_id",
    "model_name",
    "family",
    "timm_model_name",
    "input_size",
    "batch_size",
    "max_epochs",
    "early_stop_patience",
    "use_pretrained_weights",
    "primary_selection_metric",
    "primary_metric_level",
    "best_checkpoint",
    "latest_checkpoint",
    "training_log",
    "val_predictions",
    "val_pseudo_patient_predictions",
]

missing_full_training_columns = [
    col for col in required_full_training_columns
    if col not in full_training_plan.columns
]

if missing_full_training_columns:
    raise ValueError(f"Missing full-training plan columns: {missing_full_training_columns}")

# Backward-compatible alias for later cells that may still expect val_patient_predictions.
# This is not a true patient-level artifact.
if "val_patient_predictions" not in full_training_plan.columns:
    full_training_plan["val_patient_predictions"] = full_training_plan["val_pseudo_patient_predictions"]

if not full_training_plan["primary_selection_metric"].astype(str).eq("image_val_auroc").all():
    raise RuntimeError("AIROGS-light full training must use image_val_auroc as primary metric.")

if not full_training_plan["primary_metric_level"].astype(str).eq("image_level").all():
    raise RuntimeError("AIROGS-light full training must remain image-level.")

if "internal_patient_level_claim_allowed" in full_training_plan.columns:
    full_training_plan["internal_patient_level_claim_allowed"] = as_bool_series(
        full_training_plan["internal_patient_level_claim_allowed"]
    )

    if full_training_plan["internal_patient_level_claim_allowed"].any():
        raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

# Leave this as None to automatically train the next pending full-training run.
# To force a specific run, use for example:
# FULL_RUN_ID_TO_TRAIN = "ConvNeXtTiny__airogs_light_full_familywinner_seed42"
FULL_RUN_ID_TO_TRAIN = None

# Keep True to avoid accidental CPU training.
SKIP_TRAINING_IF_NO_CUDA = True

FULL_TRAINING_RESULT_DIR = RESULTS_07_DIR / "full_training_run_results"
FULL_TRAINING_RESULT_DIR.mkdir(parents=True, exist_ok=True)


def parse_bool_value(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, (int, np.integer)):
        return bool(value)

    if isinstance(value, float):
        if np.isnan(value):
            return False
        return bool(int(value))

    value_str = str(value).strip().lower()

    if value_str in {"true", "1", "yes", "y"}:
        return True

    if value_str in {"false", "0", "no", "n", "nan", "none", ""}:
        return False

    raise ValueError(f"Cannot parse boolean value: {value}")


def full_training_result_path(run_id):
    safe_run_id = str(run_id).replace("/", "_")
    return FULL_TRAINING_RESULT_DIR / f"notebook07_full_training_result_{safe_run_id}.csv"


def full_training_run_completed(plan_row):
    run_id = str(plan_row["run_id"])

    result_csv = full_training_result_path(run_id)
    best_checkpoint = Path(str(plan_row["best_checkpoint"]))
    training_log = Path(str(plan_row["training_log"]))
    val_predictions = Path(str(plan_row["val_predictions"]))
    val_pseudo_patient_predictions = Path(str(plan_row["val_pseudo_patient_predictions"]))

    if not result_csv.exists():
        return False

    if not best_checkpoint.exists():
        return False

    if not training_log.exists():
        return False

    if not val_predictions.exists():
        return False

    if not val_pseudo_patient_predictions.exists():
        return False

    try:
        result_df = pd.read_csv(result_csv)
        log_df = pd.read_csv(training_log)
        val_pred_df = pd.read_csv(val_predictions)
        val_pseudo_df = pd.read_csv(val_pseudo_patient_predictions)
    except Exception:
        return False

    if len(result_df) != 1:
        return False

    if len(log_df) == 0:
        return False

    if len(val_pred_df) == 0:
        return False

    if len(val_pseudo_df) == 0:
        return False

    required_result_columns = [
        "best_primary_metric",
        "best_image_val_auroc",
        "best_image_val_auprc",
        "best_image_val_balanced_accuracy",
        "best_image_val_brier_score",
        "best_checkpoint",
        "training_log",
        "val_predictions",
        "val_pseudo_patient_predictions",
    ]

    missing_result_columns = [
        col for col in required_result_columns
        if col not in result_df.columns
    ]

    if missing_result_columns:
        return False

    if not np.isfinite(pd.to_numeric(result_df["best_primary_metric"], errors="coerce").iloc[0]):
        return False

    if not np.isfinite(pd.to_numeric(result_df["best_image_val_auroc"], errors="coerce").iloc[0]):
        return False

    if str(result_df["primary_selection_metric"].iloc[0]) != "image_val_auroc":
        return False

    if bool(result_df["internal_patient_level_claim_allowed"].iloc[0]):
        return False

    if not Path(str(result_df["best_checkpoint"].iloc[0])).exists():
        return False

    if not Path(str(result_df["training_log"].iloc[0])).exists():
        return False

    if not Path(str(result_df["val_predictions"].iloc[0])).exists():
        return False

    if not Path(str(result_df["val_pseudo_patient_predictions"].iloc[0])).exists():
        return False

    return True


full_training_plan["result_csv"] = full_training_plan["run_id"].apply(
    lambda run_id: str(full_training_result_path(run_id))
)

full_training_plan["completed"] = full_training_plan.apply(
    full_training_run_completed,
    axis=1,
)

pending_full_runs = (
    full_training_plan[
        ~full_training_plan["completed"]
    ]
    .copy()
    .reset_index(drop=True)
)

if FULL_RUN_ID_TO_TRAIN is None:
    if len(pending_full_runs) == 0:
        selected_full_run = None
    else:
        selected_full_run = pending_full_runs.iloc[0].to_dict()
else:
    matching_runs = full_training_plan[
        full_training_plan["run_id"] == FULL_RUN_ID_TO_TRAIN
    ].copy()

    if len(matching_runs) != 1:
        raise ValueError(f"FULL_RUN_ID_TO_TRAIN not found or not unique: {FULL_RUN_ID_TO_TRAIN}")

    selected_full_run = matching_runs.iloc[0].to_dict()

full_training_progress_csv = RESULTS_07_DIR / "notebook07_full_training_progress.csv"
full_training_plan.to_csv(full_training_progress_csv, index=False)

print("AIROGS-light full-training progress:")
print(
    full_training_plan[
        [
            "run_id",
            "model_name",
            "family",
            "primary_selection_metric",
            "primary_metric_level",
            "completed",
            "result_csv",
        ]
    ].to_string(index=False)
)

print(f"\nCompleted full-training runs: {int(full_training_plan['completed'].sum())}/{len(full_training_plan)}")
print(f"Pending full-training runs: {len(pending_full_runs)}")
print(f"Progress CSV: {full_training_progress_csv}")

if selected_full_run is None:
    print("\nAll AIROGS-light full-training runs are already completed.")
    print("Status: PASS")

else:
    selected_full_run_pretrained = parse_bool_value(selected_full_run["use_pretrained_weights"])

    selected_full_run_summary = {
        "run_id": selected_full_run["run_id"],
        "model_name": selected_full_run["model_name"],
        "family": selected_full_run["family"],
        "timm_model_name": selected_full_run["timm_model_name"],
        "input_size": int(selected_full_run["input_size"]),
        "batch_size": int(selected_full_run["batch_size"]),
        "max_epochs": int(selected_full_run["max_epochs"]),
        "early_stop_patience": int(selected_full_run["early_stop_patience"]),
        "use_pretrained_weights": bool(selected_full_run_pretrained),
        "primary_selection_metric": selected_full_run["primary_selection_metric"],
        "primary_metric_level": selected_full_run["primary_metric_level"],
        "primary_training_dataset": "EyePACS-AIROGS-light-V2",
        "selection_policy": selected_full_run.get(
            "selection_policy",
            "best_model_per_architectural_family_by_screening_selection_score",
        ),
        "device": str(DEVICE),
        "cuda_available": bool(torch.cuda.is_available()),
        "internal_patient_level_claim_allowed": False,
    }

    print("\nSelected full-training run:")
    print(json.dumps(selected_full_run_summary, indent=2))

    if DEVICE.type != "cuda" and SKIP_TRAINING_IF_NO_CUDA:
        print("\nTraining skipped because CUDA is not available.")
        print("Set SKIP_TRAINING_IF_NO_CUDA = False only if CPU training is intentional.")
        print("Status: PASS")

    else:
        run_start_time = time.time()

        result = run_classifier_training_loop(
            model_name=str(selected_full_run["model_name"]),
            run_id=str(selected_full_run["run_id"]),
            root_dir=FULL_TRAINING_ROOT,
            input_size=int(selected_full_run["input_size"]),
            batch_size=int(selected_full_run["batch_size"]),
            max_epochs=int(selected_full_run["max_epochs"]),
            patience=int(selected_full_run["early_stop_patience"]),
            pretrained=bool(selected_full_run_pretrained),
            stage="full_training",
        )

        result["elapsed_min"] = round((time.time() - run_start_time) / 60.0, 3)
        result["primary_training_dataset"] = "EyePACS-AIROGS-light-V2"
        result["primary_metric_level"] = "image_level"
        result["primary_selection_metric"] = "image_val_auroc"
        result["selection_policy"] = selected_full_run.get(
            "selection_policy",
            "best_model_per_architectural_family_by_screening_selection_score",
        )
        result["true_patient_id_available"] = False
        result["internal_patient_level_claim_allowed"] = False

        # Backward-compatible alias. This remains pseudo-patient only.
        if "val_pseudo_patient_predictions" in result:
            result["val_patient_predictions"] = result["val_pseudo_patient_predictions"]

        result_df = pd.DataFrame([result])
        result_csv = full_training_result_path(selected_full_run["run_id"])
        result_df.to_csv(result_csv, index=False)

        print("\nAIROGS-light full-training run result:")
        print(
            result_df[
                [
                    "model_name",
                    "family",
                    "best_epoch_one_indexed",
                    "primary_selection_metric",
                    "best_primary_metric",
                    "best_image_val_auroc",
                    "best_image_val_auprc",
                    "best_image_val_balanced_accuracy",
                    "best_image_val_brier_score",
                    "best_image_val_sensitivity",
                    "best_image_val_specificity",
                    "best_image_val_f1",
                    "epochs_ran",
                    "elapsed_min",
                    "internal_patient_level_claim_allowed",
                ]
            ].to_string(index=False)
        )

        print(f"\nRun result CSV: {result_csv}")
        print(f"Best checkpoint: {result['best_checkpoint']}")
        print(f"Training log: {result['training_log']}")
        print(f"Validation image predictions: {result['val_predictions']}")
        print(f"Validation pseudo-patient predictions: {result['val_pseudo_patient_predictions']}")

        if not Path(result["best_checkpoint"]).exists():
            raise RuntimeError("Best checkpoint was not saved.")

        if not Path(result["training_log"]).exists():
            raise RuntimeError("Training log was not saved.")

        if not Path(result["val_predictions"]).exists():
            raise RuntimeError("Validation image predictions were not saved.")

        if not Path(result["val_pseudo_patient_predictions"]).exists():
            raise RuntimeError("Validation pseudo-patient predictions were not saved.")

        if not np.isfinite(float(result["best_primary_metric"])):
            raise RuntimeError("Best primary validation metric is not finite.")

        if not np.isfinite(float(result["best_image_val_auroc"])):
            raise RuntimeError("Best image-level validation AUROC is not finite.")

        if result["primary_selection_metric"] != "image_val_auroc":
            raise RuntimeError("Unexpected primary metric in full-training result.")

        if bool(result["internal_patient_level_claim_allowed"]):
            raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

        print("Status: PASS")

In [ ]:
# Cell 12: Consolidate AIROGS-light full-training results and select final classifier using validation only

FULL_TRAINING_PLAN_CSV = RESULTS_07_DIR / "notebook07_full_training_plan.csv"
FULL_TRAINING_PROGRESS_CSV = RESULTS_07_DIR / "notebook07_full_training_progress.csv"

FULL_TRAINING_RESULTS_CONSOLIDATED_CSV = RESULTS_07_DIR / "notebook07_full_training_results_consolidated.csv"
FULL_TRAINING_RANKING_CSV = RESULTS_07_DIR / "notebook07_full_training_validation_ranking.csv"
FULL_TRAINING_ARTIFACT_AUDIT_CSV = RESULTS_07_DIR / "notebook07_full_training_artifact_audit.csv"
FULL_TRAINING_FAMILY_SUMMARY_CSV = RESULTS_07_DIR / "notebook07_full_training_family_summary.csv"

FINAL_CLASSIFIER_DECISION_JSON = RESULTS_07_DIR / "notebook07_validation_selected_final_classifier.json"
FINAL_CLASSIFIER_DECISION_CSV = RESULTS_07_DIR / "notebook07_validation_selected_final_classifier.csv"

if FULL_TRAINING_PROGRESS_CSV.exists():
    full_training_plan_runtime = pd.read_csv(FULL_TRAINING_PROGRESS_CSV)
elif FULL_TRAINING_PLAN_CSV.exists():
    full_training_plan_runtime = pd.read_csv(FULL_TRAINING_PLAN_CSV)
else:
    raise FileNotFoundError("Missing full-training plan/progress CSV.")

full_training_plan_runtime = full_training_plan_runtime.copy().reset_index(drop=True)

FULL_TRAINING_RESULT_DIR = RESULTS_07_DIR / "full_training_run_results"
FULL_TRAINING_RESULT_DIR.mkdir(parents=True, exist_ok=True)

if "result_csv" not in full_training_plan_runtime.columns:
    full_training_plan_runtime["result_csv"] = full_training_plan_runtime["run_id"].apply(
        lambda run_id: str(
            FULL_TRAINING_RESULT_DIR / f"notebook07_full_training_result_{str(run_id).replace('/', '_')}.csv"
        )
    )

if "val_pseudo_patient_predictions" not in full_training_plan_runtime.columns:
    if "val_patient_predictions" in full_training_plan_runtime.columns:
        full_training_plan_runtime["val_pseudo_patient_predictions"] = full_training_plan_runtime["val_patient_predictions"]
    else:
        full_training_plan_runtime["val_pseudo_patient_predictions"] = ""

if "val_patient_predictions" not in full_training_plan_runtime.columns:
    full_training_plan_runtime["val_patient_predictions"] = full_training_plan_runtime["val_pseudo_patient_predictions"]

required_plan_cols = [
    "run_id",
    "model_name",
    "family",
    "timm_model_name",
    "result_csv",
    "best_checkpoint",
    "latest_checkpoint",
    "training_log",
    "val_predictions",
    "val_pseudo_patient_predictions",
]

missing_plan_cols = [
    col for col in required_plan_cols
    if col not in full_training_plan_runtime.columns
]

if missing_plan_cols:
    raise ValueError(f"Missing full-training plan columns: {missing_plan_cols}")


def path_exists_or_blank(path_value):
    path_text = str(path_value).strip()

    if path_text in {"", "nan", "None"}:
        return False

    return Path(path_text).exists()


result_rows = []
artifact_rows = []

for _, plan_row in full_training_plan_runtime.iterrows():
    run_id = str(plan_row["run_id"])

    result_csv = Path(str(plan_row["result_csv"]))
    best_checkpoint = Path(str(plan_row["best_checkpoint"]))
    latest_checkpoint = Path(str(plan_row["latest_checkpoint"]))
    training_log = Path(str(plan_row["training_log"]))
    val_predictions = Path(str(plan_row["val_predictions"]))

    val_pseudo_patient_predictions_text = str(plan_row["val_pseudo_patient_predictions"]).strip()
    val_pseudo_patient_predictions = (
        Path(val_pseudo_patient_predictions_text)
        if val_pseudo_patient_predictions_text not in {"", "nan", "None"}
        else None
    )

    artifact_rows.append(
        {
            "run_id": run_id,
            "model_name": str(plan_row["model_name"]),
            "family": str(plan_row["family"]),
            "result_csv": str(result_csv),
            "result_csv_exists": result_csv.exists(),
            "best_checkpoint": str(best_checkpoint),
            "best_checkpoint_exists": best_checkpoint.exists(),
            "latest_checkpoint": str(latest_checkpoint),
            "latest_checkpoint_exists": latest_checkpoint.exists(),
            "training_log": str(training_log),
            "training_log_exists": training_log.exists(),
            "val_predictions": str(val_predictions),
            "val_predictions_exists": val_predictions.exists(),
            "val_pseudo_patient_predictions": (
                str(val_pseudo_patient_predictions)
                if val_pseudo_patient_predictions is not None
                else ""
            ),
            "val_pseudo_patient_predictions_exists": (
                val_pseudo_patient_predictions.exists()
                if val_pseudo_patient_predictions is not None
                else False
            ),
        }
    )

    if not result_csv.exists():
        continue

    result_df = pd.read_csv(result_csv)

    if len(result_df) != 1:
        raise RuntimeError(f"Expected one result row for {run_id}, found {len(result_df)}")

    result_row = result_df.iloc[0].to_dict()

    for key in [
        "run_id",
        "model_name",
        "family",
        "timm_model_name",
        "best_checkpoint",
        "latest_checkpoint",
        "training_log",
        "val_predictions",
        "val_pseudo_patient_predictions",
    ]:
        if key not in result_row or pd.isna(result_row[key]):
            result_row[key] = plan_row[key] if key in plan_row.index else ""

    result_row["primary_training_dataset"] = "EyePACS-AIROGS-light-V2"
    result_row["primary_metric_level"] = "image_level"
    result_row["primary_selection_metric"] = "image_val_auroc"
    result_row["true_patient_id_available"] = False
    result_row["internal_patient_level_claim_allowed"] = False

    result_rows.append(result_row)

full_training_artifact_audit = pd.DataFrame(artifact_rows)

if len(result_rows) == 0:
    raise RuntimeError("No full-training result rows found.")

full_training_results = pd.DataFrame(result_rows)

numeric_cols = [
    "best_epoch",
    "best_epoch_one_indexed",
    "best_primary_metric",
    "best_image_val_auroc",
    "best_image_val_auprc",
    "best_image_val_accuracy",
    "best_image_val_balanced_accuracy",
    "best_image_val_brier_score",
    "best_image_val_sensitivity",
    "best_image_val_specificity",
    "best_image_val_f1",
    "best_pseudo_patient_val_auroc",
    "best_pseudo_patient_val_auprc",
    "best_pseudo_patient_val_balanced_accuracy",
    "best_pseudo_patient_val_brier_score",
    "best_patient_val_auroc",
    "best_patient_val_auprc",
    "best_patient_val_balanced_accuracy",
    "best_patient_val_brier_score",
    "epochs_ran",
    "elapsed_min",
]

for col in numeric_cols:
    if col in full_training_results.columns:
        full_training_results[col] = pd.to_numeric(full_training_results[col], errors="coerce")

required_result_cols = [
    "run_id",
    "model_name",
    "family",
    "timm_model_name",
    "best_epoch_one_indexed",
    "best_primary_metric",
    "best_image_val_auroc",
    "best_image_val_auprc",
    "best_image_val_balanced_accuracy",
    "best_image_val_brier_score",
    "best_image_val_sensitivity",
    "best_image_val_specificity",
    "best_image_val_f1",
    "epochs_ran",
    "elapsed_min",
    "best_checkpoint",
    "training_log",
    "val_predictions",
    "val_pseudo_patient_predictions",
]

missing_result_cols = [
    col for col in required_result_cols
    if col not in full_training_results.columns
]

if missing_result_cols:
    raise ValueError(f"Missing full-training result columns: {missing_result_cols}")

finite_metric_cols = [
    "best_primary_metric",
    "best_image_val_auroc",
    "best_image_val_auprc",
    "best_image_val_balanced_accuracy",
    "best_image_val_brier_score",
    "best_image_val_sensitivity",
    "best_image_val_specificity",
    "best_image_val_f1",
]

for col in finite_metric_cols:
    if not np.isfinite(full_training_results[col]).all():
        bad_rows = full_training_results.loc[
            ~np.isfinite(full_training_results[col]),
            ["run_id", "model_name", col],
        ]

        print(f"Rows with non-finite {col}:")
        print(bad_rows.to_string(index=False))

        raise RuntimeError(f"Non-finite values found in {col}")

if "primary_selection_metric" in full_training_results.columns:
    bad_primary_metric_rows = full_training_results[
        full_training_results["primary_selection_metric"].astype(str) != "image_val_auroc"
    ]

    if len(bad_primary_metric_rows) > 0:
        print(bad_primary_metric_rows[["run_id", "model_name", "primary_selection_metric"]].to_string(index=False))
        raise RuntimeError("All full-training runs must use image_val_auroc as primary metric.")

if "primary_metric_level" in full_training_results.columns:
    bad_metric_level_rows = full_training_results[
        full_training_results["primary_metric_level"].astype(str) != "image_level"
    ]

    if len(bad_metric_level_rows) > 0:
        print(bad_metric_level_rows[["run_id", "model_name", "primary_metric_level"]].to_string(index=False))
        raise RuntimeError("All full-training runs must be image-level for AIROGS-light.")

if "internal_patient_level_claim_allowed" in full_training_results.columns:
    full_training_results["internal_patient_level_claim_allowed"] = as_bool_series(
        full_training_results["internal_patient_level_claim_allowed"]
    )

    if full_training_results["internal_patient_level_claim_allowed"].any():
        raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

# True patient-level metrics must not be finite for AIROGS-light internal selection.
for patient_metric_col in [
    "best_patient_val_auroc",
    "best_patient_val_auprc",
    "best_patient_val_balanced_accuracy",
    "best_patient_val_brier_score",
]:
    if patient_metric_col in full_training_results.columns:
        finite_patient_metrics = int(
            np.isfinite(pd.to_numeric(full_training_results[patient_metric_col], errors="coerce")).sum()
        )

        if finite_patient_metrics > 0:
            raise RuntimeError(
                f"Unexpected finite true patient-level metric found in {patient_metric_col}. "
                "AIROGS-light internal selection must remain image-level."
            )

# Optional parameter counts from the model smoke test, used only as a tie-breaker.
MODEL_SMOKE_TEST_CSV = RESULTS_07_DIR / "notebook07_model_registry_smoke_test.csv"

if MODEL_SMOKE_TEST_CSV.exists():
    model_smoke_test = pd.read_csv(MODEL_SMOKE_TEST_CSV)

    if "model_name" in model_smoke_test.columns and "trainable_parameters" in model_smoke_test.columns:
        model_smoke_test = model_smoke_test[
            ["model_name", "trainable_parameters"]
        ].copy()

        model_smoke_test["trainable_parameters"] = pd.to_numeric(
            model_smoke_test["trainable_parameters"],
            errors="coerce",
        )

        full_training_results = full_training_results.merge(
            model_smoke_test,
            on="model_name",
            how="left",
        )
    else:
        full_training_results["trainable_parameters"] = np.nan
else:
    full_training_results["trainable_parameters"] = np.nan

elapsed = full_training_results["elapsed_min"].replace([np.inf, -np.inf], np.nan)

if elapsed.notna().sum() > 1 and elapsed.max() > elapsed.min():
    full_training_results["elapsed_min_norm"] = (
        (elapsed - elapsed.min()) / (elapsed.max() - elapsed.min())
    )
else:
    full_training_results["elapsed_min_norm"] = 0.0

params = full_training_results["trainable_parameters"].replace([np.inf, -np.inf], np.nan)

if params.notna().sum() > 1 and params.max() > params.min():
    full_training_results["trainable_parameters_norm"] = (
        (params - params.min()) / (params.max() - params.min())
    )
else:
    full_training_results["trainable_parameters_norm"] = 0.0

# Validation-only selection score for AIROGS-light.
# The locked test set remains untouched.
# Patient-level metrics are intentionally excluded because AIROGS-light-v2 has no true patient IDs.
full_training_results["full_validation_selection_score"] = (
    0.35 * full_training_results["best_image_val_auroc"]
    + 0.25 * full_training_results["best_image_val_auprc"]
    + 0.18 * full_training_results["best_image_val_balanced_accuracy"]
    + 0.08 * full_training_results["best_image_val_sensitivity"]
    + 0.08 * full_training_results["best_image_val_specificity"]
    + 0.06 * full_training_results["best_image_val_f1"]
    - 0.08 * full_training_results["best_image_val_brier_score"]
    - 0.015 * full_training_results["elapsed_min_norm"].fillna(0.0)
    - 0.015 * full_training_results["trainable_parameters_norm"].fillna(0.0)
)

full_training_ranking = full_training_results.sort_values(
    [
        "full_validation_selection_score",
        "best_image_val_auroc",
        "best_image_val_auprc",
        "best_image_val_balanced_accuracy",
        "best_image_val_brier_score",
        "best_image_val_sensitivity",
        "best_image_val_specificity",
    ],
    ascending=[False, False, False, False, True, False, False],
).reset_index(drop=True)

full_training_ranking["full_training_rank"] = np.arange(1, len(full_training_ranking) + 1)

full_training_ranking["validation_decision"] = np.where(
    full_training_ranking["full_training_rank"] == 1,
    "validation_selected_final_classifier",
    "not_selected_after_full_training",
)

selected_final_classifier = full_training_ranking.iloc[0].copy()

full_training_family_summary = (
    full_training_ranking
    .groupby("family")
    .agg(
        n_models=("model_name", "count"),
        best_model=("model_name", "first"),
        best_rank=("full_training_rank", "min"),
        best_full_validation_selection_score=("full_validation_selection_score", "max"),
        best_image_val_auroc=("best_image_val_auroc", "max"),
        best_image_val_auprc=("best_image_val_auprc", "max"),
        best_image_val_balanced_accuracy=("best_image_val_balanced_accuracy", "max"),
        best_image_val_brier_score=("best_image_val_brier_score", "min"),
        best_image_val_sensitivity=("best_image_val_sensitivity", "max"),
        best_image_val_specificity=("best_image_val_specificity", "max"),
    )
    .reset_index()
    .sort_values(["best_rank", "best_full_validation_selection_score"], ascending=[True, False])
    .reset_index(drop=True)
)

full_training_results.to_csv(FULL_TRAINING_RESULTS_CONSOLIDATED_CSV, index=False)
full_training_ranking.to_csv(FULL_TRAINING_RANKING_CSV, index=False)
full_training_artifact_audit.to_csv(FULL_TRAINING_ARTIFACT_AUDIT_CSV, index=False)
full_training_family_summary.to_csv(FULL_TRAINING_FAMILY_SUMMARY_CSV, index=False)

selected_final_classifier_df = pd.DataFrame([selected_final_classifier.to_dict()])
selected_final_classifier_df.to_csv(FINAL_CLASSIFIER_DECISION_CSV, index=False)

final_classifier_decision = {
    "selection_stage": "validation_only_full_training_selection",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "split_policy": "official_train_val_test_image_level",
    "selection_policy": "best_family_representatives_full_training_then_validation_only_final_selection",
    "selected_model_name": str(selected_final_classifier["model_name"]),
    "selected_family": str(selected_final_classifier["family"]),
    "selected_timm_model_name": str(selected_final_classifier["timm_model_name"]),
    "selected_run_id": str(selected_final_classifier["run_id"]),
    "selected_best_checkpoint": str(selected_final_classifier["best_checkpoint"]),
    "selected_best_epoch_one_indexed": int(selected_final_classifier["best_epoch_one_indexed"]),

    "selected_best_image_val_auroc": float(selected_final_classifier["best_image_val_auroc"]),
    "selected_best_image_val_auprc": float(selected_final_classifier["best_image_val_auprc"]),
    "selected_best_image_val_balanced_accuracy": float(selected_final_classifier["best_image_val_balanced_accuracy"]),
    "selected_best_image_val_brier_score": float(selected_final_classifier["best_image_val_brier_score"]),
    "selected_best_image_val_sensitivity": float(selected_final_classifier["best_image_val_sensitivity"]),
    "selected_best_image_val_specificity": float(selected_final_classifier["best_image_val_specificity"]),
    "selected_best_image_val_f1": float(selected_final_classifier["best_image_val_f1"]),
    "selected_full_validation_score": float(selected_final_classifier["full_validation_selection_score"]),

    "true_patient_id_available": False,
    "internal_patient_level_claim_allowed": False,
    "patient_level_external_evaluation_reserved_for": ["HYGD"],
    "quality_external_evaluation_reserved_for": ["HYGD"],
    "domain_shift_external_evaluation_reserved_for": ["HYGD", "PAPILA", "SMDG"],

    "test_set_policy": (
        "The locked AIROGS-light test split has not been used for model selection. "
        "The selected final classifier will be evaluated on the locked test split in the next cell."
    ),
    "selection_score_formula": (
        "0.35*image_AUROC + 0.25*image_AUPRC + 0.18*image_balanced_accuracy "
        "+ 0.08*image_sensitivity + 0.08*image_specificity + 0.06*image_F1 "
        "- 0.08*image_Brier - 0.015*normalized_elapsed_min "
        "- 0.015*normalized_trainable_parameters"
    ),
    "ranking_csv": str(FULL_TRAINING_RANKING_CSV),
    "family_summary_csv": str(FULL_TRAINING_FAMILY_SUMMARY_CSV),
    "artifact_audit_csv": str(FULL_TRAINING_ARTIFACT_AUDIT_CSV),
    "final_classifier_decision_csv": str(FINAL_CLASSIFIER_DECISION_CSV),
}

with open(FINAL_CLASSIFIER_DECISION_JSON, "w", encoding="utf-8") as f:
    json.dump(final_classifier_decision, f, indent=2)

print("AIROGS-light full-training validation ranking:")
print(
    full_training_ranking[
        [
            "full_training_rank",
            "model_name",
            "family",
            "best_epoch_one_indexed",
            "best_image_val_auroc",
            "best_image_val_auprc",
            "best_image_val_balanced_accuracy",
            "best_image_val_brier_score",
            "best_image_val_sensitivity",
            "best_image_val_specificity",
            "best_image_val_f1",
            "epochs_ran",
            "elapsed_min",
            "trainable_parameters",
            "full_validation_selection_score",
            "validation_decision",
        ]
    ].to_string(index=False)
)

print("\nFamily summary:")
print(full_training_family_summary.to_string(index=False))

print("\nArtifact audit:")
print(
    full_training_artifact_audit[
        [
            "model_name",
            "result_csv_exists",
            "best_checkpoint_exists",
            "training_log_exists",
            "val_predictions_exists",
            "val_pseudo_patient_predictions_exists",
        ]
    ].to_string(index=False)
)

print("\nValidation-selected final classifier:")
print(json.dumps(final_classifier_decision, indent=2))

print(f"\nConsolidated full-training results: {FULL_TRAINING_RESULTS_CONSOLIDATED_CSV}")
print(f"Full-training ranking: {FULL_TRAINING_RANKING_CSV}")
print(f"Family summary: {FULL_TRAINING_FAMILY_SUMMARY_CSV}")
print(f"Final classifier decision CSV: {FINAL_CLASSIFIER_DECISION_CSV}")
print(f"Final classifier decision JSON: {FINAL_CLASSIFIER_DECISION_JSON}")

n_expected_full_training_runs = int(full_training_plan_runtime["family"].nunique())

if len(full_training_ranking) != n_expected_full_training_runs:
    raise RuntimeError(
        f"Expected {n_expected_full_training_runs} full-training results "
        f"from one model per family, found {len(full_training_ranking)}"
    )

if full_training_ranking["family"].duplicated().any():
    duplicated_families = full_training_ranking.loc[
        full_training_ranking["family"].duplicated(),
        "family",
    ].tolist()

    raise RuntimeError(f"More than one full-training result found for family: {duplicated_families}")

missing_artifacts = full_training_artifact_audit[
    ~(
        full_training_artifact_audit["result_csv_exists"]
        & full_training_artifact_audit["best_checkpoint_exists"]
        & full_training_artifact_audit["training_log_exists"]
        & full_training_artifact_audit["val_predictions_exists"]
        & full_training_artifact_audit["val_pseudo_patient_predictions_exists"]
    )
]

if len(missing_artifacts) > 0:
    print("\nRuns with missing artifacts:")
    print(missing_artifacts.to_string(index=False))
    raise RuntimeError("One or more full-training runs have missing artifacts.")

if not Path(str(selected_final_classifier["best_checkpoint"])).exists():
    raise RuntimeError("Selected final classifier checkpoint does not exist.")

if not np.isfinite(float(selected_final_classifier["full_validation_selection_score"])):
    raise RuntimeError("Selected classifier has non-finite validation selection score.")

if not np.isfinite(float(selected_final_classifier["best_image_val_auroc"])):
    raise RuntimeError("Selected classifier has non-finite image-level validation AUROC.")

if bool(selected_final_classifier.get("internal_patient_level_claim_allowed", False)):
    raise RuntimeError("Selected classifier cannot allow internal patient-level claims for AIROGS-light.")

print("Status: PASS")

In [ ]:
# Cell 13: Evaluate all full-trained AIROGS-light classifiers on the locked test set after validation-only selection

FINAL_CLASSIFIER_DECISION_JSON = RESULTS_07_DIR / "notebook07_validation_selected_final_classifier.json"
FULL_TRAINING_PLAN_CSV = RESULTS_07_DIR / "notebook07_full_training_plan.csv"
FULL_TRAINING_RANKING_CSV = RESULTS_07_DIR / "notebook07_full_training_validation_ranking.csv"

FINAL_TEST_EVAL_DIR = RESULTS_07_DIR / "full_models_locked_test_evaluation"
FINAL_TEST_EVAL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PREDICTION_DIR = FINAL_TEST_EVAL_DIR / "model_predictions"
MODEL_PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

ALL_MODELS_TEST_METRICS_CSV = FINAL_TEST_EVAL_DIR / "all_full_trained_models_locked_test_metrics.csv"
ALL_MODELS_THRESHOLD_TABLE_CSV = FINAL_TEST_EVAL_DIR / "all_full_trained_models_validation_selected_thresholds.csv"
ALL_MODELS_TEST_SUMMARY_CSV = FINAL_TEST_EVAL_DIR / "all_full_trained_models_locked_test_summary.csv"
ALL_MODELS_TEST_SUMMARY_JSON = FINAL_TEST_EVAL_DIR / "all_full_trained_models_locked_test_summary.json"

FINAL_SELECTED_TEST_PREDICTIONS_CSV = FINAL_TEST_EVAL_DIR / "validation_selected_final_classifier_locked_test_image_predictions.csv"
FINAL_SELECTED_VAL_PREDICTIONS_CSV = FINAL_TEST_EVAL_DIR / "validation_selected_final_classifier_validation_image_predictions.csv"

if not FINAL_CLASSIFIER_DECISION_JSON.exists():
    raise FileNotFoundError(f"Missing final classifier decision: {FINAL_CLASSIFIER_DECISION_JSON}")

if not FULL_TRAINING_PLAN_CSV.exists():
    raise FileNotFoundError(f"Missing full-training plan: {FULL_TRAINING_PLAN_CSV}")

if not FULL_TRAINING_RANKING_CSV.exists():
    raise FileNotFoundError(f"Missing full-training validation ranking: {FULL_TRAINING_RANKING_CSV}")

with open(FINAL_CLASSIFIER_DECISION_JSON, "r", encoding="utf-8") as f:
    final_classifier_decision = json.load(f)

full_training_plan = pd.read_csv(FULL_TRAINING_PLAN_CSV)
full_training_ranking = pd.read_csv(FULL_TRAINING_RANKING_CSV)

selected_model_name = str(final_classifier_decision["selected_model_name"])
selected_run_id = str(final_classifier_decision["selected_run_id"])

required_ranking_cols = [
    "run_id",
    "model_name",
    "family",
    "timm_model_name",
    "best_checkpoint",
    "best_epoch_one_indexed",
    "full_training_rank",
    "full_validation_selection_score",
    "validation_decision",
]

missing_ranking_cols = [
    col for col in required_ranking_cols
    if col not in full_training_ranking.columns
]

if missing_ranking_cols:
    raise ValueError(f"Missing full-training ranking columns: {missing_ranking_cols}")

required_plan_cols = [
    "run_id",
    "input_size",
    "batch_size",
    "max_epochs",
    "early_stop_patience",
]

missing_plan_cols = [
    col for col in required_plan_cols
    if col not in full_training_plan.columns
]

if missing_plan_cols:
    raise ValueError(f"Missing full-training plan columns: {missing_plan_cols}")

if "primary_metric_level" in full_training_ranking.columns:
    bad_level_rows = full_training_ranking[
        full_training_ranking["primary_metric_level"].astype(str) != "image_level"
    ]

    if len(bad_level_rows) > 0:
        print(bad_level_rows[["run_id", "model_name", "primary_metric_level"]].to_string(index=False))
        raise RuntimeError("AIROGS-light locked-test evaluation must remain image-level.")

if "internal_patient_level_claim_allowed" in full_training_ranking.columns:
    full_training_ranking["internal_patient_level_claim_allowed"] = as_bool_series(
        full_training_ranking["internal_patient_level_claim_allowed"]
    )

    if full_training_ranking["internal_patient_level_claim_allowed"].any():
        raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light.")

full_training_eval_table = full_training_ranking.merge(
    full_training_plan[
        [
            "run_id",
            "input_size",
            "batch_size",
            "max_epochs",
            "early_stop_patience",
        ]
    ],
    on="run_id",
    how="left",
    validate="one_to_one",
)

full_training_eval_table["is_validation_selected_final_classifier"] = (
    full_training_eval_table["run_id"].astype(str) == selected_run_id
)

if int(full_training_eval_table["is_validation_selected_final_classifier"].sum()) != 1:
    raise RuntimeError("Expected exactly one validation-selected final classifier in evaluation table.")

if "test_df" not in globals() or "val_df" not in globals():
    if "image_split_manifest" not in globals():
        if not Path(IMAGE_SPLIT_MANIFEST_CSV).exists():
            raise FileNotFoundError(f"Missing image split manifest: {IMAGE_SPLIT_MANIFEST_CSV}")

        image_split_manifest = pd.read_csv(IMAGE_SPLIT_MANIFEST_CSV)

    val_df = (
        image_split_manifest[image_split_manifest["glaucoma_split"] == "val"]
        .copy()
        .reset_index(drop=True)
    )

    test_df = (
        image_split_manifest[image_split_manifest["glaucoma_split"] == "test"]
        .copy()
        .reset_index(drop=True)
    )

if len(val_df) != 770:
    raise RuntimeError(f"Expected 770 validation images, found {len(val_df)}")

if len(test_df) != 770:
    raise RuntimeError(f"Expected 770 locked test images, found {len(test_df)}")


def load_trained_classifier_for_inference(model_name, checkpoint_path, input_size):
    model = build_classifier_model(
        model_name=model_name,
        pretrained=False,
        input_size=input_size,
    ).to(DEVICE)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model, checkpoint


def compute_threshold_candidates(y_prob):
    y_prob = np.asarray(y_prob, dtype=float)
    y_prob = y_prob[np.isfinite(y_prob)]

    candidates = np.unique(
        np.concatenate(
            [
                np.linspace(0.01, 0.99, 99),
                np.array([0.5]),
                y_prob,
            ]
        )
    )

    candidates = candidates[
        (candidates >= 0.0)
        & (candidates <= 1.0)
    ]

    return candidates


def select_threshold_from_validation(y_true, y_prob, objective="balanced_accuracy"):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    valid_mask = np.isfinite(y_prob)
    y_true = y_true[valid_mask]
    y_prob = y_prob[valid_mask]

    if len(np.unique(y_true)) != 2:
        raise RuntimeError("Threshold selection requires both classes in validation data.")

    rows = []

    for threshold in compute_threshold_candidates(y_prob):
        y_pred = (y_prob >= threshold).astype(int)

        bal_acc = balanced_accuracy_score(y_true, y_pred)
        acc = accuracy_score(y_true, y_pred)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        brier = brier_score_loss(y_true, y_prob)

        rows.append(
            {
                "threshold": float(threshold),
                "balanced_accuracy": float(bal_acc),
                "accuracy": float(acc),
                "precision": float(precision),
                "sensitivity": float(recall),
                "specificity": float(specificity),
                "f1": float(f1),
                "brier_score": float(brier),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
                "distance_to_0_5": float(abs(threshold - 0.5)),
            }
        )

    threshold_df = pd.DataFrame(rows)

    if objective == "balanced_accuracy":
        threshold_df = threshold_df.sort_values(
            [
                "balanced_accuracy",
                "f1",
                "specificity",
                "sensitivity",
                "distance_to_0_5",
            ],
            ascending=[False, False, False, False, True],
        ).reset_index(drop=True)

    elif objective == "f1":
        threshold_df = threshold_df.sort_values(
            [
                "f1",
                "balanced_accuracy",
                "specificity",
                "sensitivity",
                "distance_to_0_5",
            ],
            ascending=[False, False, False, False, True],
        ).reset_index(drop=True)

    else:
        raise ValueError(f"Unknown threshold objective: {objective}")

    selected = threshold_df.iloc[0].to_dict()

    return selected, threshold_df


def metrics_row_from_predictions(
    model_name,
    run_id,
    family,
    split_name,
    level,
    aggregation_method,
    threshold,
    y_true,
    y_prob,
    is_validation_selected_final_classifier,
    true_patient_level,
):
    metrics = compute_binary_classification_metrics(
        y_true=y_true,
        y_prob=y_prob,
        threshold=threshold,
        prefix="",
    )

    return {
        "model_name": model_name,
        "run_id": run_id,
        "family": family,
        "split": split_name,
        "level": level,
        "aggregation_method": aggregation_method,
        "threshold": float(threshold),
        "is_validation_selected_final_classifier": bool(is_validation_selected_final_classifier),
        "true_patient_level": bool(true_patient_level),
        "internal_patient_level_claim_allowed": False,
        **metrics,
    }


def add_pseudo_patient_predictions_for_all_aggregations(image_predictions):
    all_pseudo_patient_predictions = []

    for method in ["mean", "max", "quality_weighted_mean"]:
        pseudo_patient_predictions = aggregate_predictions_to_patient_level(
            image_predictions=image_predictions,
            method=method,
        )

        pseudo_patient_predictions["level"] = "pseudo_patient"
        pseudo_patient_predictions["true_patient_level"] = False
        pseudo_patient_predictions["internal_patient_level_claim_allowed"] = False

        all_pseudo_patient_predictions.append(pseudo_patient_predictions)

    return pd.concat(all_pseudo_patient_predictions, axis=0, ignore_index=True)


all_metric_rows = []
all_threshold_rows = []
all_summary_rows = []

print("Full-trained AIROGS-light models to evaluate on locked test:")
print(
    full_training_eval_table[
        [
            "full_training_rank",
            "model_name",
            "family",
            "run_id",
            "is_validation_selected_final_classifier",
            "best_checkpoint",
            "input_size",
            "batch_size",
        ]
    ].to_string(index=False)
)

for _, model_row in full_training_eval_table.sort_values("full_training_rank").iterrows():
    model_name = str(model_row["model_name"])
    run_id = str(model_row["run_id"])
    family = str(model_row["family"])
    checkpoint_path = Path(str(model_row["best_checkpoint"]))
    input_size = int(model_row["input_size"])
    batch_size = int(model_row["batch_size"])
    is_selected_final = bool(model_row["is_validation_selected_final_classifier"])

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint for {model_name}: {checkpoint_path}")

    print(f"\nEvaluating {model_name} | run_id={run_id} | selected_final={is_selected_final}")

    model_prediction_dir = MODEL_PREDICTION_DIR / run_id
    model_prediction_dir.mkdir(parents=True, exist_ok=True)

    val_image_predictions_csv = model_prediction_dir / "val_image_predictions.csv"
    test_image_predictions_csv = model_prediction_dir / "test_image_predictions.csv"
    val_pseudo_patient_predictions_csv = model_prediction_dir / "val_pseudo_patient_predictions_all_aggregations.csv"
    test_pseudo_patient_predictions_csv = model_prediction_dir / "test_pseudo_patient_predictions_all_aggregations.csv"
    threshold_grid_csv = model_prediction_dir / "validation_threshold_grid_image.csv"

    classifier_model, checkpoint = load_trained_classifier_for_inference(
        model_name=model_name,
        checkpoint_path=checkpoint_path,
        input_size=input_size,
    )

    val_loader_model = make_classifier_loader(
        dataframe=val_df,
        input_size=input_size,
        batch_size=batch_size,
        split_name="val",
        augment=False,
        balanced_sampler=False,
    )

    test_loader_model = make_classifier_loader(
        dataframe=test_df,
        input_size=input_size,
        batch_size=batch_size,
        split_name="test",
        augment=False,
        balanced_sampler=False,
    )

    val_image_predictions = predict_classifier(
        model=classifier_model,
        loader=val_loader_model,
        device=DEVICE,
        split_name="val",
        model_name=model_name,
    )

    test_image_predictions = predict_classifier(
        model=classifier_model,
        loader=test_loader_model,
        device=DEVICE,
        split_name="test",
        model_name=model_name,
    )

    val_image_predictions["run_id"] = run_id
    test_image_predictions["run_id"] = run_id
    val_image_predictions["family"] = family
    test_image_predictions["family"] = family
    val_image_predictions["is_validation_selected_final_classifier"] = is_selected_final
    test_image_predictions["is_validation_selected_final_classifier"] = is_selected_final
    val_image_predictions["true_patient_level"] = False
    test_image_predictions["true_patient_level"] = False
    val_image_predictions["internal_patient_level_claim_allowed"] = False
    test_image_predictions["internal_patient_level_claim_allowed"] = False

    val_pseudo_patient_predictions_all = add_pseudo_patient_predictions_for_all_aggregations(
        image_predictions=val_image_predictions
    )

    test_pseudo_patient_predictions_all = add_pseudo_patient_predictions_for_all_aggregations(
        image_predictions=test_image_predictions
    )

    val_pseudo_patient_predictions_all["model_name"] = model_name
    val_pseudo_patient_predictions_all["run_id"] = run_id
    val_pseudo_patient_predictions_all["family"] = family
    test_pseudo_patient_predictions_all["model_name"] = model_name
    test_pseudo_patient_predictions_all["run_id"] = run_id
    test_pseudo_patient_predictions_all["family"] = family

    selected_image_threshold, image_threshold_grid = select_threshold_from_validation(
        y_true=val_image_predictions["y_true"].values,
        y_prob=val_image_predictions["y_prob"].values,
        objective="balanced_accuracy",
    )

    image_threshold_grid.to_csv(threshold_grid_csv, index=False)

    all_threshold_rows.append(
        {
            "model_name": model_name,
            "run_id": run_id,
            "family": family,
            "level": "image",
            "aggregation_method": "none",
            "threshold_source": "validation",
            "threshold_objective": "balanced_accuracy",
            "is_validation_selected_final_classifier": is_selected_final,
            "true_patient_level": False,
            "internal_patient_level_claim_allowed": False,
            **selected_image_threshold,
        }
    )

    image_threshold = float(selected_image_threshold["threshold"])

    for split_name, image_predictions in [
        ("val", val_image_predictions),
        ("test", test_image_predictions),
    ]:
        all_metric_rows.append(
            metrics_row_from_predictions(
                model_name=model_name,
                run_id=run_id,
                family=family,
                split_name=split_name,
                level="image",
                aggregation_method="none",
                threshold=image_threshold,
                y_true=image_predictions["y_true"].values,
                y_prob=image_predictions["y_prob"].values,
                is_validation_selected_final_classifier=is_selected_final,
                true_patient_level=False,
            )
        )

    # Pseudo-patient metrics are saved for backward compatibility only.
    # Since AIROGS-light-v2 has no true patient IDs and each pseudo-patient is one image,
    # these are not valid patient-level evidence.
    for split_name, pseudo_predictions_all in [
        ("val", val_pseudo_patient_predictions_all),
        ("test", test_pseudo_patient_predictions_all),
    ]:
        for method in ["mean", "max", "quality_weighted_mean"]:
            pseudo_predictions = pseudo_predictions_all[
                pseudo_predictions_all["aggregation_method"] == method
            ].copy()

            all_metric_rows.append(
                metrics_row_from_predictions(
                    model_name=model_name,
                    run_id=run_id,
                    family=family,
                    split_name=split_name,
                    level="pseudo_patient",
                    aggregation_method=method,
                    threshold=image_threshold,
                    y_true=pseudo_predictions["y_true"].values,
                    y_prob=pseudo_predictions["y_prob"].values,
                    is_validation_selected_final_classifier=is_selected_final,
                    true_patient_level=False,
                )
            )

    val_image_predictions.to_csv(val_image_predictions_csv, index=False)
    test_image_predictions.to_csv(test_image_predictions_csv, index=False)
    val_pseudo_patient_predictions_all.to_csv(val_pseudo_patient_predictions_csv, index=False)
    test_pseudo_patient_predictions_all.to_csv(test_pseudo_patient_predictions_csv, index=False)

    if is_selected_final:
        val_image_predictions.to_csv(FINAL_SELECTED_VAL_PREDICTIONS_CSV, index=False)
        test_image_predictions.to_csv(FINAL_SELECTED_TEST_PREDICTIONS_CSV, index=False)

    image_test = [
        row for row in all_metric_rows
        if row["model_name"] == model_name
        and row["run_id"] == run_id
        and row["split"] == "test"
        and row["level"] == "image"
        and row["aggregation_method"] == "none"
    ][0]

    image_val = [
        row for row in all_metric_rows
        if row["model_name"] == model_name
        and row["run_id"] == run_id
        and row["split"] == "val"
        and row["level"] == "image"
        and row["aggregation_method"] == "none"
    ][0]

    all_summary_rows.append(
        {
            "model_name": model_name,
            "run_id": run_id,
            "family": family,
            "is_validation_selected_final_classifier": is_selected_final,
            "checkpoint": str(checkpoint_path),
            "input_size": input_size,
            "batch_size": batch_size,

            "validation_selected_threshold": image_threshold,

            "val_image_auroc": float(image_val["auroc"]),
            "val_image_auprc": float(image_val["auprc"]),
            "val_image_accuracy": float(image_val["accuracy"]),
            "val_image_balanced_accuracy": float(image_val["balanced_accuracy"]),
            "val_image_sensitivity": float(image_val["recall_sensitivity"]),
            "val_image_specificity": float(image_val["specificity"]),
            "val_image_f1": float(image_val["f1"]),
            "val_image_brier_score": float(image_val["brier_score"]),

            "primary_test_image_threshold": float(image_test["threshold"]),
            "primary_test_image_auroc": float(image_test["auroc"]),
            "primary_test_image_auprc": float(image_test["auprc"]),
            "primary_test_image_accuracy": float(image_test["accuracy"]),
            "primary_test_image_balanced_accuracy": float(image_test["balanced_accuracy"]),
            "primary_test_image_sensitivity": float(image_test["recall_sensitivity"]),
            "primary_test_image_specificity": float(image_test["specificity"]),
            "primary_test_image_f1": float(image_test["f1"]),
            "primary_test_image_brier_score": float(image_test["brier_score"]),

            "true_patient_level_available": False,
            "internal_patient_level_claim_allowed": False,

            "val_image_predictions": str(val_image_predictions_csv),
            "test_image_predictions": str(test_image_predictions_csv),
            "val_pseudo_patient_predictions": str(val_pseudo_patient_predictions_csv),
            "test_pseudo_patient_predictions": str(test_pseudo_patient_predictions_csv),
        }
    )

    del classifier_model, checkpoint

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

all_models_threshold_table = pd.DataFrame(all_threshold_rows)
all_models_test_metrics = pd.DataFrame(all_metric_rows)
all_models_test_summary = pd.DataFrame(all_summary_rows)

all_models_test_summary = all_models_test_summary.sort_values(
    [
        "is_validation_selected_final_classifier",
        "primary_test_image_auroc",
        "primary_test_image_auprc",
        "primary_test_image_balanced_accuracy",
        "primary_test_image_brier_score",
    ],
    ascending=[False, False, False, False, True],
).reset_index(drop=True)

all_models_threshold_table.to_csv(ALL_MODELS_THRESHOLD_TABLE_CSV, index=False)
all_models_test_metrics.to_csv(ALL_MODELS_TEST_METRICS_CSV, index=False)
all_models_test_summary.to_csv(ALL_MODELS_TEST_SUMMARY_CSV, index=False)

selected_summary = all_models_test_summary[
    all_models_test_summary["is_validation_selected_final_classifier"]
]

if len(selected_summary) != 1:
    raise RuntimeError("Expected exactly one validation-selected final classifier in test summary.")

selected_summary = selected_summary.iloc[0]

test_summary_json = {
    "evaluation_stage": "locked_test_evaluation_for_all_full_trained_models",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "split_policy": "official_train_val_test_image_level",
    "selection_policy": (
        "The final classifier was selected using validation only. "
        "All full-trained family-representative models are evaluated on the locked test split only after selection."
    ),
    "validation_selected_model_name": selected_model_name,
    "validation_selected_run_id": selected_run_id,
    "validation_selected_family": final_classifier_decision["selected_family"],
    "validation_selected_checkpoint": final_classifier_decision["selected_best_checkpoint"],

    "primary_metric_level": "image",
    "true_patient_level_available": False,
    "internal_patient_level_claim_allowed": False,
    "pseudo_patient_metrics_policy": (
        "Pseudo-patient metrics are saved only for backward compatibility. "
        "They are not valid patient-level evidence because AIROGS-light-v2 does not provide true patient IDs."
    ),

    "threshold_policy": "Each model uses a threshold selected on validation only by balanced accuracy.",
    "test_set_policy": "The locked test split is used only after validation-only model selection.",

    "validation_selected_primary_test_image_threshold": float(selected_summary["primary_test_image_threshold"]),
    "validation_selected_primary_test_image_auroc": float(selected_summary["primary_test_image_auroc"]),
    "validation_selected_primary_test_image_auprc": float(selected_summary["primary_test_image_auprc"]),
    "validation_selected_primary_test_image_accuracy": float(selected_summary["primary_test_image_accuracy"]),
    "validation_selected_primary_test_image_balanced_accuracy": float(selected_summary["primary_test_image_balanced_accuracy"]),
    "validation_selected_primary_test_image_sensitivity": float(selected_summary["primary_test_image_sensitivity"]),
    "validation_selected_primary_test_image_specificity": float(selected_summary["primary_test_image_specificity"]),
    "validation_selected_primary_test_image_f1": float(selected_summary["primary_test_image_f1"]),
    "validation_selected_primary_test_image_brier_score": float(selected_summary["primary_test_image_brier_score"]),

    "n_models_evaluated": int(len(all_models_test_summary)),
    "n_locked_test_images": int(len(test_df)),
    "n_locked_test_rg": int((test_df["gon_binary"].astype(int) == 1).sum()),
    "n_locked_test_nrg": int((test_df["gon_binary"].astype(int) == 0).sum()),

    "all_models_test_summary_csv": str(ALL_MODELS_TEST_SUMMARY_CSV),
    "all_models_test_metrics_csv": str(ALL_MODELS_TEST_METRICS_CSV),
    "all_models_threshold_table_csv": str(ALL_MODELS_THRESHOLD_TABLE_CSV),
    "final_selected_val_predictions_csv": str(FINAL_SELECTED_VAL_PREDICTIONS_CSV),
    "final_selected_test_predictions_csv": str(FINAL_SELECTED_TEST_PREDICTIONS_CSV),
}

with open(ALL_MODELS_TEST_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(test_summary_json, f, indent=2)

print("\nValidation-selected image thresholds:")
print(
    all_models_threshold_table[
        [
            "model_name",
            "level",
            "aggregation_method",
            "threshold",
            "balanced_accuracy",
            "accuracy",
            "sensitivity",
            "specificity",
            "f1",
            "is_validation_selected_final_classifier",
        ]
    ].to_string(index=False)
)

print("\nAll full-trained AIROGS-light models on locked test:")
print(
    all_models_test_summary[
        [
            "model_name",
            "family",
            "is_validation_selected_final_classifier",
            "primary_test_image_threshold",
            "primary_test_image_auroc",
            "primary_test_image_auprc",
            "primary_test_image_accuracy",
            "primary_test_image_balanced_accuracy",
            "primary_test_image_sensitivity",
            "primary_test_image_specificity",
            "primary_test_image_f1",
            "primary_test_image_brier_score",
        ]
    ].to_string(index=False)
)

print("\nLocked-test summary JSON:")
print(json.dumps(test_summary_json, indent=2))

print(f"\nAll models test summary CSV: {ALL_MODELS_TEST_SUMMARY_CSV}")
print(f"All models test metrics CSV: {ALL_MODELS_TEST_METRICS_CSV}")
print(f"Threshold table CSV: {ALL_MODELS_THRESHOLD_TABLE_CSV}")
print(f"Final selected test predictions: {FINAL_SELECTED_TEST_PREDICTIONS_CSV}")
print(f"Test summary JSON: {ALL_MODELS_TEST_SUMMARY_JSON}")

if len(all_models_test_summary) != int(full_training_eval_table["family"].nunique()):
    raise RuntimeError(
        f"Expected one locked-test result per selected family, found {len(all_models_test_summary)}"
    )

if all_models_test_summary["family"].duplicated().any():
    duplicated_families = all_models_test_summary.loc[
        all_models_test_summary["family"].duplicated(),
        "family",
    ].tolist()

    raise RuntimeError(f"More than one locked-test model found for family: {duplicated_families}")

if int(all_models_test_summary["is_validation_selected_final_classifier"].sum()) != 1:
    raise RuntimeError("Expected exactly one validation-selected final classifier.")

if not Path(str(selected_summary["test_image_predictions"])).exists():
    raise RuntimeError("Selected final classifier test prediction file does not exist.")

if not np.isfinite(float(selected_summary["primary_test_image_auroc"])):
    raise RuntimeError("Selected final classifier test AUROC is not finite.")

if not np.isfinite(float(selected_summary["primary_test_image_auprc"])):
    raise RuntimeError("Selected final classifier test AUPRC is not finite.")

if selected_summary["true_patient_level_available"]:
    raise RuntimeError("AIROGS-light locked-test summary must not claim true patient-level availability.")

if selected_summary["internal_patient_level_claim_allowed"]:
    raise RuntimeError("AIROGS-light locked-test summary must not allow internal patient-level claims.")

print("Status: PASS")

In [ ]:
# Cell 14: Bootstrap confidence intervals and paired locked-test comparison for the AQPR model panel

import time
from pathlib import Path

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    brier_score_loss,
)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

LOCKED_TEST_EVAL_DIR = RESULTS_07_DIR / "full_models_locked_test_evaluation"

ALL_MODELS_TEST_SUMMARY_CSV = LOCKED_TEST_EVAL_DIR / "all_full_trained_models_locked_test_summary.csv"
ALL_MODELS_TEST_METRICS_CSV = LOCKED_TEST_EVAL_DIR / "all_full_trained_models_locked_test_metrics.csv"
ALL_MODELS_THRESHOLD_TABLE_CSV = LOCKED_TEST_EVAL_DIR / "all_full_trained_models_validation_selected_thresholds.csv"

AQPR_PANEL_DIR = LOCKED_TEST_EVAL_DIR / "aqpr_model_panel_bootstrap"
AQPR_PANEL_DIR.mkdir(parents=True, exist_ok=True)

AQPR_PANEL_MANIFEST_CSV = AQPR_PANEL_DIR / "aqpr_model_panel_manifest.csv"
AQPR_PANEL_BOOTSTRAP_CI_CSV = AQPR_PANEL_DIR / "aqpr_model_panel_locked_test_bootstrap_ci.csv"
AQPR_PANEL_PAIRED_COMPARISON_CSV = AQPR_PANEL_DIR / "aqpr_model_panel_paired_comparison_vs_primary.csv"
AQPR_PANEL_POINT_METRICS_CSV = AQPR_PANEL_DIR / "aqpr_model_panel_locked_test_point_metrics.csv"
AQPR_PANEL_BOOTSTRAP_SUMMARY_JSON = AQPR_PANEL_DIR / "aqpr_model_panel_bootstrap_summary.json"

if not ALL_MODELS_TEST_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing locked-test summary: {ALL_MODELS_TEST_SUMMARY_CSV}")

if not ALL_MODELS_TEST_METRICS_CSV.exists():
    raise FileNotFoundError(f"Missing locked-test metrics: {ALL_MODELS_TEST_METRICS_CSV}")

if not ALL_MODELS_THRESHOLD_TABLE_CSV.exists():
    raise FileNotFoundError(f"Missing threshold table: {ALL_MODELS_THRESHOLD_TABLE_CSV}")

all_models_test_summary = pd.read_csv(ALL_MODELS_TEST_SUMMARY_CSV)
all_models_test_metrics = pd.read_csv(ALL_MODELS_TEST_METRICS_CSV)
all_models_threshold_table = pd.read_csv(ALL_MODELS_THRESHOLD_TABLE_CSV)

PRIMARY_MODEL_NAME = "EfficientNetB0"

SECONDARY_AQPR_MODEL_NAMES = [
    "DenseNet121",
    "SwinTiny",
    "ConvNeXtTiny",
]

AQPR_PANEL_MODEL_NAMES = [
    PRIMARY_MODEL_NAME,
    *SECONDARY_AQPR_MODEL_NAMES,
]

AQPR_PANEL_ROLES = {
    PRIMARY_MODEL_NAME: "primary_validation_selected_classifier",
    "DenseNet121": "secondary_aqpr_architecture_panel",
    "SwinTiny": "secondary_aqpr_architecture_panel",
    "ConvNeXtTiny": "secondary_aqpr_architecture_panel",
}

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = int(SEED)
BOOTSTRAP_CI_LOW = 2.5
BOOTSTRAP_CI_HIGH = 97.5

rng = np.random.default_rng(BOOTSTRAP_SEED)

METRIC_NAMES = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1",
    "brier_score",
]

PAIRED_METRIC_NAMES = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "f1",
    "brier_score",
]

LOWER_IS_BETTER_METRICS = {
    "brier_score",
}


def parse_bool_local(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, (int, np.integer)):
        return bool(value)

    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return False
        return bool(int(value))

    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def binary_metric_bundle(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    valid_mask = np.isfinite(y_prob)
    y_true = y_true[valid_mask]
    y_prob = y_prob[valid_mask]

    if len(y_true) == 0:
        return {metric: np.nan for metric in METRIC_NAMES}

    y_pred = (y_prob >= float(threshold)).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    output = {
        "auroc": np.nan,
        "auprc": np.nan,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "sensitivity": float(sensitivity) if np.isfinite(sensitivity) else np.nan,
        "specificity": float(specificity) if np.isfinite(specificity) else np.nan,
        "f1": float(f1),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if len(np.unique(y_true)) == 2:
        output["auroc"] = float(roc_auc_score(y_true, y_prob))
        output["auprc"] = float(average_precision_score(y_true, y_prob))

    return output


def percentile_ci(values, low=2.5, high=97.5):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, 0

    return (
        float(np.percentile(values, low)),
        float(np.percentile(values, high)),
        int(len(values)),
    )


def paired_bootstrap_p_value(delta_values):
    delta_values = np.asarray(delta_values, dtype=float)
    delta_values = delta_values[np.isfinite(delta_values)]

    if len(delta_values) == 0:
        return np.nan

    p_lower_or_equal_zero = np.mean(delta_values <= 0.0)
    p_greater_or_equal_zero = np.mean(delta_values >= 0.0)

    p_two_sided = 2.0 * min(p_lower_or_equal_zero, p_greater_or_equal_zero)

    return float(min(1.0, max(0.0, p_two_sided)))


def get_model_summary_row(model_name):
    rows = all_models_test_summary[
        all_models_test_summary["model_name"].astype(str) == model_name
    ].copy()

    if len(rows) != 1:
        raise RuntimeError(f"Expected exactly one locked-test summary row for {model_name}, found {len(rows)}")

    return rows.iloc[0].to_dict()


def load_model_test_predictions(model_name):
    summary_row = get_model_summary_row(model_name)

    if "test_image_predictions" not in summary_row:
        raise ValueError(f"Missing test_image_predictions path for {model_name}")

    pred_path = Path(str(summary_row["test_image_predictions"]))

    if not pred_path.exists():
        raise FileNotFoundError(f"Missing test predictions for {model_name}: {pred_path}")

    pred_df = pd.read_csv(pred_path)

    required_prediction_cols = [
        "sample_id",
        "y_true",
        "y_prob",
    ]

    missing_prediction_cols = [
        col for col in required_prediction_cols
        if col not in pred_df.columns
    ]

    if missing_prediction_cols:
        raise ValueError(
            f"Missing prediction columns for {model_name}: {missing_prediction_cols}"
        )

    pred_df = pred_df.copy()
    pred_df["sample_id"] = pred_df["sample_id"].astype(str)
    pred_df["y_true"] = pd.to_numeric(pred_df["y_true"], errors="coerce").astype(int)
    pred_df["y_prob"] = pd.to_numeric(pred_df["y_prob"], errors="coerce").astype(float)

    if pred_df["sample_id"].duplicated().any():
        raise RuntimeError(f"Duplicate sample_id values found in predictions for {model_name}")

    return pred_df, summary_row


missing_panel_models = [
    model_name for model_name in AQPR_PANEL_MODEL_NAMES
    if model_name not in set(all_models_test_summary["model_name"].astype(str))
]

if missing_panel_models:
    raise RuntimeError(f"Missing AQPR panel models in locked-test summary: {missing_panel_models}")

panel_prediction_data = {}
panel_manifest_rows = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    pred_df, summary_row = load_model_test_predictions(model_name)

    threshold = float(summary_row["primary_test_image_threshold"])

    panel_prediction_data[model_name] = {
        "predictions": pred_df,
        "threshold": threshold,
        "summary_row": summary_row,
    }

    panel_manifest_rows.append(
        {
            "model_name": model_name,
            "aqpr_panel_role": AQPR_PANEL_ROLES[model_name],
            "family": str(summary_row["family"]),
            "run_id": str(summary_row["run_id"]),
            "threshold_source": "validation_balanced_accuracy",
            "locked_test_threshold": threshold,
            "test_image_predictions": str(summary_row["test_image_predictions"]),
            "is_primary_validation_selected_classifier": model_name == PRIMARY_MODEL_NAME,
            "used_to_redefine_primary_model_after_test": False,
            "true_patient_level_available": False,
            "internal_patient_level_claim_allowed": False,
        }
    )

aqpr_panel_manifest = pd.DataFrame(panel_manifest_rows)
aqpr_panel_manifest.to_csv(AQPR_PANEL_MANIFEST_CSV, index=False)

# Verify that all panel models use exactly the same locked-test samples and labels.
primary_predictions = panel_prediction_data[PRIMARY_MODEL_NAME]["predictions"].copy()
primary_predictions = primary_predictions.sort_values("sample_id").reset_index(drop=True)

primary_sample_ids = primary_predictions["sample_id"].tolist()
primary_y_true = primary_predictions["y_true"].values

for model_name in SECONDARY_AQPR_MODEL_NAMES:
    model_predictions = panel_prediction_data[model_name]["predictions"].copy()
    model_predictions = model_predictions.sort_values("sample_id").reset_index(drop=True)

    if model_predictions["sample_id"].tolist() != primary_sample_ids:
        raise RuntimeError(f"Locked-test sample order mismatch between {PRIMARY_MODEL_NAME} and {model_name}")

    if not np.array_equal(model_predictions["y_true"].values, primary_y_true):
        raise RuntimeError(f"Locked-test labels mismatch between {PRIMARY_MODEL_NAME} and {model_name}")

# Build point metrics.
point_metric_rows = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    pred_df = panel_prediction_data[model_name]["predictions"].copy()
    threshold = panel_prediction_data[model_name]["threshold"]
    summary_row = panel_prediction_data[model_name]["summary_row"]

    metric_bundle = binary_metric_bundle(
        y_true=pred_df["y_true"].values,
        y_prob=pred_df["y_prob"].values,
        threshold=threshold,
    )

    point_metric_rows.append(
        {
            "model_name": model_name,
            "aqpr_panel_role": AQPR_PANEL_ROLES[model_name],
            "family": str(summary_row["family"]),
            "run_id": str(summary_row["run_id"]),
            "split": "locked_test",
            "level": "image",
            "threshold": threshold,
            "n_samples": int(len(pred_df)),
            "n_positive": int((pred_df["y_true"].values == 1).sum()),
            "n_negative": int((pred_df["y_true"].values == 0).sum()),
            "true_patient_level_available": False,
            "internal_patient_level_claim_allowed": False,
            **metric_bundle,
        }
    )

aqpr_panel_point_metrics = pd.DataFrame(point_metric_rows)
aqpr_panel_point_metrics.to_csv(AQPR_PANEL_POINT_METRICS_CSV, index=False)

# Bootstrap CIs for each model.
bootstrap_start_time = time.time()

bootstrap_ci_rows = []
bootstrap_metric_values = {
    model_name: {metric: [] for metric in METRIC_NAMES}
    for model_name in AQPR_PANEL_MODEL_NAMES
}

n_samples = int(len(primary_predictions))
bootstrap_indices = rng.integers(
    low=0,
    high=n_samples,
    size=(N_BOOTSTRAP, n_samples),
)

iterator = range(N_BOOTSTRAP)

if tqdm is not None:
    iterator = tqdm(iterator, desc="Bootstrap CI for AQPR panel", unit="replicate")

for bootstrap_idx in iterator:
    sampled_indices = bootstrap_indices[bootstrap_idx]

    for model_name in AQPR_PANEL_MODEL_NAMES:
        pred_df = panel_prediction_data[model_name]["predictions"].copy()
        pred_df = pred_df.sort_values("sample_id").reset_index(drop=True)

        threshold = panel_prediction_data[model_name]["threshold"]

        sampled_y_true = pred_df["y_true"].values[sampled_indices]
        sampled_y_prob = pred_df["y_prob"].values[sampled_indices]

        metric_bundle = binary_metric_bundle(
            y_true=sampled_y_true,
            y_prob=sampled_y_prob,
            threshold=threshold,
        )

        for metric_name in METRIC_NAMES:
            bootstrap_metric_values[model_name][metric_name].append(metric_bundle[metric_name])

for model_name in AQPR_PANEL_MODEL_NAMES:
    summary_row = panel_prediction_data[model_name]["summary_row"]
    threshold = panel_prediction_data[model_name]["threshold"]

    point_metrics = aqpr_panel_point_metrics[
        aqpr_panel_point_metrics["model_name"] == model_name
    ].iloc[0].to_dict()

    for metric_name in METRIC_NAMES:
        ci_low, ci_high, n_valid_bootstrap = percentile_ci(
            bootstrap_metric_values[model_name][metric_name],
            low=BOOTSTRAP_CI_LOW,
            high=BOOTSTRAP_CI_HIGH,
        )

        bootstrap_ci_rows.append(
            {
                "model_name": model_name,
                "aqpr_panel_role": AQPR_PANEL_ROLES[model_name],
                "family": str(summary_row["family"]),
                "run_id": str(summary_row["run_id"]),
                "split": "locked_test",
                "level": "image",
                "threshold": threshold,
                "metric": metric_name,
                "point_estimate": float(point_metrics[metric_name]),
                "ci_low": ci_low,
                "ci_high": ci_high,
                "n_bootstrap": int(N_BOOTSTRAP),
                "n_valid_bootstrap": int(n_valid_bootstrap),
                "bootstrap_seed": int(BOOTSTRAP_SEED),
                "true_patient_level_available": False,
                "internal_patient_level_claim_allowed": False,
            }
        )

aqpr_panel_bootstrap_ci = pd.DataFrame(bootstrap_ci_rows)
aqpr_panel_bootstrap_ci.to_csv(AQPR_PANEL_BOOTSTRAP_CI_CSV, index=False)

# Paired bootstrap comparisons vs the primary validation-selected model.
paired_comparison_rows = []

primary_threshold = panel_prediction_data[PRIMARY_MODEL_NAME]["threshold"]
primary_predictions_sorted = (
    panel_prediction_data[PRIMARY_MODEL_NAME]["predictions"]
    .copy()
    .sort_values("sample_id")
    .reset_index(drop=True)
)

for secondary_model_name in SECONDARY_AQPR_MODEL_NAMES:
    secondary_threshold = panel_prediction_data[secondary_model_name]["threshold"]

    secondary_predictions_sorted = (
        panel_prediction_data[secondary_model_name]["predictions"]
        .copy()
        .sort_values("sample_id")
        .reset_index(drop=True)
    )

    primary_point = binary_metric_bundle(
        y_true=primary_predictions_sorted["y_true"].values,
        y_prob=primary_predictions_sorted["y_prob"].values,
        threshold=primary_threshold,
    )

    secondary_point = binary_metric_bundle(
        y_true=secondary_predictions_sorted["y_true"].values,
        y_prob=secondary_predictions_sorted["y_prob"].values,
        threshold=secondary_threshold,
    )

    paired_delta_values = {
        metric_name: []
        for metric_name in PAIRED_METRIC_NAMES
    }

    iterator = range(N_BOOTSTRAP)

    if tqdm is not None:
        iterator = tqdm(
            iterator,
            desc=f"Paired bootstrap: {secondary_model_name} vs {PRIMARY_MODEL_NAME}",
            unit="replicate",
        )

    for bootstrap_idx in iterator:
        sampled_indices = bootstrap_indices[bootstrap_idx]

        sampled_y_true = primary_predictions_sorted["y_true"].values[sampled_indices]

        primary_sampled_prob = primary_predictions_sorted["y_prob"].values[sampled_indices]
        secondary_sampled_prob = secondary_predictions_sorted["y_prob"].values[sampled_indices]

        primary_metrics = binary_metric_bundle(
            y_true=sampled_y_true,
            y_prob=primary_sampled_prob,
            threshold=primary_threshold,
        )

        secondary_metrics = binary_metric_bundle(
            y_true=sampled_y_true,
            y_prob=secondary_sampled_prob,
            threshold=secondary_threshold,
        )

        for metric_name in PAIRED_METRIC_NAMES:
            paired_delta_values[metric_name].append(
                secondary_metrics[metric_name] - primary_metrics[metric_name]
            )

    for metric_name in PAIRED_METRIC_NAMES:
        delta_values = np.asarray(paired_delta_values[metric_name], dtype=float)
        delta_values = delta_values[np.isfinite(delta_values)]

        delta_point = float(secondary_point[metric_name] - primary_point[metric_name])
        ci_low, ci_high, n_valid_bootstrap = percentile_ci(
            delta_values,
            low=BOOTSTRAP_CI_LOW,
            high=BOOTSTRAP_CI_HIGH,
        )

        p_two_sided = paired_bootstrap_p_value(delta_values)

        if metric_name in LOWER_IS_BETTER_METRICS:
            favors_secondary = delta_point < 0.0
            interpretation = (
                "secondary_lower_is_better"
                if favors_secondary
                else "primary_lower_is_better_or_tie"
            )
        else:
            favors_secondary = delta_point > 0.0
            interpretation = (
                "secondary_higher_is_better"
                if favors_secondary
                else "primary_higher_is_better_or_tie"
            )

        paired_comparison_rows.append(
            {
                "primary_model_name": PRIMARY_MODEL_NAME,
                "primary_family": str(panel_prediction_data[PRIMARY_MODEL_NAME]["summary_row"]["family"]),
                "secondary_model_name": secondary_model_name,
                "secondary_family": str(panel_prediction_data[secondary_model_name]["summary_row"]["family"]),
                "metric": metric_name,
                "primary_point_estimate": float(primary_point[metric_name]),
                "secondary_point_estimate": float(secondary_point[metric_name]),
                "delta_secondary_minus_primary": delta_point,
                "delta_ci_low": ci_low,
                "delta_ci_high": ci_high,
                "paired_bootstrap_p_two_sided": p_two_sided,
                "n_bootstrap": int(N_BOOTSTRAP),
                "n_valid_bootstrap": int(n_valid_bootstrap),
                "bootstrap_seed": int(BOOTSTRAP_SEED),
                "comparison_level": "image_locked_test",
                "test_used_to_redefine_primary_model": False,
                "favors_secondary_by_point_estimate": bool(favors_secondary),
                "interpretation": interpretation,
                "primary_model_policy": (
                    "EfficientNetB0 remains the validation-selected primary classifier. "
                    "Paired locked-test comparisons are post-selection evidence and do not redefine the primary model."
                ),
            }
        )

aqpr_panel_paired_comparison = pd.DataFrame(paired_comparison_rows)
aqpr_panel_paired_comparison.to_csv(AQPR_PANEL_PAIRED_COMPARISON_CSV, index=False)

elapsed_min = (time.time() - bootstrap_start_time) / 60.0

summary_json = {
    "cell": "Cell 14",
    "analysis": "AQPR model panel locked-test bootstrap CI and paired comparison",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "locked_test_level": "image",
    "true_patient_level_available": False,
    "internal_patient_level_claim_allowed": False,
    "primary_model_name": PRIMARY_MODEL_NAME,
    "primary_model_role": AQPR_PANEL_ROLES[PRIMARY_MODEL_NAME],
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_panel_models": int(len(AQPR_PANEL_MODEL_NAMES)),
    "n_locked_test_samples": int(n_samples),
    "n_locked_test_positive": int((primary_y_true == 1).sum()),
    "n_locked_test_negative": int((primary_y_true == 0).sum()),
    "n_bootstrap": int(N_BOOTSTRAP),
    "bootstrap_seed": int(BOOTSTRAP_SEED),
    "ci_percentiles": [float(BOOTSTRAP_CI_LOW), float(BOOTSTRAP_CI_HIGH)],
    "paired_comparison_reference": PRIMARY_MODEL_NAME,
    "test_set_policy": (
        "The locked test set is used only after validation-only model selection. "
        "DenseNet121, SwinTiny, and ConvNeXtTiny are retained as secondary AQPR models, "
        "not as retrospective replacements of the primary classifier."
    ),
    "primary_classifier_policy": (
        "EfficientNetB0 remains the primary classifier because it was selected using validation only. "
        "The secondary models are used to evaluate model-dependence of AQPR findings."
    ),
    "aqpr_panel_manifest_csv": str(AQPR_PANEL_MANIFEST_CSV),
    "point_metrics_csv": str(AQPR_PANEL_POINT_METRICS_CSV),
    "bootstrap_ci_csv": str(AQPR_PANEL_BOOTSTRAP_CI_CSV),
    "paired_comparison_csv": str(AQPR_PANEL_PAIRED_COMPARISON_CSV),
    "elapsed_min": round(elapsed_min, 3),
}

with open(AQPR_PANEL_BOOTSTRAP_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2)

print("AQPR model panel manifest:")
print(
    aqpr_panel_manifest[
        [
            "model_name",
            "aqpr_panel_role",
            "family",
            "locked_test_threshold",
            "is_primary_validation_selected_classifier",
            "used_to_redefine_primary_model_after_test",
        ]
    ].to_string(index=False)
)

print("\nAQPR panel locked-test point metrics:")
print(
    aqpr_panel_point_metrics[
        [
            "model_name",
            "aqpr_panel_role",
            "family",
            "threshold",
            "auroc",
            "auprc",
            "accuracy",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "f1",
            "brier_score",
        ]
    ].to_string(index=False)
)

print("\nAQPR panel bootstrap confidence intervals:")
print(
    aqpr_panel_bootstrap_ci[
        aqpr_panel_bootstrap_ci["metric"].isin(
            ["auroc", "auprc", "balanced_accuracy", "sensitivity", "specificity", "brier_score"]
        )
    ][
        [
            "model_name",
            "metric",
            "point_estimate",
            "ci_low",
            "ci_high",
            "n_valid_bootstrap",
        ]
    ].to_string(index=False)
)

print("\nPaired locked-test comparisons vs primary validation-selected model:")
print(
    aqpr_panel_paired_comparison[
        aqpr_panel_paired_comparison["metric"].isin(
            ["auroc", "auprc", "balanced_accuracy", "brier_score"]
        )
    ][
        [
            "secondary_model_name",
            "metric",
            "primary_point_estimate",
            "secondary_point_estimate",
            "delta_secondary_minus_primary",
            "delta_ci_low",
            "delta_ci_high",
            "paired_bootstrap_p_two_sided",
            "interpretation",
        ]
    ].to_string(index=False)
)

print("\nBootstrap summary:")
print(json.dumps(summary_json, indent=2))

print(f"\nAQPR panel manifest: {AQPR_PANEL_MANIFEST_CSV}")
print(f"Point metrics: {AQPR_PANEL_POINT_METRICS_CSV}")
print(f"Bootstrap CI: {AQPR_PANEL_BOOTSTRAP_CI_CSV}")
print(f"Paired comparison: {AQPR_PANEL_PAIRED_COMPARISON_CSV}")
print(f"Bootstrap summary JSON: {AQPR_PANEL_BOOTSTRAP_SUMMARY_JSON}")

if len(aqpr_panel_manifest) != 4:
    raise RuntimeError(f"Expected 4 AQPR panel models, found {len(aqpr_panel_manifest)}")

if PRIMARY_MODEL_NAME not in set(aqpr_panel_manifest["model_name"].astype(str)):
    raise RuntimeError("Primary validation-selected model is missing from AQPR panel.")

if int(aqpr_panel_manifest["is_primary_validation_selected_classifier"].sum()) != 1:
    raise RuntimeError("Expected exactly one primary validation-selected classifier.")

if aqpr_panel_manifest["used_to_redefine_primary_model_after_test"].any():
    raise RuntimeError("Locked test must not redefine the primary classifier.")

if len(aqpr_panel_bootstrap_ci) != len(AQPR_PANEL_MODEL_NAMES) * len(METRIC_NAMES):
    raise RuntimeError("Unexpected number of bootstrap CI rows.")

if len(aqpr_panel_paired_comparison) != len(SECONDARY_AQPR_MODEL_NAMES) * len(PAIRED_METRIC_NAMES):
    raise RuntimeError("Unexpected number of paired comparison rows.")

if aqpr_panel_bootstrap_ci["point_estimate"].isna().any():
    raise RuntimeError("NaN point estimates found in bootstrap CI table.")

if aqpr_panel_paired_comparison["delta_secondary_minus_primary"].isna().any():
    raise RuntimeError("NaN paired deltas found.")

print("Status: PASS")

In [ ]:
# Cell 15: AQPR panel calibration and reliability analysis on validation and locked test

import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

LOCKED_TEST_EVAL_DIR = RESULTS_07_DIR / "full_models_locked_test_evaluation"

ALL_MODELS_TEST_SUMMARY_CSV = LOCKED_TEST_EVAL_DIR / "all_full_trained_models_locked_test_summary.csv"

AQPR_PANEL_BOOTSTRAP_DIR = LOCKED_TEST_EVAL_DIR / "aqpr_model_panel_bootstrap"
AQPR_PANEL_MANIFEST_CSV = AQPR_PANEL_BOOTSTRAP_DIR / "aqpr_model_panel_manifest.csv"

CALIBRATION_DIR = LOCKED_TEST_EVAL_DIR / "aqpr_model_panel_calibration_analysis"
CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)

CALIBRATION_SUMMARY_CSV = CALIBRATION_DIR / "aqpr_panel_calibration_summary.csv"
CALIBRATION_BINS_CSV = CALIBRATION_DIR / "aqpr_panel_calibration_bins.csv"
TEMPERATURE_SCALING_CSV = CALIBRATION_DIR / "aqpr_panel_temperature_scaling_parameters.csv"
CALIBRATION_SUMMARY_JSON = CALIBRATION_DIR / "aqpr_panel_calibration_summary.json"

LOCKED_TEST_RAW_RELIABILITY_PNG = CALIBRATION_DIR / "aqpr_panel_locked_test_raw_reliability.png"
LOCKED_TEST_RAW_RELIABILITY_PDF = CALIBRATION_DIR / "aqpr_panel_locked_test_raw_reliability.pdf"

LOCKED_TEST_TEMP_RELIABILITY_PNG = CALIBRATION_DIR / "aqpr_panel_locked_test_temperature_scaled_reliability.png"
LOCKED_TEST_TEMP_RELIABILITY_PDF = CALIBRATION_DIR / "aqpr_panel_locked_test_temperature_scaled_reliability.pdf"

VALIDATION_RAW_RELIABILITY_PNG = CALIBRATION_DIR / "aqpr_panel_validation_raw_reliability.png"
VALIDATION_RAW_RELIABILITY_PDF = CALIBRATION_DIR / "aqpr_panel_validation_raw_reliability.pdf"

VALIDATION_TEMP_RELIABILITY_PNG = CALIBRATION_DIR / "aqpr_panel_validation_temperature_scaled_reliability.png"
VALIDATION_TEMP_RELIABILITY_PDF = CALIBRATION_DIR / "aqpr_panel_validation_temperature_scaled_reliability.pdf"

CALIBRATION_COMPARISON_PNG = CALIBRATION_DIR / "aqpr_panel_locked_test_brier_ece_comparison.png"
CALIBRATION_COMPARISON_PDF = CALIBRATION_DIR / "aqpr_panel_locked_test_brier_ece_comparison.pdf"

if not ALL_MODELS_TEST_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing locked-test model summary: {ALL_MODELS_TEST_SUMMARY_CSV}")

if not AQPR_PANEL_MANIFEST_CSV.exists():
    raise FileNotFoundError(
        f"Missing AQPR panel manifest from Cell 14: {AQPR_PANEL_MANIFEST_CSV}"
    )

all_models_test_summary = pd.read_csv(ALL_MODELS_TEST_SUMMARY_CSV)
aqpr_panel_manifest = pd.read_csv(AQPR_PANEL_MANIFEST_CSV)

PRIMARY_MODEL_NAME = "EfficientNetB0"

SECONDARY_AQPR_MODEL_NAMES = [
    "DenseNet121",
    "SwinTiny",
    "ConvNeXtTiny",
]

AQPR_PANEL_MODEL_NAMES = [
    PRIMARY_MODEL_NAME,
    *SECONDARY_AQPR_MODEL_NAMES,
]

EXPECTED_PANEL_ROLES = {
    "EfficientNetB0": "primary_validation_selected_classifier",
    "DenseNet121": "secondary_aqpr_architecture_panel",
    "SwinTiny": "secondary_aqpr_architecture_panel",
    "ConvNeXtTiny": "secondary_aqpr_architecture_panel",
}

N_BINS_LIST = [10, 15]
PRIMARY_N_BINS = 10

TEMPERATURE_GRID = np.concatenate(
    [
        np.linspace(0.5, 5.0, 181),
        np.linspace(5.25, 10.0, 20),
    ]
)


def sigmoid_np(x):
    x = np.asarray(x, dtype=float)
    return 1.0 / (1.0 + np.exp(-x))


def logit_from_probability(prob):
    prob = np.asarray(prob, dtype=float)
    prob = np.clip(prob, 1e-7, 1.0 - 1e-7)
    return np.log(prob / (1.0 - prob))


def parse_bool_local(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, (int, np.integer)):
        return bool(value)

    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return False
        return bool(int(value))

    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def load_prediction_table(path_value, model_name, split_name):
    path = Path(str(path_value))

    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} predictions for {model_name}: {path}")

    df = pd.read_csv(path)

    required_cols = [
        "sample_id",
        "y_true",
        "y_prob",
    ]

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing required columns in {split_name} predictions for {model_name}: {missing_cols}"
        )

    df = df.copy()
    df["sample_id"] = df["sample_id"].astype(str)
    df["y_true"] = pd.to_numeric(df["y_true"], errors="coerce").astype(int)
    df["y_prob"] = pd.to_numeric(df["y_prob"], errors="coerce").astype(float)

    if "logit" in df.columns:
        df["logit"] = pd.to_numeric(df["logit"], errors="coerce").astype(float)
    else:
        df["logit"] = logit_from_probability(df["y_prob"].values)

    if df["sample_id"].duplicated().any():
        raise RuntimeError(f"Duplicate sample_id values in {split_name} predictions for {model_name}")

    if not np.isfinite(df["y_prob"]).all():
        raise RuntimeError(f"Non-finite probabilities in {split_name} predictions for {model_name}")

    if not np.isfinite(df["logit"]).all():
        raise RuntimeError(f"Non-finite logits in {split_name} predictions for {model_name}")

    return df


def compute_calibration_bins(y_true, y_prob, n_bins):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    valid_mask = np.isfinite(y_prob)
    y_true = y_true[valid_mask]
    y_prob = y_prob[valid_mask]

    bin_edges = np.linspace(0.0, 1.0, int(n_bins) + 1)

    rows = []

    for bin_idx in range(int(n_bins)):
        left = float(bin_edges[bin_idx])
        right = float(bin_edges[bin_idx + 1])

        if bin_idx == int(n_bins) - 1:
            mask = (y_prob >= left) & (y_prob <= right)
        else:
            mask = (y_prob >= left) & (y_prob < right)

        n_bin = int(mask.sum())

        if n_bin == 0:
            probability_mean = np.nan
            event_rate = np.nan
            absolute_gap = np.nan
        else:
            probability_mean = float(np.mean(y_prob[mask]))
            event_rate = float(np.mean(y_true[mask]))
            absolute_gap = float(abs(event_rate - probability_mean))

        rows.append(
            {
                "n_bins": int(n_bins),
                "bin_index": int(bin_idx),
                "bin_left": left,
                "bin_right": right,
                "n_bin": n_bin,
                "probability_mean": probability_mean,
                "event_rate": event_rate,
                "absolute_gap": absolute_gap,
            }
        )

    bins_df = pd.DataFrame(rows)

    n_total = int(len(y_true))
    non_empty = bins_df[bins_df["n_bin"] > 0].copy()

    if n_total == 0 or len(non_empty) == 0:
        ece = np.nan
        mce = np.nan
    else:
        ece = float(
            np.sum(
                (non_empty["n_bin"].values / n_total)
                * non_empty["absolute_gap"].values
            )
        )
        mce = float(non_empty["absolute_gap"].max())

    return bins_df, ece, mce


def compute_calibration_summary(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    valid_mask = np.isfinite(y_prob)
    y_true = y_true[valid_mask]
    y_prob = y_prob[valid_mask]

    y_prob_clipped = np.clip(y_prob, 1e-7, 1.0 - 1e-7)

    bins_10, ece_10, mce_10 = compute_calibration_bins(
        y_true=y_true,
        y_prob=y_prob,
        n_bins=10,
    )

    bins_15, ece_15, mce_15 = compute_calibration_bins(
        y_true=y_true,
        y_prob=y_prob,
        n_bins=15,
    )

    if len(np.unique(y_true)) == 2:
        auroc = float(roc_auc_score(y_true, y_prob))
        auprc = float(average_precision_score(y_true, y_prob))
    else:
        auroc = np.nan
        auprc = np.nan

    return {
        "n_samples": int(len(y_true)),
        "n_positive": int((y_true == 1).sum()),
        "n_negative": int((y_true == 0).sum()),
        "observed_event_rate": float(np.mean(y_true)),
        "mean_predicted_probability": float(np.mean(y_prob)),
        "mean_calibration_error_signed": float(np.mean(y_prob) - np.mean(y_true)),
        "auroc": auroc,
        "auprc": auprc,
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "negative_log_likelihood": float(log_loss(y_true, y_prob_clipped, labels=[0, 1])),
        "ece_10": float(ece_10),
        "mce_10": float(mce_10),
        "ece_15": float(ece_15),
        "mce_15": float(mce_15),
    }


def fit_temperature_on_validation(y_true, logits):
    y_true = np.asarray(y_true).astype(int)
    logits = np.asarray(logits).astype(float)

    valid_mask = np.isfinite(logits)
    y_true = y_true[valid_mask]
    logits = logits[valid_mask]

    if len(y_true) == 0:
        raise RuntimeError("No valid validation samples for temperature scaling.")

    rows = []

    for temperature in TEMPERATURE_GRID:
        scaled_prob = sigmoid_np(logits / float(temperature))
        scaled_prob = np.clip(scaled_prob, 1e-7, 1.0 - 1e-7)

        nll = log_loss(y_true, scaled_prob, labels=[0, 1])
        brier = brier_score_loss(y_true, scaled_prob)

        bins_10, ece_10, mce_10 = compute_calibration_bins(
            y_true=y_true,
            y_prob=scaled_prob,
            n_bins=10,
        )

        rows.append(
            {
                "temperature": float(temperature),
                "validation_negative_log_likelihood": float(nll),
                "validation_brier_score": float(brier),
                "validation_ece_10": float(ece_10),
                "validation_mce_10": float(mce_10),
            }
        )

    temp_df = pd.DataFrame(rows)

    temp_df = temp_df.sort_values(
        [
            "validation_negative_log_likelihood",
            "validation_brier_score",
            "validation_ece_10",
            "temperature",
        ],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)

    selected = temp_df.iloc[0].to_dict()

    return selected, temp_df


def apply_temperature_scaling(logits, temperature):
    logits = np.asarray(logits).astype(float)
    return sigmoid_np(logits / float(temperature))


def save_reliability_plot(
    calibration_bins,
    split_name,
    probability_variant,
    output_png,
    output_pdf,
    title,
):
    plot_df = calibration_bins[
        (calibration_bins["split"].astype(str) == split_name)
        & (calibration_bins["probability_variant"].astype(str) == probability_variant)
        & (calibration_bins["n_bins"].astype(int) == PRIMARY_N_BINS)
    ].copy()

    if len(plot_df) == 0:
        raise RuntimeError(
            f"No calibration bins for split={split_name}, variant={probability_variant}"
        )

    fig, ax = plt.subplots(figsize=(7.5, 6.0))

    ax.plot([0.0, 1.0], [0.0, 1.0], linestyle="--", linewidth=1.0, label="Perfect calibration")

    for model_name in AQPR_PANEL_MODEL_NAMES:
        model_bins = plot_df[
            (plot_df["model_name"].astype(str) == model_name)
            & (plot_df["n_bin"] > 0)
        ].copy()

        model_bins = model_bins.sort_values("probability_mean")

        ax.plot(
            model_bins["probability_mean"],
            model_bins["event_rate"],
            marker="o",
            linewidth=1.5,
            label=model_name,
        )

    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed event rate")
    ax.set_title(title)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()

    fig.savefig(output_png, dpi=300, bbox_inches="tight")
    fig.savefig(output_pdf, bbox_inches="tight")

    display(fig)
    plt.close(fig)


def save_calibration_metric_comparison_plot(summary_df, output_png, output_pdf):
    plot_df = summary_df[
        (summary_df["split"].astype(str) == "test")
        & (summary_df["probability_variant"].astype(str).isin(["raw", "temperature_scaled"]))
    ].copy()

    plot_df["model_variant"] = (
        plot_df["model_name"].astype(str)
        + " | "
        + plot_df["probability_variant"].astype(str)
    )

    plot_df = plot_df.sort_values(
        [
            "model_name",
            "probability_variant",
        ]
    ).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(10.5, 5.5))

    x = np.arange(len(plot_df))
    width = 0.38

    ax.bar(
        x - width / 2,
        plot_df["brier_score"].values,
        width,
        label="Brier score",
    )

    ax.bar(
        x + width / 2,
        plot_df["ece_10"].values,
        width,
        label="ECE-10",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["model_variant"].tolist(), rotation=45, ha="right")
    ax.set_ylabel("Calibration error")
    ax.set_title("Locked-test calibration comparison for AQPR panel")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()

    fig.savefig(output_png, dpi=300, bbox_inches="tight")
    fig.savefig(output_pdf, bbox_inches="tight")

    display(fig)
    plt.close(fig)


missing_panel_models = [
    model_name
    for model_name in AQPR_PANEL_MODEL_NAMES
    if model_name not in set(aqpr_panel_manifest["model_name"].astype(str))
]

if missing_panel_models:
    raise RuntimeError(f"Missing AQPR panel models in manifest: {missing_panel_models}")

if int((aqpr_panel_manifest["model_name"].astype(str) == PRIMARY_MODEL_NAME).sum()) != 1:
    raise RuntimeError("Expected exactly one EfficientNetB0 row in AQPR panel manifest.")

primary_row = aqpr_panel_manifest[
    aqpr_panel_manifest["model_name"].astype(str) == PRIMARY_MODEL_NAME
].iloc[0]

if not parse_bool_local(primary_row["is_primary_validation_selected_classifier"]):
    raise RuntimeError("EfficientNetB0 must remain the primary validation-selected classifier.")

if parse_bool_local(primary_row["used_to_redefine_primary_model_after_test"]):
    raise RuntimeError("The locked test must not redefine the primary classifier.")

prediction_paths = {}

for model_name in AQPR_PANEL_MODEL_NAMES:
    summary_rows = all_models_test_summary[
        all_models_test_summary["model_name"].astype(str) == model_name
    ].copy()

    if len(summary_rows) != 1:
        raise RuntimeError(f"Expected exactly one locked-test summary row for {model_name}")

    row = summary_rows.iloc[0].to_dict()

    for required_path_col in [
        "val_image_predictions",
        "test_image_predictions",
    ]:
        if required_path_col not in row:
            raise ValueError(f"Missing {required_path_col} in locked-test summary for {model_name}")

    prediction_paths[model_name] = {
        "family": str(row["family"]),
        "run_id": str(row["run_id"]),
        "val_image_predictions": str(row["val_image_predictions"]),
        "test_image_predictions": str(row["test_image_predictions"]),
        "is_primary": model_name == PRIMARY_MODEL_NAME,
        "aqpr_panel_role": EXPECTED_PANEL_ROLES[model_name],
    }

calibration_summary_rows = []
calibration_bin_rows = []
temperature_rows = []
temperature_grid_rows = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    model_info = prediction_paths[model_name]

    val_predictions = load_prediction_table(
        path_value=model_info["val_image_predictions"],
        model_name=model_name,
        split_name="val",
    )

    test_predictions = load_prediction_table(
        path_value=model_info["test_image_predictions"],
        model_name=model_name,
        split_name="test",
    )

    selected_temperature, temperature_grid_df = fit_temperature_on_validation(
        y_true=val_predictions["y_true"].values,
        logits=val_predictions["logit"].values,
    )

    selected_temperature_value = float(selected_temperature["temperature"])

    temperature_rows.append(
        {
            "model_name": model_name,
            "aqpr_panel_role": model_info["aqpr_panel_role"],
            "family": model_info["family"],
            "run_id": model_info["run_id"],
            "temperature": selected_temperature_value,
            "temperature_fit_split": "validation",
            "temperature_selection_objective": "minimum_validation_negative_log_likelihood",
            "validation_negative_log_likelihood": float(
                selected_temperature["validation_negative_log_likelihood"]
            ),
            "validation_brier_score": float(selected_temperature["validation_brier_score"]),
            "validation_ece_10": float(selected_temperature["validation_ece_10"]),
            "validation_mce_10": float(selected_temperature["validation_mce_10"]),
            "test_used_to_fit_temperature": False,
            "is_primary_validation_selected_classifier": bool(model_info["is_primary"]),
        }
    )

    temperature_grid_df["model_name"] = model_name
    temperature_grid_df["family"] = model_info["family"]
    temperature_grid_df["run_id"] = model_info["run_id"]
    temperature_grid_rows.append(temperature_grid_df)

    split_tables = {
        "val": val_predictions,
        "test": test_predictions,
    }

    for split_name, pred_df in split_tables.items():
        probability_variants = {
            "raw": pred_df["y_prob"].values,
            "temperature_scaled": apply_temperature_scaling(
                logits=pred_df["logit"].values,
                temperature=selected_temperature_value,
            ),
        }

        for probability_variant, probabilities in probability_variants.items():
            summary = compute_calibration_summary(
                y_true=pred_df["y_true"].values,
                y_prob=probabilities,
            )

            calibration_summary_rows.append(
                {
                    "model_name": model_name,
                    "aqpr_panel_role": model_info["aqpr_panel_role"],
                    "family": model_info["family"],
                    "run_id": model_info["run_id"],
                    "split": split_name,
                    "level": "image",
                    "probability_variant": probability_variant,
                    "temperature": (
                        selected_temperature_value
                        if probability_variant == "temperature_scaled"
                        else 1.0
                    ),
                    "temperature_fit_split": (
                        "validation"
                        if probability_variant == "temperature_scaled"
                        else "none"
                    ),
                    "test_used_to_fit_temperature": False,
                    "true_patient_level_available": False,
                    "internal_patient_level_claim_allowed": False,
                    "is_primary_validation_selected_classifier": bool(model_info["is_primary"]),
                    **summary,
                }
            )

            for n_bins in N_BINS_LIST:
                bins_df, ece, mce = compute_calibration_bins(
                    y_true=pred_df["y_true"].values,
                    y_prob=probabilities,
                    n_bins=n_bins,
                )

                bins_df["model_name"] = model_name
                bins_df["aqpr_panel_role"] = model_info["aqpr_panel_role"]
                bins_df["family"] = model_info["family"]
                bins_df["run_id"] = model_info["run_id"]
                bins_df["split"] = split_name
                bins_df["level"] = "image"
                bins_df["probability_variant"] = probability_variant
                bins_df["temperature"] = (
                    selected_temperature_value
                    if probability_variant == "temperature_scaled"
                    else 1.0
                )
                bins_df["ece"] = float(ece)
                bins_df["mce"] = float(mce)
                bins_df["true_patient_level_available"] = False
                bins_df["internal_patient_level_claim_allowed"] = False
                bins_df["is_primary_validation_selected_classifier"] = bool(model_info["is_primary"])

                calibration_bin_rows.append(bins_df)

calibration_summary = pd.DataFrame(calibration_summary_rows)
calibration_bins = pd.concat(calibration_bin_rows, axis=0, ignore_index=True)
temperature_scaling = pd.DataFrame(temperature_rows)
temperature_grid_all = pd.concat(temperature_grid_rows, axis=0, ignore_index=True)

temperature_grid_csv = CALIBRATION_DIR / "aqpr_panel_temperature_grid_search.csv"
temperature_grid_all.to_csv(temperature_grid_csv, index=False)

calibration_summary.to_csv(CALIBRATION_SUMMARY_CSV, index=False)
calibration_bins.to_csv(CALIBRATION_BINS_CSV, index=False)
temperature_scaling.to_csv(TEMPERATURE_SCALING_CSV, index=False)

save_reliability_plot(
    calibration_bins=calibration_bins,
    split_name="test",
    probability_variant="raw",
    output_png=LOCKED_TEST_RAW_RELIABILITY_PNG,
    output_pdf=LOCKED_TEST_RAW_RELIABILITY_PDF,
    title="AQPR panel reliability on locked AIROGS-light test set: raw probabilities",
)

save_reliability_plot(
    calibration_bins=calibration_bins,
    split_name="test",
    probability_variant="temperature_scaled",
    output_png=LOCKED_TEST_TEMP_RELIABILITY_PNG,
    output_pdf=LOCKED_TEST_TEMP_RELIABILITY_PDF,
    title="AQPR panel reliability on locked AIROGS-light test set: temperature scaled",
)

save_reliability_plot(
    calibration_bins=calibration_bins,
    split_name="val",
    probability_variant="raw",
    output_png=VALIDATION_RAW_RELIABILITY_PNG,
    output_pdf=VALIDATION_RAW_RELIABILITY_PDF,
    title="AQPR panel reliability on AIROGS-light validation set: raw probabilities",
)

save_reliability_plot(
    calibration_bins=calibration_bins,
    split_name="val",
    probability_variant="temperature_scaled",
    output_png=VALIDATION_TEMP_RELIABILITY_PNG,
    output_pdf=VALIDATION_TEMP_RELIABILITY_PDF,
    title="AQPR panel reliability on AIROGS-light validation set: temperature scaled",
)

save_calibration_metric_comparison_plot(
    summary_df=calibration_summary,
    output_png=CALIBRATION_COMPARISON_PNG,
    output_pdf=CALIBRATION_COMPARISON_PDF,
)

primary_test_raw = calibration_summary[
    (calibration_summary["model_name"].astype(str) == PRIMARY_MODEL_NAME)
    & (calibration_summary["split"].astype(str) == "test")
    & (calibration_summary["probability_variant"].astype(str) == "raw")
].iloc[0].to_dict()

primary_test_temp = calibration_summary[
    (calibration_summary["model_name"].astype(str) == PRIMARY_MODEL_NAME)
    & (calibration_summary["split"].astype(str) == "test")
    & (calibration_summary["probability_variant"].astype(str) == "temperature_scaled")
].iloc[0].to_dict()

locked_test_raw_summary = calibration_summary[
    (calibration_summary["split"].astype(str) == "test")
    & (calibration_summary["probability_variant"].astype(str) == "raw")
].copy()

locked_test_temp_summary = calibration_summary[
    (calibration_summary["split"].astype(str) == "test")
    & (calibration_summary["probability_variant"].astype(str) == "temperature_scaled")
].copy()

best_raw_brier_model = locked_test_raw_summary.sort_values("brier_score").iloc[0].to_dict()
best_raw_ece_model = locked_test_raw_summary.sort_values("ece_10").iloc[0].to_dict()
best_temp_brier_model = locked_test_temp_summary.sort_values("brier_score").iloc[0].to_dict()
best_temp_ece_model = locked_test_temp_summary.sort_values("ece_10").iloc[0].to_dict()

calibration_summary_json = {
    "cell": "Cell 15",
    "analysis": "AQPR model panel image-level calibration and reliability analysis",
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "calibration_level": "image",
    "true_patient_level_available": False,
    "internal_patient_level_claim_allowed": False,
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "n_panel_models": int(len(AQPR_PANEL_MODEL_NAMES)),
    "probability_variants": ["raw", "temperature_scaled"],
    "temperature_scaling_policy": (
        "A scalar temperature is fitted separately for each model using validation logits only, "
        "minimizing validation negative log-likelihood. The locked test set is used only for evaluation."
    ),
    "test_set_policy": (
        "The locked AIROGS-light test set is not used to select the primary classifier or fit calibration parameters."
    ),
    "primary_classifier_policy": (
        "EfficientNetB0 remains the primary validation-selected classifier. "
        "DenseNet121, SwinTiny, and ConvNeXtTiny remain secondary AQPR models."
    ),
    "primary_test_raw_brier_score": float(primary_test_raw["brier_score"]),
    "primary_test_raw_ece_10": float(primary_test_raw["ece_10"]),
    "primary_test_raw_negative_log_likelihood": float(primary_test_raw["negative_log_likelihood"]),
    "primary_test_temperature_scaled_brier_score": float(primary_test_temp["brier_score"]),
    "primary_test_temperature_scaled_ece_10": float(primary_test_temp["ece_10"]),
    "primary_test_temperature_scaled_negative_log_likelihood": float(
        primary_test_temp["negative_log_likelihood"]
    ),
    "best_locked_test_raw_brier_model": str(best_raw_brier_model["model_name"]),
    "best_locked_test_raw_ece_10_model": str(best_raw_ece_model["model_name"]),
    "best_locked_test_temperature_scaled_brier_model": str(best_temp_brier_model["model_name"]),
    "best_locked_test_temperature_scaled_ece_10_model": str(best_temp_ece_model["model_name"]),
    "n_bins_reported": N_BINS_LIST,
    "primary_n_bins_for_figures": int(PRIMARY_N_BINS),
    "calibration_summary_csv": str(CALIBRATION_SUMMARY_CSV),
    "calibration_bins_csv": str(CALIBRATION_BINS_CSV),
    "temperature_scaling_csv": str(TEMPERATURE_SCALING_CSV),
    "temperature_grid_csv": str(temperature_grid_csv),
    "locked_test_raw_reliability_png": str(LOCKED_TEST_RAW_RELIABILITY_PNG),
    "locked_test_temperature_scaled_reliability_png": str(LOCKED_TEST_TEMP_RELIABILITY_PNG),
    "validation_raw_reliability_png": str(VALIDATION_RAW_RELIABILITY_PNG),
    "validation_temperature_scaled_reliability_png": str(VALIDATION_TEMP_RELIABILITY_PNG),
    "calibration_comparison_png": str(CALIBRATION_COMPARISON_PNG),
}

with open(CALIBRATION_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(calibration_summary_json, f, indent=2)

print("AQPR panel temperature scaling parameters:")
print(
    temperature_scaling[
        [
            "model_name",
            "aqpr_panel_role",
            "family",
            "temperature",
            "validation_negative_log_likelihood",
            "validation_brier_score",
            "validation_ece_10",
            "test_used_to_fit_temperature",
        ]
    ].to_string(index=False)
)

print("\nAQPR panel calibration summary:")
print(
    calibration_summary[
        [
            "model_name",
            "aqpr_panel_role",
            "split",
            "probability_variant",
            "n_samples",
            "observed_event_rate",
            "mean_predicted_probability",
            "mean_calibration_error_signed",
            "brier_score",
            "negative_log_likelihood",
            "ece_10",
            "mce_10",
            "ece_15",
            "mce_15",
        ]
    ].to_string(index=False)
)

print("\nLocked-test raw calibration ranking by Brier score:")
print(
    locked_test_raw_summary[
        [
            "model_name",
            "aqpr_panel_role",
            "brier_score",
            "negative_log_likelihood",
            "ece_10",
            "mce_10",
            "mean_calibration_error_signed",
        ]
    ]
    .sort_values("brier_score")
    .to_string(index=False)
)

print("\nLocked-test temperature-scaled calibration ranking by Brier score:")
print(
    locked_test_temp_summary[
        [
            "model_name",
            "aqpr_panel_role",
            "brier_score",
            "negative_log_likelihood",
            "ece_10",
            "mce_10",
            "mean_calibration_error_signed",
        ]
    ]
    .sort_values("brier_score")
    .to_string(index=False)
)

print("\nCalibration summary JSON:")
print(json.dumps(calibration_summary_json, indent=2))

print(f"\nCalibration summary CSV: {CALIBRATION_SUMMARY_CSV}")
print(f"Calibration bins CSV: {CALIBRATION_BINS_CSV}")
print(f"Temperature scaling CSV: {TEMPERATURE_SCALING_CSV}")
print(f"Temperature grid CSV: {temperature_grid_csv}")
print(f"Calibration summary JSON: {CALIBRATION_SUMMARY_JSON}")

expected_summary_rows = len(AQPR_PANEL_MODEL_NAMES) * 2 * 2

if len(calibration_summary) != expected_summary_rows:
    raise RuntimeError(
        f"Expected {expected_summary_rows} calibration summary rows, found {len(calibration_summary)}"
    )

expected_bin_rows = len(AQPR_PANEL_MODEL_NAMES) * 2 * 2 * sum(N_BINS_LIST)

if len(calibration_bins) != expected_bin_rows:
    raise RuntimeError(
        f"Expected {expected_bin_rows} calibration bin rows, found {len(calibration_bins)}"
    )

if len(temperature_scaling) != len(AQPR_PANEL_MODEL_NAMES):
    raise RuntimeError(
        f"Expected one temperature row per AQPR panel model, found {len(temperature_scaling)}"
    )

if not np.isfinite(temperature_scaling["temperature"]).all():
    raise RuntimeError("Non-finite temperature values found.")

if (temperature_scaling["temperature"] <= 0).any():
    raise RuntimeError("Temperature values must be positive.")

for output_path in [
    LOCKED_TEST_RAW_RELIABILITY_PNG,
    LOCKED_TEST_RAW_RELIABILITY_PDF,
    LOCKED_TEST_TEMP_RELIABILITY_PNG,
    LOCKED_TEST_TEMP_RELIABILITY_PDF,
    VALIDATION_RAW_RELIABILITY_PNG,
    VALIDATION_RAW_RELIABILITY_PDF,
    VALIDATION_TEMP_RELIABILITY_PNG,
    VALIDATION_TEMP_RELIABILITY_PDF,
    CALIBRATION_COMPARISON_PNG,
    CALIBRATION_COMPARISON_PDF,
]:
    if not Path(output_path).exists():
        raise RuntimeError(f"Expected calibration figure was not saved: {output_path}")

if calibration_summary["internal_patient_level_claim_allowed"].any():
    raise RuntimeError("Internal patient-level claims must remain disabled for AIROGS-light calibration.")

if calibration_summary["true_patient_level_available"].any():
    raise RuntimeError("AIROGS-light calibration must remain image-level.")

if parse_bool_local(primary_row["used_to_redefine_primary_model_after_test"]):
    raise RuntimeError("The locked test must not redefine the primary classifier.")

print("Status: PASS")

In [ ]:
# Cell 16: Freeze AQPR model panel and create Notebook 08 handoff manifest

import json
import time
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# Keep False for fast workflow continuation.
# Set True only for the final reproducibility release.
COMPUTE_CHECKPOINT_SHA256 = False

PRIMARY_MODEL_NAME = "EfficientNetB0"
SECONDARY_AQPR_MODEL_NAMES = ["DenseNet121", "SwinTiny", "ConvNeXtTiny"]
AQPR_PANEL_MODEL_NAMES = [PRIMARY_MODEL_NAME] + SECONDARY_AQPR_MODEL_NAMES

AQPR_PANEL_ROLES = {
    "EfficientNetB0": "primary_validation_selected_classifier",
    "DenseNet121": "secondary_aqpr_architecture_panel",
    "SwinTiny": "secondary_aqpr_architecture_panel",
    "ConvNeXtTiny": "secondary_aqpr_architecture_panel",
}

LOCKED_TEST_EVAL_DIR = RESULTS_07_DIR / "full_models_locked_test_evaluation"
BOOTSTRAP_DIR = LOCKED_TEST_EVAL_DIR / "aqpr_model_panel_bootstrap"
CALIBRATION_DIR = LOCKED_TEST_EVAL_DIR / "aqpr_model_panel_calibration_analysis"

FROZEN_PANEL_DIR = CLASSIFIER_MODEL_ROOT / "frozen_aqpr_model_panel_notebook07"
FROZEN_PANEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DECISION_JSON = RESULTS_07_DIR / "notebook07_validation_selected_final_classifier.json"
TEST_SUMMARY_CSV = LOCKED_TEST_EVAL_DIR / "all_full_trained_models_locked_test_summary.csv"

AQPR_PANEL_MANIFEST_CSV = BOOTSTRAP_DIR / "aqpr_model_panel_manifest.csv"
AQPR_POINT_METRICS_CSV = BOOTSTRAP_DIR / "aqpr_model_panel_locked_test_point_metrics.csv"
AQPR_BOOTSTRAP_CI_CSV = BOOTSTRAP_DIR / "aqpr_model_panel_locked_test_bootstrap_ci.csv"
AQPR_PAIRED_COMPARISON_CSV = BOOTSTRAP_DIR / "aqpr_model_panel_paired_comparison_vs_primary.csv"

CALIBRATION_SUMMARY_CSV = CALIBRATION_DIR / "aqpr_panel_calibration_summary.csv"
TEMPERATURE_SCALING_CSV = CALIBRATION_DIR / "aqpr_panel_temperature_scaling_parameters.csv"

FROZEN_PANEL_CSV = FROZEN_PANEL_DIR / "notebook07_frozen_aqpr_model_panel_manifest.csv"
FROZEN_PANEL_JSON = FROZEN_PANEL_DIR / "notebook07_frozen_aqpr_model_panel_manifest.json"
DOWNSTREAM_REGISTRY_CSV = FROZEN_PANEL_DIR / "notebook07_downstream_artifact_registry_for_notebooks08_to16.csv"

FINAL_STATUS_JSON = RESULTS_07_DIR / "notebook07_final_status.json"
FINAL_STATUS_TXT = RESULTS_07_DIR / "notebook07_FINAL_STATUS.txt"
FINAL_CHECKS_CSV = RESULTS_07_DIR / "notebook07_final_status_checks.csv"


def as_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, np.integer)):
        return bool(value)
    if isinstance(value, (float, np.floating)):
        return False if np.isnan(value) else bool(int(value))
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def jsonable(value):
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [jsonable(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return [jsonable(v) for v in value.tolist()]
    if isinstance(value, pd.DataFrame):
        return [jsonable(r) for r in value.to_dict(orient="records")]
    if isinstance(value, pd.Series):
        return [jsonable(v) for v in value.tolist()]
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return value


def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def one_row(df, label, **filters):
    mask = pd.Series(True, index=df.index)

    for col, value in filters.items():
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in {label}")
        mask &= df[col].astype(str).eq(str(value))

    rows = df[mask].copy()

    if len(rows) != 1:
        raise RuntimeError(f"Expected one row for {label} with {filters}, found {len(rows)}")

    return rows.iloc[0].to_dict()


def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)

    if not path.exists():
        return ""

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()


def size_mb(path):
    path = Path(path)
    return round(path.stat().st_size / (1024 * 1024), 3) if path.exists() else np.nan


required_files = {
    "final decision": FINAL_DECISION_JSON,
    "locked-test summary": TEST_SUMMARY_CSV,
    "AQPR panel manifest": AQPR_PANEL_MANIFEST_CSV,
    "AQPR point metrics": AQPR_POINT_METRICS_CSV,
    "AQPR bootstrap CI": AQPR_BOOTSTRAP_CI_CSV,
    "AQPR paired comparison": AQPR_PAIRED_COMPARISON_CSV,
    "calibration summary": CALIBRATION_SUMMARY_CSV,
    "temperature scaling": TEMPERATURE_SCALING_CSV,
}

for label, path in required_files.items():
    require_file(path, label)

with open(FINAL_DECISION_JSON, "r", encoding="utf-8") as f:
    final_decision = json.load(f)

test_summary = pd.read_csv(TEST_SUMMARY_CSV)
panel_manifest = pd.read_csv(AQPR_PANEL_MANIFEST_CSV)
point_metrics = pd.read_csv(AQPR_POINT_METRICS_CSV)
bootstrap_ci = pd.read_csv(AQPR_BOOTSTRAP_CI_CSV)
calibration_summary = pd.read_csv(CALIBRATION_SUMMARY_CSV)
temperature_scaling = pd.read_csv(TEMPERATURE_SCALING_CSV)

if final_decision["selected_model_name"] != PRIMARY_MODEL_NAME:
    raise RuntimeError(
        f"Primary model mismatch. Expected {PRIMARY_MODEL_NAME}, "
        f"found {final_decision['selected_model_name']}"
    )

observed_panel_models = panel_manifest["model_name"].astype(str).tolist()

if observed_panel_models != AQPR_PANEL_MODEL_NAMES:
    raise RuntimeError(
        f"Unexpected AQPR panel. Expected {AQPR_PANEL_MODEL_NAMES}, "
        f"found {observed_panel_models}"
    )

if "used_to_redefine_primary_model_after_test" in panel_manifest.columns:
    if panel_manifest["used_to_redefine_primary_model_after_test"].apply(as_bool).any():
        raise RuntimeError("Locked test must not redefine the primary model.")

frozen_rows = []

for model_name in AQPR_PANEL_MODEL_NAMES:
    role = AQPR_PANEL_ROLES[model_name]

    summary_row = one_row(
        test_summary,
        "locked-test summary",
        model_name=model_name,
    )

    metric_row = one_row(
        point_metrics,
        "AQPR point metrics",
        model_name=model_name,
    )

    temp_row = one_row(
        temperature_scaling,
        "temperature scaling",
        model_name=model_name,
    )

    raw_cal_row = one_row(
        calibration_summary,
        "raw test calibration",
        model_name=model_name,
        split="test",
        probability_variant="raw",
    )

    temp_cal_row = one_row(
        calibration_summary,
        "temperature-scaled test calibration",
        model_name=model_name,
        split="test",
        probability_variant="temperature_scaled",
    )

    checkpoint_path = Path(str(summary_row["checkpoint"]))
    require_file(checkpoint_path, f"{model_name} checkpoint")

    ci_values = {}

    for metric in ["auroc", "auprc", "balanced_accuracy", "sensitivity", "specificity", "brier_score"]:
        ci_row = one_row(
            bootstrap_ci,
            "AQPR bootstrap CI",
            model_name=model_name,
            metric=metric,
        )

        ci_values[f"test_{metric}_ci_low"] = float(ci_row["ci_low"])
        ci_values[f"test_{metric}_ci_high"] = float(ci_row["ci_high"])

    frozen_rows.append(
        {
            "model_name": model_name,
            "aqpr_panel_role": role,
            "is_primary_validation_selected_classifier": model_name == PRIMARY_MODEL_NAME,
            "is_secondary_aqpr_model": model_name in SECONDARY_AQPR_MODEL_NAMES,
            "family": str(summary_row["family"]),
            "run_id": str(summary_row["run_id"]),
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_exists": checkpoint_path.exists(),
            "checkpoint_size_mb": size_mb(checkpoint_path),
            "checkpoint_sha256": sha256_file(checkpoint_path) if COMPUTE_CHECKPOINT_SHA256 else "",
            "input_size": int(summary_row["input_size"]),
            "batch_size": int(summary_row["batch_size"]),
            "validation_selected_threshold": float(summary_row["validation_selected_threshold"]),
            "threshold_source": "validation_balanced_accuracy",
            "test_used_to_redefine_primary_model": False,
            "test_used_to_fit_temperature": False,
            "true_patient_level_available": False,
            "internal_patient_level_claim_allowed": False,

            "val_image_auroc": float(summary_row["val_image_auroc"]),
            "val_image_auprc": float(summary_row["val_image_auprc"]),
            "val_image_balanced_accuracy": float(summary_row["val_image_balanced_accuracy"]),

            "test_image_auroc": float(metric_row["auroc"]),
            "test_image_auprc": float(metric_row["auprc"]),
            "test_image_accuracy": float(metric_row["accuracy"]),
            "test_image_balanced_accuracy": float(metric_row["balanced_accuracy"]),
            "test_image_sensitivity": float(metric_row["sensitivity"]),
            "test_image_specificity": float(metric_row["specificity"]),
            "test_image_f1": float(metric_row["f1"]),
            "test_image_brier_score_raw": float(metric_row["brier_score"]),

            "temperature": float(temp_row["temperature"]),
            "test_raw_ece_10": float(raw_cal_row["ece_10"]),
            "test_raw_negative_log_likelihood": float(raw_cal_row["negative_log_likelihood"]),
            "test_temperature_scaled_brier_score": float(temp_cal_row["brier_score"]),
            "test_temperature_scaled_ece_10": float(temp_cal_row["ece_10"]),
            "test_temperature_scaled_negative_log_likelihood": float(temp_cal_row["negative_log_likelihood"]),

            "val_image_predictions": str(summary_row["val_image_predictions"]),
            "test_image_predictions": str(summary_row["test_image_predictions"]),

            **ci_values,
        }
    )

frozen_panel = pd.DataFrame(frozen_rows)
frozen_panel.to_csv(FROZEN_PANEL_CSV, index=False)

frozen_panel_json = {
    "notebook": "07_train_or_load_airogs_light_glaucoma_classifiers",
    "created_timestamp_unix": time.time(),
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "aqpr_panel_model_names": AQPR_PANEL_MODEL_NAMES,
    "selection_policy": (
        "EfficientNetB0 remains the primary validation-selected classifier. "
        "DenseNet121, SwinTiny, and ConvNeXtTiny are secondary AQPR models. "
        "The locked test set did not redefine the primary model."
    ),
    "temperature_scaling_policy": (
        "Temperature was fitted on validation logits only and evaluated on locked test."
    ),
    "internal_patient_level_claim_allowed": False,
    "true_patient_level_available_internal": False,
    "manifest_csv": str(FROZEN_PANEL_CSV),
    "models": frozen_panel.to_dict(orient="records"),
}

with open(FROZEN_PANEL_JSON, "w", encoding="utf-8") as f:
    json.dump(jsonable(frozen_panel_json), f, indent=2)

registry_rows = []

for _, row in frozen_panel.iterrows():
    registry_rows.extend(
        [
            {
                "artifact_key": f"{row['model_name']}_checkpoint",
                "artifact_type": "checkpoint",
                "path": row["checkpoint_path"],
                "exists": Path(row["checkpoint_path"]).exists(),
                "required_for": "notebooks08_to16",
            },
            {
                "artifact_key": f"{row['model_name']}_val_predictions",
                "artifact_type": "prediction_table",
                "path": row["val_image_predictions"],
                "exists": Path(row["val_image_predictions"]).exists(),
                "required_for": "notebooks08_to16",
            },
            {
                "artifact_key": f"{row['model_name']}_test_predictions",
                "artifact_type": "prediction_table",
                "path": row["test_image_predictions"],
                "exists": Path(row["test_image_predictions"]).exists(),
                "required_for": "notebook07_reporting",
            },
        ]
    )

registry_rows.extend(
    [
        {
            "artifact_key": "frozen_aqpr_panel_manifest_csv",
            "artifact_type": "manifest",
            "path": str(FROZEN_PANEL_CSV),
            "exists": FROZEN_PANEL_CSV.exists(),
            "required_for": "notebooks08_to16",
        },
        {
            "artifact_key": "frozen_aqpr_panel_manifest_json",
            "artifact_type": "manifest",
            "path": str(FROZEN_PANEL_JSON),
            "exists": FROZEN_PANEL_JSON.exists(),
            "required_for": "notebooks08_to16",
        },
    ]
)

downstream_registry = pd.DataFrame(registry_rows)
downstream_registry.to_csv(DOWNSTREAM_REGISTRY_CSV, index=False)

checks = pd.DataFrame(
    [
        {
            "check": "primary_model_is_efficientnetb0",
            "passed": final_decision["selected_model_name"] == PRIMARY_MODEL_NAME,
        },
        {
            "check": "panel_has_four_models",
            "passed": len(frozen_panel) == 4,
        },
        {
            "check": "panel_models_match_expected_order",
            "passed": frozen_panel["model_name"].tolist() == AQPR_PANEL_MODEL_NAMES,
        },
        {
            "check": "exactly_one_primary_model",
            "passed": int(frozen_panel["is_primary_validation_selected_classifier"].sum()) == 1,
        },
        {
            "check": "three_secondary_models",
            "passed": int(frozen_panel["is_secondary_aqpr_model"].sum()) == 3,
        },
        {
            "check": "locked_test_did_not_redefine_primary_model",
            "passed": not frozen_panel["test_used_to_redefine_primary_model"].any(),
        },
        {
            "check": "temperature_not_fit_on_test",
            "passed": not frozen_panel["test_used_to_fit_temperature"].any(),
        },
        {
            "check": "all_checkpoints_exist",
            "passed": frozen_panel["checkpoint_exists"].all(),
        },
        {
            "check": "all_val_prediction_files_exist",
            "passed": all(Path(p).exists() for p in frozen_panel["val_image_predictions"]),
        },
        {
            "check": "frozen_manifests_exist",
            "passed": FROZEN_PANEL_CSV.exists() and FROZEN_PANEL_JSON.exists(),
        },
    ]
)

checks.to_csv(FINAL_CHECKS_CSV, index=False)

all_passed = bool(checks["passed"].all())

primary_row = frozen_panel[frozen_panel["model_name"] == PRIMARY_MODEL_NAME].iloc[0].to_dict()

final_status = {
    "notebook": "07_train_or_load_airogs_light_glaucoma_classifiers",
    "status": "PASS" if all_passed else "FAIL",
    "ready_for_notebook08": all_passed,
    "primary_training_dataset": "EyePACS-AIROGS-light-V2",
    "primary_model_name": PRIMARY_MODEL_NAME,
    "secondary_aqpr_model_names": SECONDARY_AQPR_MODEL_NAMES,
    "frozen_panel_manifest_csv": str(FROZEN_PANEL_CSV),
    "frozen_panel_manifest_json": str(FROZEN_PANEL_JSON),
    "downstream_registry_csv": str(DOWNSTREAM_REGISTRY_CSV),
    "final_checks_csv": str(FINAL_CHECKS_CSV),
    "primary_test_auroc": float(primary_row["test_image_auroc"]),
    "primary_test_auprc": float(primary_row["test_image_auprc"]),
    "primary_test_balanced_accuracy": float(primary_row["test_image_balanced_accuracy"]),
    "primary_test_sensitivity": float(primary_row["test_image_sensitivity"]),
    "primary_test_specificity": float(primary_row["test_image_specificity"]),
    "primary_temperature_scaled_brier": float(primary_row["test_temperature_scaled_brier_score"]),
    "internal_patient_level_claim_allowed": False,
}

with open(FINAL_STATUS_JSON, "w", encoding="utf-8") as f:
    json.dump(jsonable(final_status), f, indent=2)

status_text = f"""
Notebook 07 final status
========================

Status: {final_status["status"]}
Ready for Notebook 08: {final_status["ready_for_notebook08"]}

Primary model:
  {PRIMARY_MODEL_NAME}

Secondary AQPR models:
  {", ".join(SECONDARY_AQPR_MODEL_NAMES)}

Frozen panel manifest:
  {FROZEN_PANEL_CSV}

Policy:
  EfficientNetB0 remains the validation-selected primary classifier.
  DenseNet121, SwinTiny, and ConvNeXtTiny are secondary AQPR models.
  The locked test set was not used to redefine the primary model.
  Temperature scaling was fitted on validation only.
  Internal AIROGS-light evaluation remains image-level.
""".strip()

with open(FINAL_STATUS_TXT, "w", encoding="utf-8") as f:
    f.write(status_text)

print("Notebook 07 final checks:")
print(checks.to_string(index=False))

print("\nFrozen AQPR panel:")
print(
    frozen_panel[
        [
            "model_name",
            "aqpr_panel_role",
            "family",
            "checkpoint_exists",
            "test_image_auroc",
            "test_image_auprc",
            "test_image_balanced_accuracy",
            "test_image_sensitivity",
            "test_image_specificity",
            "test_temperature_scaled_brier_score",
            "test_temperature_scaled_ece_10",
        ]
    ].to_string(index=False)
)

print("\nFinal status:")
print(json.dumps(jsonable(final_status), indent=2))

print(f"\nFrozen panel manifest CSV: {FROZEN_PANEL_CSV}")
print(f"Frozen panel manifest JSON: {FROZEN_PANEL_JSON}")
print(f"Downstream registry CSV: {DOWNSTREAM_REGISTRY_CSV}")
print(f"Final checks CSV: {FINAL_CHECKS_CSV}")
print(f"Final status JSON: {FINAL_STATUS_JSON}")
print(f"Final status TXT: {FINAL_STATUS_TXT}")

if not all_passed:
    failed = checks[~checks["passed"]].copy()
    print("\nFailed checks:")
    print(failed.to_string(index=False))
    raise RuntimeError("Notebook 07 final checks failed.")

print("Status: PASS")